# NLP Project 3 — Icelandic ASR
## Person 3: Whisper Small Fine-Tuning

**Dataset:** Samrómur 21.05 (OpenSLR 112)  
**Model:** OpenAI Whisper Small Multilingual  
**Platform:** Google Colab  
**Deadline:** November 23, 2026

### Objectives
1. Fine-tune Whisper Small on approximately 15 hours of Icelandic speech.
2. Monitor training and validation performance.
3. Record training hyperparameters and GPU usage.
4. Select the best checkpoint using validation results.
5. Generate predictions on the shared 3-hour test set.
6. Export the model, training logs, and predictions.

### Experimental Rules
- Use only the official training subset for fine-tuning.
- Use validation data for model selection.
- Keep the test set untouched until final evaluation.
- Record all hyperparameters and random seeds.
- Preserve Icelandic characters and original transcripts.
- Save checkpoints to persistent storage.

In [ ]:
# Reproducibility setup — Person 3
# Install the exact package versions documented in the repository.
# Run this cell in a fresh Colab environment before running other code.

%pip install -q -r https://raw.githubusercontent.com/RanimAlyaseen/icelandic-asr-project3/person3-whisper-reproducibility/training/requirements-person3.txt

### Model and Dataset Version Tracking

**Base model:** `openai/whisper-small` (multilingual)

**Original base-model revision:** Not recorded during the original training run.

**Dataset:** Samrómur 21.05 (OpenSLR 112)

**Dataset selection:** `samromur-21.05-20h-v1`

**Dataset preparation reference:** Git commit `55f5a68`

**Original dataset checkout revision:** Not recorded during the original training run.

**Reproducibility note:** The dataset preparation reference identifies the team's frozen dataset selection. It should not be interpreted as a verified record of the exact checkout used during the original training run.

**Recovered model:** Whisper Small fine-tuning checkpoint 1000.

**Checkpoint verification:** SHA-256 checksums are documented and verified by `training/download_whisper_checkpoint.py`.

The original training results are preserved. No retraining is required.

In [ ]:
# Record exact revisions for future reproducible runs.
# This cell does not load the model or start training.

import subprocess
from huggingface_hub import HfApi

BASE_MODEL_ID = "openai/whisper-small"

# Resolve the current Hugging Face model revision.
BASE_MODEL_REVISION = HfApi().model_info(BASE_MODEL_ID).sha

# Record the repository revision if running inside a Git checkout.
result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=False,
)

DATASET_REPOSITORY_REVISION = (
    result.stdout.strip() if result.returncode == 0 else None
)

print("Base model:", BASE_MODEL_ID)
print("Resolved base-model revision:", BASE_MODEL_REVISION)
print("Dataset repository revision:", DATASET_REPOSITORY_REVISION)
print("Dataset selection: samromur-21.05-20h-v1")

In [ ]:
import sys
import platform
import subprocess
import torch
import pandas as pd

print("=" * 60)
print("NLP PROJECT 3 — TRAINING ENVIRONMENT")
print("=" * 60)

print(f"\nPython version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"Operating system: {platform.system()}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    gpu_properties = torch.cuda.get_device_properties(0)

    gpu_memory = gpu_properties.total_memory / (1024**3)

    print(f"\nGPU: {gpu_name}")
    print(f"GPU memory: {gpu_memory:.2f} GB")

    print("\nNVIDIA GPU information:")

    subprocess.run(["nvidia-smi"])

else:
    print("\nWARNING: No GPU detected!")
    print("Enable GPU from Runtime > Change runtime type.")

print("\nEnvironment inspection completed.")

NLP PROJECT 3 — TRAINING ENVIRONMENT

Python version: 3.13.16
PyTorch version: 2.14.1+cu130
Operating system: Linux
CUDA available: True

GPU: Tesla T4
GPU memory: 14.56 GB

NVIDIA GPU information:

Environment inspection completed.


## Step 2 — Dependencies and Model Initialization

### Objective
Install and verify the libraries required for Whisper Small fine-tuning.

### Base Model
- Model ID: `openai/whisper-small`
- Architecture: Encoder-decoder Transformer
- Parameters: Approximately 244 million
- Language: Icelandic (`is`)
- Task: Automatic Speech Recognition

### Required Libraries
- Transformers: Model and processor
- Datasets: Audio dataset handling
- Accelerate: GPU training
- Evaluate: WER and CER
- JiWER: Speech recognition evaluation
- SoundFile: Audio loading
- Librosa: Audio processing

### Validation Checks
1. Verify installed library versions.
2. Load the Whisper processor.
3. Load the pretrained model.
4. Configure Icelandic transcription.
5. Verify GPU compatibility.

In [ ]:
%pip install -q "evaluate==0.4.6" "jiwer==4.0.0"

In [ ]:
%pip install -q "evaluate==0.4.6" "jiwer==4.0.0"


## Step 2B — Whisper Small Model Initialization

### Objective
Load the pretrained Whisper Small Multilingual model and configure it for Icelandic automatic speech recognition.

### Model Configuration
- Model: openai/whisper-small
- Architecture: Encoder-decoder Transformer
- Language: Icelandic
- Task: Transcription

### Validation
- Verify model and processor loading.
- Check the total number of parameters.
- Verify Icelandic language configuration.
- Confirm the model is ready for inference and fine-tuning.

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-small"

print("=" * 60)
print("WHISPER SMALL — MODEL INITIALIZATION")
print("=" * 60)

# Load processor for Icelandic transcription
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language="icelandic",
    task="transcribe"
)

print("\nProcessor loaded successfully.")

# Load pretrained model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID
)

print("Model loaded successfully.")

# Configure transcription language
model.generation_config.language = "icelandic"
model.generation_config.task = "transcribe"

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("\nMODEL INFORMATION")
print("-" * 60)

print(f"Model ID: {MODEL_ID}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params / 1e6:.2f}M")

print("\nLANGUAGE CONFIGURATION")
print("-" * 60)

print("Language:", model.generation_config.language)
print("Task:", model.generation_config.task)

print("\nDEVICE INFORMATION")
print("-" * 60)

print("Model device:", next(model.parameters()).device)
print("GPU available:", torch.cuda.is_available())

print("\nMODEL INITIALIZATION COMPLETE")

WHISPER SMALL — MODEL INITIALIZATION


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

... (6 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Model loaded successfully.

MODEL INFORMATION
------------------------------------------------------------
Model ID: openai/whisper-small
Total parameters: 241,734,912
Trainable parameters: 241,734,912
Model size: 241.73M

LANGUAGE CONFIGURATION
------------------------------------------------------------
Language: icelandic
Task: transcribe

DEVICE INFORMATION
------------------------------------------------------------
Model device: cpu
GPU available: True

MODEL INITIALIZATION COMPLETE


## Step 3 — Fine-Tuning Configuration

### Objective
Configure Whisper Small Multilingual for Icelandic ASR fine-tuning.

### Training Strategy
- Base model: `openai/whisper-small`
- Training data: Approximately 15 hours
- Validation data: Approximately 2 hours
- Test data: Approximately 3 hours
- GPU: NVIDIA Tesla T4
- Training method: Full fine-tuning
- Precision: FP16
- Gradient accumulation: Enabled
- Gradient checkpointing: Enabled if required

### Experimental Requirements
1. Record all hyperparameters.
2. Use a fixed random seed for reproducibility.
3. Monitor training and validation losses.
4. Calculate validation WER and CER.
5. Save checkpoints and training logs.
6. Select the best checkpoint using validation results.

### Important
The test dataset must not be used during training or checkpoint selection.

In [ ]:
import json
import random
import numpy as np
import torch

from pathlib import Path
from datetime import datetime, timezone

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Project configuration
TRAINING_CONFIG = {
    "project": "Icelandic ASR - NLP Project 3",
    "model_id": "openai/whisper-small",
    "language": "icelandic",
    "task": "transcribe",

    # Dataset targets
    "train_hours_target": 15,
    "validation_hours_target": 2,
    "test_hours_target": 3,

    # Initial training hyperparameters
    "learning_rate": 1e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "linear",

    # Memory optimization
    "fp16": True,
    "gradient_checkpointing": True,

    # Experiment tracking
    "seed": SEED,
    "logging_steps": 25,
    "save_strategy": "epoch",
    "evaluation_strategy": "epoch",
    "save_total_limit": 2,

    # Model selection
    "metric_for_best_model": "wer",
    "greater_is_better": False,

    # Audio
    "target_sample_rate": 16000,
    "max_audio_duration_seconds": 30,

    # Metadata
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),
    "pytorch_version": torch.__version__
}

# Effective batch size for one GPU
effective_batch_size = (
    TRAINING_CONFIG["per_device_train_batch_size"]
    * TRAINING_CONFIG["gradient_accumulation_steps"]
)

# Save configuration locally
output_dir = Path("/content/icelandic_asr")
output_dir.mkdir(parents=True, exist_ok=True)

config_path = output_dir / "training_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(TRAINING_CONFIG, f, indent=4, ensure_ascii=False)

print("=" * 60)
print("WHISPER SMALL — FINE-TUNING CONFIGURATION")
print("=" * 60)

print(f"\nModel: {TRAINING_CONFIG['model_id']}")
print(f"GPU: {TRAINING_CONFIG['gpu']}")
print(f"Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"Batch size: {TRAINING_CONFIG['per_device_train_batch_size']}")
print(f"Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"Effective batch size: {effective_batch_size}")
print(f"FP16: {TRAINING_CONFIG['fp16']}")
print(f"Gradient checkpointing: {TRAINING_CONFIG['gradient_checkpointing']}")

print(f"\nConfiguration saved to: {config_path}")
print("\nSTEP 3 COMPLETE")

WHISPER SMALL — FINE-TUNING CONFIGURATION

Model: openai/whisper-small
GPU: CPU
Learning rate: 1e-05
Epochs: 3
Batch size: 2
Gradient accumulation: 8
Effective batch size: 16
FP16: True
Gradient checkpointing: True

Configuration saved to: /content/icelandic_asr/training_config.json

STEP 3 COMPLETE


## Step 4 — Persistent Storage and Experiment Tracking

### Objective
Configure persistent Google Drive storage for Whisper Small fine-tuning.

### Storage Structure
- `configs/`: Training configurations and environment information.
- `checkpoints/`: Model checkpoints during training.
- `logs/`: Training and validation logs.
- `results/`: Training graphs, metrics, and predictions.
- `final_model/`: Best fine-tuned Whisper Small model.

### Requirements
1. Mount Google Drive.
2. Create the project directories.
3. Back up the training configuration.
4. Verify that files can be written and read.
5. Preserve all important training outputs across Colab sessions.

### Note
Checkpoints will be saved directly to Google Drive to prevent
data loss if the Colab runtime disconnects.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

print("=" * 60)
print("STEP 4 — PERSISTENT STORAGE SETUP")
print("=" * 60)

# Mount Google Drive
drive.mount("/content/drive")

# Main project directory
PROJECT_DIR = Path(
    "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning"
)

# Define project folders
DIRECTORIES = {
    "configs": PROJECT_DIR / "configs",
    "checkpoints": PROJECT_DIR / "checkpoints",
    "logs": PROJECT_DIR / "logs",
    "results": PROJECT_DIR / "results",
    "final_model": PROJECT_DIR / "final_model"
}

# Create directories
for name, path in DIRECTORIES.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"Created/verified: {name}")

# Back up training configuration
source_config = Path(
    "/content/icelandic_asr/training_config.json"
)

destination_config = (
    DIRECTORIES["configs"] / "training_config.json"
)

assert source_config.exists(), (
    "Training configuration not found. "
    "Please rerun Step 3."
)

shutil.copy2(source_config, destination_config)

# Verify file integrity
with open(destination_config, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

assert saved_config["model_id"] == "openai/whisper-small"
assert saved_config["seed"] == 42

# Test Drive write/read access
test_file = PROJECT_DIR / "storage_test.txt"
test_file.write_text("Storage verification successful.", encoding="utf-8")

assert test_file.read_text(
    encoding="utf-8"
) == "Storage verification successful."

test_file.unlink()

print("\n" + "=" * 60)
print("STORAGE VERIFICATION")
print("=" * 60)

print(f"Project directory: {PROJECT_DIR}")
print(f"Configuration backup: {destination_config}")
print("Google Drive write/read test: PASSED")

print("\nSTEP 4 COMPLETE")

STEP 4 — PERSISTENT STORAGE SETUP
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created/verified: configs
Created/verified: checkpoints
Created/verified: logs
Created/verified: results
Created/verified: final_model

STORAGE VERIFICATION
Project directory: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning
Configuration backup: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/configs/training_config.json
Google Drive write/read test: PASSED

STEP 4 COMPLETE


## Step 5 — Whisper Fine-Tuning Pipeline

### Objective
Prepare the training and evaluation components for
Whisper Small Multilingual.

### Components
1. WER and CER evaluation metrics.
2. Whisper data collator.
3. Training arguments and optimization settings.
4. Checkpoint selection and experiment logging.

### Evaluation Strategy
- Training loss measures model optimization.
- Validation loss measures performance on unseen validation data.
- Validation WER measures word recognition errors.
- Validation CER measures character recognition errors.
- The best checkpoint will be selected using validation WER.

### Important
The test dataset will not be used during training.

Final statistical evaluation will be performed by Person 4.

In [ ]:
import evaluate
import numpy as np

print("=" * 60)
print("STEP 5A — EVALUATION METRICS")
print("=" * 60)

# Load evaluation metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print("\nWER metric loaded successfully.")
print("CER metric loaded successfully.")

# Test the metrics using sample sentences
references = [
    "hello world",
    "this is a test"
]

predictions = [
    "hello world",
    "this is test"
]

wer = wer_metric.compute(
    predictions=predictions,
    references=references
)

cer = cer_metric.compute(
    predictions=predictions,
    references=references
)

print("\nMETRIC VALIDATION")
print("-" * 60)

print(f"WER: {wer:.4f} ({wer * 100:.2f}%)")
print(f"CER: {cer:.4f} ({cer * 100:.2f}%)")

# Verify expected WER:
# 1 deletion across 6 reference words
expected_wer = 1 / 6

assert np.isclose(wer, expected_wer), (
    f"Unexpected WER: {wer}"
)

assert 0 <= cer <= 1, "Unexpected CER value."

print("\nMetric validation: PASSED")
print("STEP 5A COMPLETE")

STEP 5A — EVALUATION METRICS



WER metric loaded successfully.
CER metric loaded successfully.

METRIC VALIDATION
------------------------------------------------------------
WER: 0.1667 (16.67%)
CER: 0.0800 (8.00%)

Metric validation: PASSED
STEP 5A COMPLETE


### Step 5B — Whisper Data Collator

**Objective:** Prepare audio features and tokenized transcripts for training.

**Data Collator Responsibilities:**
1. Combine audio features into training batches.
2. Pad variable-length transcript sequences.
3. Replace label padding with -100 to exclude it from loss calculations.
4. Handle Whisper's decoder start token correctly.
5. Produce tensors compatible with the Whisper training pipeline.

**Expected Output:**
A reusable data collator for training and validation.

**Note:** The collator will be tested with Person 1's dataset before training.

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from transformers import WhisperProcessor

print("=" * 60)
print("STEP 5B — WHISPER DATA COLLATOR")
print("=" * 60)

# Reload processor after runtime restart
MODEL_ID = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language="icelandic",
    task="transcribe"
)


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:

    processor: Any

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        # Prepare audio features
        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # Prepare transcript labels
        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        # Ignore padding tokens during loss calculation
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        # Remove decoder start token if already present
        decoder_start_token_id = (
            self.processor.tokenizer.convert_tokens_to_ids(
                "<|startoftranscript|>"
            )
        )

        if (
            labels.shape[1] > 0
            and (labels[:, 0] == decoder_start_token_id).all().item()
        ):
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


# Initialize data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

print("\nProcessor loaded successfully.")
print("Data collator initialized successfully.")

print("\nConfiguration:")
print("Model:", MODEL_ID)
print("Language: Icelandic")
print("Task: Transcription")
print("Padding labels masked with: -100")

print("\nSTEP 5B COMPLETE")

STEP 5B — WHISPER DATA COLLATOR

Processor loaded successfully.
Data collator initialized successfully.

Configuration:
Model: openai/whisper-small
Language: Icelandic
Task: Transcription
Padding labels masked with: -100

STEP 5B COMPLETE


### Step 5C — Training Arguments and Checkpoint Configuration

**Objective:** Configure the Hugging Face Seq2SeqTrainer for
Whisper Small fine-tuning.

**Training Configuration:**
- Learning rate: 1e-5
- Epochs: 3
- Training batch size: 2
- Gradient accumulation: 8
- Effective batch size: 16
- Precision: FP16
- Gradient checkpointing: Enabled
- Validation: Every epoch
- Checkpoint saving: Every epoch
- Best checkpoint: Lowest validation WER

**Experiment Tracking:**
- Record training loss and validation loss.
- Record validation WER and CER.
- Record training time and GPU memory usage.
- Save training logs and model checkpoints.

**Important:**
- Use validation data for checkpoint selection.
- Do not evaluate on the test set during training.
- Verify compatibility with the installed Transformers version.

In [ ]:
import json
import inspect
import torch

from pathlib import Path
from transformers import Seq2SeqTrainingArguments

print("=" * 60)
print("STEP 5C — TRAINING ARGUMENTS")
print("=" * 60)

# Reload configuration from Google Drive
PROJECT_DIR = Path(
    "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning"
)

config_path = PROJECT_DIR / "configs" / "training_config.json"

assert config_path.exists(), "Training configuration not found."

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

# Check supported arguments in installed Transformers
supported_args = inspect.signature(
    Seq2SeqTrainingArguments
).parameters

# Handle differences between Transformers versions
if "eval_strategy" in supported_args:
    eval_key = "eval_strategy"
elif "evaluation_strategy" in supported_args:
    eval_key = "evaluation_strategy"
else:
    raise RuntimeError(
        "No supported evaluation strategy argument found."
    )

print(f"\nEvaluation argument: {eval_key}")

# Training arguments
args_dict = {
    "output_dir": str(PROJECT_DIR / "checkpoints"),

    "learning_rate": config["learning_rate"],
    "num_train_epochs": config["num_train_epochs"],

    "per_device_train_batch_size":
        config["per_device_train_batch_size"],

    "per_device_eval_batch_size":
        config["per_device_eval_batch_size"],

    "gradient_accumulation_steps":
        config["gradient_accumulation_steps"],

    "weight_decay": config["weight_decay"],
    "warmup_steps": 0,
    "lr_scheduler_type": config["lr_scheduler_type"],

    "fp16": config["fp16"],
    "gradient_checkpointing":
        config["gradient_checkpointing"],

    "logging_strategy": "steps",
    "logging_steps": config["logging_steps"],

    "save_strategy": config["save_strategy"],
    "save_total_limit": config["save_total_limit"],

    "predict_with_generate": True,
    "generation_max_length": 225,

    "load_best_model_at_end": True,
    "metric_for_best_model":
        config["metric_for_best_model"],

    "greater_is_better":
        config["greater_is_better"],

    "seed": config["seed"],

    "report_to": "none",

    "remove_unused_columns": False,
    "dataloader_num_workers": 0
}

args_dict[eval_key] = config["evaluation_strategy"]

# Verify argument compatibility
unsupported = [
    key for key in args_dict
    if key not in supported_args
]

if unsupported:
    raise RuntimeError(
        f"Unsupported training arguments: {unsupported}"
    )

# Initialize training arguments
training_args = Seq2SeqTrainingArguments(**args_dict)

print("\nTRAINING CONFIGURATION")
print("-" * 60)

print("Learning rate:", training_args.learning_rate)
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("FP16:", training_args.fp16)
print("Gradient checkpointing:", training_args.gradient_checkpointing)

print("\nCHECKPOINT CONFIGURATION")
print("-" * 60)

print("Output directory:", training_args.output_dir)
print("Save strategy:", training_args.save_strategy)
print("Best model metric:", training_args.metric_for_best_model)
print("Load best model:", training_args.load_best_model_at_end)

print("\nSTEP 5C COMPLETE")

STEP 5C — TRAINING ARGUMENTS

Evaluation argument: eval_strategy

TRAINING CONFIGURATION
------------------------------------------------------------
Learning rate: 1e-05
Epochs: 3
Batch size: 2
Gradient accumulation: 8
FP16: True
Gradient checkpointing: True

CHECKPOINT CONFIGURATION
------------------------------------------------------------
Output directory: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/checkpoints
Save strategy: SaveStrategy.EPOCH
Best model metric: wer
Load best model: True

STEP 5C COMPLETE


### Step 5D — Training Configuration Validation

**Objective:** Validate the training configuration before initializing the trainer.

**Checks:**
- Confirm the training and evaluation strategies match.
- Confirm the best checkpoint is selected using validation WER.
- Confirm FP16 is enabled and CUDA is available.
- Confirm the effective batch size.
- Identify settings that must be finalized after receiving the dataset.

**Expected result:** All configuration checks pass.

## Step 6 — Whisper Trainer Preparation

### Step 6A — Model Initialization and GPU Verification

**Objective:** Reload the pretrained Whisper Small multilingual model and verify GPU compatibility.

**Model:** `openai/whisper-small`

**Language:** Icelandic

**Task:** Automatic Speech Recognition (Transcription)

**Checks:**
- Load the pretrained model.
- Configure Icelandic transcription.
- Verify model parameters.
- Verify GPU compatibility.
- Record GPU memory usage.

**Expected Output:** Whisper Small is initialized successfully and ready for trainer integration.

**Note:** No training or test-set evaluation occurs in this step.

# Step 7 — Dataset Integration

## Step 7A — Repository Setup

Retrieve Person 1's prepared Icelandic ASR dataset and inspect
the repository structure before loading the audio and transcripts.

In [ ]:
from pathlib import Path
import subprocess
import os

print("=" * 60)
print("STEP 7A — DATASET REPOSITORY SETUP")
print("=" * 60)

repo_url = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
repo_path = Path("/content/icelandic-asr-project3")

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(repo_path)],
        check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    )
else:
    print("Repository already exists.")

print("\nREPOSITORY CONTENTS")
print("-" * 60)

for item in sorted(repo_path.iterdir()):
    print(f"{'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")

print("\nSTEP 7A COMPLETE")

STEP 7A — DATASET REPOSITORY SETUP

REPOSITORY CONTENTS
------------------------------------------------------------
[DIR] .git
[FILE] .gitattributes
[FILE] .gitignore
[FILE] BASELINES.md
[FILE] DATA_STORAGE.md
[FILE] LICENSE
[FILE] README.md
[FILE] audit.py
[DIR] contributions
[DIR] data
[DIR] output
[FILE] prepare.py
[FILE] preprocess.py
[FILE] requirements-windows-py314.lock.txt
[FILE] requirements.txt
[DIR] review
[FILE] subset.py
[DIR] tests
[FILE] verify.py
[FILE] verify_handoff.py

STEP 7A COMPLETE


## Step 7B — Dataset File Verification

Verify the training, validation, and test manifests.

Check that the audio recordings are available and identify
whether Git LFS files were downloaded correctly.

In [ ]:
from pathlib import Path
import subprocess

print("=" * 60)
print("STEP 7B — DATASET FILE VERIFICATION")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")

# Inspect the output directory
print("\nOUTPUT DIRECTORY")
print("-" * 60)

output_dir = repo / "output"

for item in sorted(output_dir.rglob("*")):
    if item.is_file():
        print(f"{item.relative_to(repo)} — {item.stat().st_size:,} bytes")

# Inspect the data directory
print("\nDATA DIRECTORY")
print("-" * 60)

data_dir = repo / "data"
audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

audio_files = [
    p for p in data_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in audio_extensions
]

print(f"Audio files found: {len(audio_files):,}")

if audio_files:
    print("\nSample audio files:")
    for file in audio_files[:5]:
        print(f"  {file.relative_to(repo)} — {file.stat().st_size:,} bytes")

# Verify expected manifests
print("\nMANIFEST VERIFICATION")
print("-" * 60)

for name in ["train.jsonl", "dev.jsonl", "test.jsonl"]:
    path = output_dir / name

    if path.exists():
        with path.open(encoding="utf-8") as f:
            count = sum(1 for line in f if line.strip())

        print(f"{name}: FOUND — {count:,} entries")
    else:
        print(f"{name}: NOT FOUND")

# Check Git LFS availability
print("\nGIT LFS STATUS")
print("-" * 60)

result = subprocess.run(
    ["git", "lfs", "version"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout.strip())
else:
    print("Git LFS is not installed.")

print("\nSTEP 7B COMPLETE")

STEP 7B — DATASET FILE VERIFICATION

OUTPUT DIRECTORY
------------------------------------------------------------
output/audio_audit.jsonl.gz — 10,327,189 bytes
output/ctc_vocab.json — 429 bytes
output/dev.jsonl — 657,876 bytes
output/evaluation_references.jsonl — 564,522 bytes
output/evaluation_references.tsv — 411,047 bytes
output/exclusions.tsv — 10,477 bytes
output/flagged_dev_test_review.tsv — 102,771 bytes
output/focused_review.tsv — 10,139 bytes
output/handoff_verification.json — 2,479 bytes
output/lfs_verification.json — 1,698 bytes
output/listening_queue.tsv — 2,001,220 bytes
output/prompt_overlap.tsv — 748 bytes
output/quality_report.md — 5,273 bytes
output/quality_summary.json — 8,906 bytes
output/raw_subset_cleaning.json — 777 bytes
output/raw_subset_inventory.tsv — 3,089,955 bytes
output/raw_subset_removals.tsv — 827 bytes
output/split_integrity.json — 751 bytes
output/test.jsonl — 969,000 bytes
output/train.jsonl — 5,200,398 bytes
output/verification.json — 2,278 bytes



## Step 7C — Git LFS Audio Retrieval

Install Git LFS and retrieve the actual Icelandic audio recordings.

Git LFS pointer files must be replaced with real audio data
before Whisper preprocessing and training.

In [ ]:
import subprocess
from pathlib import Path

print("=" * 60)
print("STEP 7C — GIT LFS AUDIO RETRIEVAL")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")

# Install Git LFS
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "git-lfs"],
    check=True
)

# Initialize Git LFS
subprocess.run(
    ["git", "lfs", "install"],
    cwd=repo,
    check=True
)

# Download tracked audio files
print("\nDownloading Git LFS audio files...")

subprocess.run(
    ["git", "lfs", "pull"],
    cwd=repo,
    check=True
)

print("\nVERIFY AUDIO FILES")
print("-" * 60)

audio_files = list((repo / "data").rglob("*.flac"))

if not audio_files:
    raise FileNotFoundError("No FLAC audio files found.")

sizes = [p.stat().st_size for p in audio_files]

print(f"FLAC files: {len(audio_files):,}")
print(f"Smallest file: {min(sizes):,} bytes")
print(f"Largest file: {max(sizes):,} bytes")

pointer_files = sum(size < 200 for size in sizes)

print(f"Possible LFS pointers remaining: {pointer_files:,}")

print("\nSTEP 7C COMPLETE")

STEP 7C — GIT LFS AUDIO RETRIEVAL


VERIFY AUDIO FILES
------------------------------------------------------------
FLAC files: 30,993
Smallest file: 15,901 bytes
Largest file: 221,849 bytes
Possible LFS pointers remaining: 0

STEP 7C COMPLETE


## Step 7D — Manifest and Audio Path Verification

Inspect the dataset manifests and verify that their audio
references correspond to existing recordings.

This step does not modify the dataset.

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("STEP 7D — MANIFEST INSPECTION")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")
output_dir = repo / "output"

for split in ["train", "dev", "test"]:

    manifest = output_dir / f"{split}.jsonl"

    with manifest.open(encoding="utf-8") as f:
        first_record = json.loads(f.readline())

    print(f"\n{split.upper()} SPLIT")
    print("-" * 60)

    print("Available fields:")
    print(list(first_record.keys()))

    print("\nFirst record:")

    for key, value in first_record.items():
        display_value = str(value)

        if len(display_value) > 200:
            display_value = display_value[:200] + "..."

        print(f"{key}: {display_value}")

print("\nSTEP 7D COMPLETE")

STEP 7D — MANIFEST INSPECTION

TRAIN SPLIT
------------------------------------------------------------
Available fields:
['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags']

First record:
id: 0072202
split: train
speaker_session_id: 007478
audio_path: train/007478/007478-0072202.flac
sample_rate: 16000
num_samples: 45120
duration_sec: 2.82
text_raw: Ég vil vita meira.
text_source_normalized: ég vil vita meira
text: ég vil vita meira
pcm_sha256: 9a8964624c97480c5228113fe1ae113793a241994fb5edd8fa203061f161accb
flags: []

DEV SPLIT
------------------------------------------------------------
Available fields:
['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags']

First record:
id: 0034220
split: dev
speaker_session_id: 004485
audio_path: dev/004485/004485-0034220.

## Step 7E — Audio Path Integrity Check

Verify that all training, validation, and test recordings
exist and are accessible.

This step checks file paths without modifying the dataset.

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("STEP 7E — AUDIO PATH INTEGRITY")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")
output_dir = repo / "output"

# Person 1's prepared audio directory
audio_root = repo / "data" / "cleaned" / "samromur-21.05-20h-v1"

print(f"\nAudio directory: {audio_root}")
print(f"Directory exists: {audio_root.exists()}")

missing = []

for split in ["train", "dev", "test"]:
    manifest = output_dir / f"{split}.jsonl"
    total = 0
    found = 0

    with manifest.open(encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            total += 1

            audio_file = audio_root / record["audio_path"]

            if audio_file.is_file() and audio_file.stat().st_size > 200:
                found += 1
            else:
                missing.append(str(audio_file))

    print(f"\n{split.upper()}")
    print(f"Expected: {total:,}")
    print(f"Available: {found:,}")
    print(f"Missing: {total - found:,}")

print("\n" + "-" * 60)

if not missing:
    print("ALL AUDIO PATHS VERIFIED")
else:
    print(f"Missing or invalid recordings: {len(missing):,}")
    print("First five:")
    for path in missing[:5]:
        print(path)

print("\nSTEP 7E COMPLETE")

STEP 7E — AUDIO PATH INTEGRITY

Audio directory: /content/icelandic-asr-project3/data/cleaned/samromur-21.05-20h-v1
Directory exists: True

TRAIN
Expected: 10,488
Available: 10,488
Missing: 0

DEV
Expected: 1,314
Available: 1,314
Missing: 0

TEST
Expected: 1,884
Available: 1,884
Missing: 0

------------------------------------------------------------
ALL AUDIO PATHS VERIFIED

STEP 7E COMPLETE


# Step 8 — Whisper Dataset Preparation

## Step 8A — Load Dataset Manifests

Load the prepared Icelandic ASR training, validation, and test
manifests into Hugging Face Datasets.

Preserve the original dataset splits and normalized transcriptions.

In [ ]:
from pathlib import Path
from datasets import load_dataset, DatasetDict

print("=" * 60)
print("STEP 8A — LOAD ICELANDIC DATASET")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")
output_dir = repo / "output"

# Load Person 1's prepared manifests
dataset = load_dataset(
    "json",
    data_files={
        "train": str(output_dir / "train.jsonl"),
        "validation": str(output_dir / "dev.jsonl"),
        "test": str(output_dir / "test.jsonl"),
    }
)

# Convert to DatasetDict
dataset = DatasetDict(dataset)

print("\nDATASET STRUCTURE")
print("-" * 60)
print(dataset)

print("\nSPLIT SIZES")
print("-" * 60)

for split in dataset:
    print(f"{split}: {len(dataset[split]):,} recordings")

print("\nSAMPLE TRANSCRIPTION")
print("-" * 60)

sample = dataset["train"][0]

print("ID:", sample["id"])
print("Audio path:", sample["audio_path"])
print("Transcription:", sample["text"])
print("Duration:", sample["duration_sec"], "seconds")
print("Sample rate:", sample["sample_rate"], "Hz")

# Verify expected split sizes
assert len(dataset["train"]) == 10488
assert len(dataset["validation"]) == 1314
assert len(dataset["test"]) == 1884

print("\nDATASET VALIDATION PASSED")
print("STEP 8A COMPLETE")

STEP 8A — LOAD ICELANDIC DATASET


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


DATASET STRUCTURE
------------------------------------------------------------
DatasetDict({
    train: Dataset({
        features: ['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags'],
        num_rows: 10488
    })
    validation: Dataset({
        features: ['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags'],
        num_rows: 1314
    })
    test: Dataset({
        features: ['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags'],
        num_rows: 1884
    })
})

SPLIT SIZES
------------------------------------------------------------
train: 10,488 recordings
validation: 1,314 recordings
test: 1,884 recordings

SAMPLE TRANSCRIPTION
--------------

## Step 8B — Audio Loading and Feature Extraction Test

Load one Icelandic recording and convert it into Whisper
input features and tokenized transcription labels.

Verify audio decoding, sampling rate, and tensor dimensions
before processing the full dataset.

In [ ]:
import numpy as np
import soundfile as sf
from pathlib import Path

print("=" * 60)
print("STEP 8B — WHISPER FEATURE EXTRACTION TEST")
print("=" * 60)

audio_root = (
    Path("/content/icelandic-asr-project3")
    / "data"
    / "cleaned"
    / "samromur-21.05-20h-v1"
)

# Select one training recording
sample = dataset["train"][0]

audio_path = audio_root / sample["audio_path"]

# Load audio
audio, sample_rate = sf.read(
    str(audio_path),
    dtype="float32"
)

print("\nAUDIO INFORMATION")
print("-" * 60)
print("File:", audio_path.name)
print("Sample rate:", sample_rate)
print("Duration:", round(len(audio) / sample_rate, 2), "seconds")
print("Audio shape:", audio.shape)

assert sample_rate == 16000, "Unexpected sample rate"
assert audio.ndim == 1, "Expected mono audio"
assert np.isfinite(audio).all(), "Audio contains invalid values"

# Extract Whisper input features
input_features = processor.feature_extractor(
    audio,
    sampling_rate=sample_rate,
    return_tensors="pt"
).input_features

# Tokenize transcription
labels = processor.tokenizer(
    sample["text"]
).input_ids

print("\nWHISPER FEATURES")
print("-" * 60)
print("Input feature shape:", tuple(input_features.shape))
print("Number of label tokens:", len(labels))
print("Transcription:", sample["text"])

# Validate features
assert input_features.shape[0] == 1
assert input_features.shape[1] == 80
assert len(labels) > 0

print("\nFEATURE EXTRACTION VALIDATION PASSED")
print("STEP 8B COMPLETE")

STEP 8B — WHISPER FEATURE EXTRACTION TEST

AUDIO INFORMATION
------------------------------------------------------------
File: 007478-0072202.flac
Sample rate: 16000
Duration: 2.82 seconds
Audio shape: (45120,)

WHISPER FEATURES
------------------------------------------------------------
Input feature shape: (1, 80, 3000)
Number of label tokens: 10
Transcription: ég vil vita meira

FEATURE EXTRACTION VALIDATION PASSED
STEP 8B COMPLETE


## Step 8C — Training and Validation Preprocessing

Convert the prepared Icelandic audio recordings into Whisper
log-Mel features and tokenize the corresponding transcriptions.

Preserve the original dataset splits and leave the test set
untouched until final evaluation.

In [ ]:
import numpy as np
import soundfile as sf
from pathlib import Path

print("=" * 60)
print("STEP 8C — WHISPER DATASET PREPROCESSING")
print("=" * 60)

audio_root = (
    Path("/content/icelandic-asr-project3")
    / "data"
    / "cleaned"
    / "samromur-21.05-20h-v1"
)

def prepare_whisper_sample(example):

    audio_path = audio_root / example["audio_path"]

    audio, sample_rate = sf.read(
        str(audio_path),
        dtype="float32"
    )

    if sample_rate != 16000:
        raise ValueError(
            f"Unexpected sample rate: {sample_rate}"
        )

    if audio.ndim != 1:
        raise ValueError(
            f"Expected mono audio: {audio_path}"
        )

    # Extract Whisper log-Mel features
    features = processor.feature_extractor(
        audio,
        sampling_rate=16000
    ).input_features[0]

    # Tokenize transcription
    labels = processor.tokenizer(
        example["text"]
    ).input_ids

    return {
        "input_features": features,
        "labels": labels
    }


print("\nPreparing training dataset...")

train_dataset = dataset["train"].map(
    prepare_whisper_sample,
    remove_columns=dataset["train"].column_names,
    num_proc=1,
    desc="Preparing training audio"
)

print("\nPreparing validation dataset...")

eval_dataset = dataset["validation"].map(
    prepare_whisper_sample,
    remove_columns=dataset["validation"].column_names,
    num_proc=1,
    desc="Preparing validation audio"
)

print("\nPREPROCESSING RESULTS")
print("-" * 60)

print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(eval_dataset):,}")

print("\nTraining features:")
print(train_dataset.features)

assert len(train_dataset) == 10488
assert len(eval_dataset) == 1314

print("\nSTEP 8C COMPLETE")

STEP 8C — WHISPER DATASET PREPROCESSING

Preparing training dataset...


Preparing training audio (num_proc=1):   0%|          | 0/10488 [00:00<?, ? examples/s]


Preparing validation dataset...


Preparing validation audio (num_proc=1):   0%|          | 0/1314 [00:00<?, ? examples/s]


PREPROCESSING RESULTS
------------------------------------------------------------
Training samples: 10,488
Validation samples: 1,314

Training features:
{'input_features': List(List(Value('float32'))), 'labels': List(Value('int64'))}

STEP 8C COMPLETE


# Step 9 — Final Training Configuration

## Step 9A — Calculate Training Steps and Warmup

Calculate the total number of optimizer updates and set
the learning-rate warmup to 5% of the training schedule.

Verify the training configuration before initializing
the Hugging Face Trainer.

In [ ]:
import math

print("=" * 60)
print("STEP 9A — TRAINING SCHEDULE")
print("=" * 60)

# Dataset
num_train_samples = len(train_dataset)

# Training configuration
batch_size = training_args.per_device_train_batch_size
gradient_accumulation = training_args.gradient_accumulation_steps
num_epochs = training_args.num_train_epochs

# Single GPU training
num_gpus = 1

effective_batch_size = (
    batch_size
    * gradient_accumulation
    * num_gpus
)

# Calculate optimizer steps
batches_per_epoch = math.ceil(
    num_train_samples / (batch_size * num_gpus)
)

steps_per_epoch = math.ceil(
    batches_per_epoch / gradient_accumulation
)

total_steps = math.ceil(
    steps_per_epoch * num_epochs
)

# 5% warmup
warmup_steps = math.ceil(total_steps * 0.05)

# Update training arguments
training_args.warmup_steps = warmup_steps

print("\nTRAINING CONFIGURATION")
print("-" * 60)

print(f"Training samples: {num_train_samples:,}")
print(f"Epochs: {num_epochs}")
print(f"Batch size per GPU: {batch_size}")
print(f"Gradient accumulation: {gradient_accumulation}")
print(f"Effective batch size: {effective_batch_size}")

print("\nOPTIMIZER SCHEDULE")
print("-" * 60)

print(f"Batches per epoch: {batches_per_epoch:,}")
print(f"Optimizer steps per epoch: {steps_per_epoch:,}")
print(f"Total optimizer steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

print("\nTRAINING ARGUMENTS")
print("-" * 60)

print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)
print("Gradient checkpointing:", training_args.gradient_checkpointing)
print("Evaluation strategy:", training_args.eval_strategy)
print("Save strategy:", training_args.save_strategy)
print("Warmup steps:", training_args.warmup_steps)

assert training_args.warmup_steps > 0
assert training_args.fp16 is True
assert training_args.load_best_model_at_end is True

print("\nSTEP 9A COMPLETE")

STEP 9A — TRAINING SCHEDULE

TRAINING CONFIGURATION
------------------------------------------------------------
Training samples: 10,488
Epochs: 3
Batch size per GPU: 2
Gradient accumulation: 8
Effective batch size: 16

OPTIMIZER SCHEDULE
------------------------------------------------------------
Batches per epoch: 5,244
Optimizer steps per epoch: 656
Total optimizer steps: 1,968
Warmup steps: 99

TRAINING ARGUMENTS
------------------------------------------------------------
Learning rate: 1e-05
FP16: True
Gradient checkpointing: True
Evaluation strategy: IntervalStrategy.EPOCH
Save strategy: SaveStrategy.EPOCH
Warmup steps: 99

STEP 9A COMPLETE


## Step 9B — WER and CER Evaluation

Configure Word Error Rate (WER) and Character Error Rate (CER)
for evaluating Whisper Small on the Icelandic validation set.

Decode model predictions and reference transcriptions,
then calculate both metrics.

Use WER to select the best training checkpoint.

In [ ]:
import numpy as np
import evaluate

print("=" * 60)
print("STEP 9B — EVALUATION METRICS")
print("=" * 60)

# Load evaluation metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):

    # Predicted token IDs
    pred_ids = pred.predictions

    # Handle models returning additional outputs
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # Reference token IDs
    label_ids = pred.label_ids.copy()

    # Replace ignored labels with padding token
    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    # Decode predictions
    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    # Decode references
    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    # Normalize whitespace
    pred_str = [
        " ".join(text.strip().split())
        for text in pred_str
    ]

    label_str = [
        " ".join(text.strip().split())
        for text in label_str
    ]

    # Calculate metrics
    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }


# Test evaluation function
class DummyPrediction:
    def __init__(self, predictions, label_ids):
        self.predictions = predictions
        self.label_ids = label_ids


reference = "ég vil vita meira"
prediction = "ég vil vita meira"

reference_ids = processor.tokenizer(
    reference
).input_ids

prediction_ids = processor.tokenizer(
    prediction
).input_ids

# Pad sequences for testing
max_length = max(
    len(reference_ids),
    len(prediction_ids)
)

pad_id = processor.tokenizer.pad_token_id

reference_ids += [pad_id] * (
    max_length - len(reference_ids)
)

prediction_ids += [pad_id] * (
    max_length - len(prediction_ids)
)

dummy = DummyPrediction(
    predictions=np.array([prediction_ids]),
    label_ids=np.array([reference_ids])
)

results = compute_metrics(dummy)

print("\nEVALUATION TEST")
print("-" * 60)

print("Reference:", reference)
print("Prediction:", prediction)

print(f"WER: {results['wer']:.4f}")
print(f"CER: {results['cer']:.4f}")

assert results["wer"] == 0.0
assert results["cer"] == 0.0

print("\nEVALUATION METRICS VERIFIED")
print("STEP 9B COMPLETE")

STEP 9B — EVALUATION METRICS


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



EVALUATION TEST
------------------------------------------------------------
Reference: ég vil vita meira
Prediction: ég vil vita meira
WER: 0.0000
CER: 0.0000

EVALUATION METRICS VERIFIED
STEP 9B COMPLETE


## Step 9C — Initialize Whisper Trainer

Initialize the Hugging Face Seq2SeqTrainer using the
preprocessed Icelandic dataset, Whisper Small model,
training configuration, data collator, and evaluation metrics.

Verify the training environment before fine-tuning.

In [ ]:
import torch
from transformers import Seq2SeqTrainer

print("=" * 60)
print("STEP 9C — WHISPER TRAINER INITIALIZATION")
print("=" * 60)

# Disable BPE token cleanup
processor.tokenizer.clean_up_tokenization_spaces = False

# Ensure generation-based evaluation is enabled
training_args.predict_with_generate = True

# Configure Whisper generation
model.generation_config.language = "icelandic"
model.generation_config.task = "transcribe"

# Disable cache during gradient checkpointing
model.config.use_cache = False

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("\nTRAINER CONFIGURATION")
print("-" * 60)

print("Model:", model.config.model_type)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))
print("Epochs:", trainer.args.num_train_epochs)
print("Batch size:", trainer.args.per_device_train_batch_size)
print("Gradient accumulation:", trainer.args.gradient_accumulation_steps)
print("Learning rate:", trainer.args.learning_rate)
print("Warmup steps:", trainer.args.warmup_steps)
print("FP16:", trainer.args.fp16)
print("Gradient checkpointing:", trainer.args.gradient_checkpointing)
print("Generate predictions:", trainer.args.predict_with_generate)
print("Checkpoint directory:", trainer.args.output_dir)

print("\nGPU")
print("-" * 60)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

assert len(trainer.train_dataset) == 10488
assert len(trainer.eval_dataset) == 1314
assert trainer.args.warmup_steps == 99
assert trainer.args.predict_with_generate is True
assert torch.cuda.is_available()

print("\nTRAINER INITIALIZATION PASSED")
print("STEP 9C COMPLETE")

STEP 9C — WHISPER TRAINER INITIALIZATION

TRAINER CONFIGURATION
------------------------------------------------------------
Model: whisper
Training samples: 10488
Validation samples: 1314
Epochs: 3
Batch size: 2
Gradient accumulation: 8
Learning rate: 1e-05
Warmup steps: 99
FP16: True
Gradient checkpointing: True
Generate predictions: True
Checkpoint directory: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/checkpoints

GPU
------------------------------------------------------------
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

TRAINER INITIALIZATION PASSED
STEP 9C COMPLETE


## Step 9D — GPU Training Compatibility Test

Test one forward and backward pass using the actual
Icelandic training dataset.

Verify GPU memory usage, loss calculation, and gradient
computation before starting full fine-tuning.

No optimizer step is performed during this test.

# Step 10 — Whisper Small Fine-Tuning

## Step 10A — Checkpoint and Recovery Configuration

Configure periodic checkpoint saving and validation
during fine-tuning.

Save checkpoints to Google Drive so training can resume
after a Colab disconnection.

Retain the latest checkpoints and the best model
according to validation WER.

In [ ]:
from pathlib import Path
from transformers import IntervalStrategy

print("=" * 60)
print("STEP 10A — CHECKPOINT RECOVERY CONFIGURATION")
print("=" * 60)

# Google Drive checkpoint directory
checkpoint_dir = Path(
    "/content/drive/MyDrive/"
    "NLP_Project3_Whisper_FineTuning/checkpoints"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Configure periodic evaluation and saving
training_args.eval_strategy = IntervalStrategy.STEPS
training_args.save_strategy = IntervalStrategy.STEPS

training_args.eval_steps = 200
training_args.save_steps = 200

# Retain a limited number of checkpoints
training_args.save_total_limit = 2

# Select best model using validation WER
training_args.load_best_model_at_end = True
training_args.metric_for_best_model = "wer"
training_args.greater_is_better = False

# Keep generation-based evaluation enabled
training_args.predict_with_generate = True

# Reinitialize Trainer with updated arguments
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("\nCHECKPOINT CONFIGURATION")
print("-" * 60)

print("Checkpoint directory:", trainer.args.output_dir)
print("Save strategy:", trainer.args.save_strategy)
print("Save steps:", trainer.args.save_steps)
print("Evaluation strategy:", trainer.args.eval_strategy)
print("Evaluation steps:", trainer.args.eval_steps)
print("Checkpoint retention:", trainer.args.save_total_limit)
print("Best model metric:", trainer.args.metric_for_best_model)
print("Lower metric is better:", not trainer.args.greater_is_better)

print("\nEXISTING CHECKPOINTS")
print("-" * 60)

checkpoints = sorted(
    checkpoint_dir.glob("checkpoint-*")
)

if checkpoints:
    for checkpoint in checkpoints:
        print(checkpoint.name)
else:
    print("No previous checkpoints found.")

# Validation
assert trainer.args.save_steps == 200
assert trainer.args.eval_steps == 200
assert trainer.args.save_total_limit == 2
assert trainer.args.load_best_model_at_end
assert trainer.args.metric_for_best_model == "wer"

print("\nCHECKPOINT CONFIGURATION VERIFIED")
print("STEP 10A COMPLETE")

STEP 10A — CHECKPOINT RECOVERY CONFIGURATION

CHECKPOINT CONFIGURATION
------------------------------------------------------------
Checkpoint directory: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/checkpoints
Save strategy: IntervalStrategy.STEPS
Save steps: 200
Evaluation strategy: IntervalStrategy.STEPS
Evaluation steps: 200
Checkpoint retention: 2
Best model metric: wer
Lower metric is better: True

EXISTING CHECKPOINTS
------------------------------------------------------------
No previous checkpoints found.

CHECKPOINT CONFIGURATION VERIFIED
STEP 10A COMPLETE


> **Archived training run:** The output below records the original completed 1,968-step training run (5.20 hours). **Do not rerun this cell** during model inspection; doing so may start or resume expensive training. The original best validation WER was 30.62% at step 1600, but those weights were not recovered.



## Step 10B — Whisper Small Fine-Tuning

Fine-tune multilingual Whisper Small on the Icelandic
Samrómur dataset.

Training configuration:
- 3 epochs
- Learning rate: 1e-5
- Effective batch size: 16
- FP16 mixed precision
- Gradient checkpointing
- Checkpoints every 200 optimizer steps
- Validation using WER and CER

Record training duration, GPU information, and final metrics.

In [ ]:
import time
import json
import torch
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint

print("=" * 60)
print("STEP 10B — WHISPER SMALL FINE-TUNING")
print("=" * 60)

# Directories
project_dir = Path(
    "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning"
)

checkpoint_dir = project_dir / "checkpoints"
results_dir = project_dir / "results"

results_dir.mkdir(parents=True, exist_ok=True)

# Verify GPU
assert torch.cuda.is_available(), "GPU unavailable!"

gpu_name = torch.cuda.get_device_name(0)

print("\nTRAINING INFORMATION")
print("-" * 60)

print("GPU:", gpu_name)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(eval_dataset))
print("Epochs:", trainer.args.num_train_epochs)
print("Learning rate:", trainer.args.learning_rate)
print("Effective batch size:",
      trainer.args.per_device_train_batch_size *
      trainer.args.gradient_accumulation_steps)

# Check for previous checkpoints
last_checkpoint = get_last_checkpoint(
    str(checkpoint_dir)
)

if last_checkpoint:
    print("\nExisting checkpoint found:")
    print(last_checkpoint)
    print("Training will resume from this checkpoint.")
else:
    print("\nNo checkpoint found.")
    print("Starting fine-tuning from the pretrained model.")

# Record starting time
start_time = time.time()

print("\nSTARTING TRAINING...")
print("=" * 60)

# Train or resume
train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)

# Record training duration
training_duration = time.time() - start_time

print("\nTRAINING COMPLETED")
print("-" * 60)

print(
    f"Training duration: {training_duration / 3600:.2f} hours"
)

print("Final training loss:", train_result.training_loss)

# Save training metrics
training_results = {
    "model": "openai/whisper-small",
    "language": "Icelandic",
    "gpu": gpu_name,
    "training_samples": len(train_dataset),
    "validation_samples": len(eval_dataset),
    "epochs": trainer.args.num_train_epochs,
    "learning_rate": trainer.args.learning_rate,
    "effective_batch_size": (
        trainer.args.per_device_train_batch_size *
        trainer.args.gradient_accumulation_steps
    ),
    "training_duration_seconds": training_duration,
    "training_loss": train_result.training_loss,
    "global_step": trainer.state.global_step
}

results_path = results_dir / "training_results.json"

with open(results_path, "w") as f:
    json.dump(training_results, f, indent=4)

print("\nTraining results saved to:")
print(results_path)

print("\nSTEP 10B COMPLETE")

STEP 10B — WHISPER SMALL FINE-TUNING

TRAINING INFORMATION
------------------------------------------------------------
GPU: Tesla T4
Training samples: 10488
Validation samples: 1314
Epochs: 3
Learning rate: 1e-05
Effective batch size: 16

No checkpoint found.
Starting fine-tuning from the pretrained model.

STARTING TRAINING...


Step,Training Loss,Validation Loss,Wer,Cer
200,5.860062,0.734294,0.509447,0.159794
400,4.098164,0.568799,0.424523,0.122550
600,3.727689,0.494144,0.382061,0.108491
800,2.202216,0.450531,0.368893,0.108507
1000,2.112906,0.429902,0.345134,0.096969
1200,2.085521,0.403523,0.322519,0.087826
1400,1.055174,0.401283,0.315935,0.086731
1600,1.036301,0.394304,0.306202,0.084147


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



TRAINING COMPLETED
------------------------------------------------------------
Training duration: 5.20 hours
Final training loss: 2.9130900536126236

Training results saved to:
/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/results/training_results.json

STEP 10B COMPLETE


## Step 11 — Recovered Model and Inference

The original run finished 1,968 optimizer steps. The best recorded validation WER was **30.62% at step 1600**; its checkpoint is unavailable. The independently exported **step-1000 recovery model** has validation WER **34.51%** and CER **9.70%**. Do not attribute the step-1600 result to the exported weights. The model is already stored in Google Drive at `NLP_Project3_Whisper_FineTuning/recovered_model_step1000`.

This section loads that existing export and runs a **single validation-recording smoke test**, not a held-out test-set evaluation. It requires only one FLAC recording.


In [ ]:
from pathlib import Path
import json, subprocess, os
import torch, soundfile as sf
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

repo = Path("/content/icelandic-asr-project3")
model_dir = Path("/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/recovered_model_step1000")
assert model_dir.is_dir(), "Mount Google Drive using Step 4 first."
if not (repo / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/RanimAlyaseen/icelandic-asr-project3.git", str(repo)], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})

with open(repo / "output/dev.jsonl", encoding="utf-8") as f:
    sample = json.loads(f.readline())
relative_audio = "data/cleaned/samromur-21.05-20h-v1/" + sample["audio_path"]
audio_path = repo / relative_audio
if not audio_path.exists() or audio_path.stat().st_size < 200:
    subprocess.run(["git", "lfs", "pull", "--include", relative_audio, "--exclude", ""], cwd=repo, check=True)
assert audio_path.exists(), f"Audio missing: {audio_path}"
with open(audio_path, "rb") as f:
    assert f.read(4) == b"fLaC", "Audio is not a downloaded FLAC file."

processor = WhisperProcessor.from_pretrained(str(model_dir), language="icelandic", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(str(model_dir), dtype=torch.float32).to("cpu").eval()
audio, sr = sf.read(str(audio_path))
if audio.ndim > 1: audio = audio.mean(axis=1)
assert sr == 16000, f"Unexpected sample rate: {sr}"
inputs = processor(audio, sampling_rate=sr, return_tensors="pt", return_attention_mask=True)
with torch.inference_mode():
    ids = model.generate(inputs.input_features, attention_mask=inputs.attention_mask, language="icelandic", task="transcribe", max_new_tokens=128)
prediction = processor.batch_decode(ids, skip_special_tokens=True)[0]
print("REFERENCE:", sample["text"])
print("PREDICTION:", prediction)
print(f"Single-sample WER: {wer(sample['text'], prediction):.2%}")
print(f"Single-sample CER: {cer(sample['text'], prediction):.2%}")
print("Note: Single-sample metrics are not the final test-set scores.")


In [2]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

if not repo.exists():
    subprocess.run([
        "git", "clone",
        "--branch", "person3-whisper-reproducibility",
        "--depth", "1",
        "https://github.com/RanimAlyaseen/icelandic-asr-project3.git",
        str(repo)
    ], check=True)

subprocess.run([
    "git", "-C", str(repo),
    "fetch", "origin", "main"
], check=True)

print("Repository ready for notebook comparison.")

Repository ready for notebook comparison.


In [3]:
import json
import subprocess
from pathlib import Path
from collections import Counter

repo = Path("/content/icelandic-asr-merge-check")
notebook = "NLP_Project3_Person3_Whisper_Clean.ipynb"

def load_notebook(ref):
    result = subprocess.run(
        ["git", "-C", str(repo), "show", f"{ref}:{notebook}"],
        capture_output=True,
        text=True,
        check=True
    )
    return json.loads(result.stdout)

main = load_notebook("FETCH_HEAD")
branch = load_notebook("HEAD")

def cell_text(cell):
    source = cell.get("source", [])
    return "".join(source) if isinstance(source, list) else source

def summarize(nb):
    cells = nb["cells"]
    return {
        "total_cells": len(cells),
        "code_cells": sum(c["cell_type"] == "code" for c in cells),
        "markdown_cells": sum(c["cell_type"] == "markdown" for c in cells),
        "cells_with_outputs": sum(
            bool(c.get("outputs")) for c in cells
        ),
    }

main_cells = [cell_text(c) for c in main["cells"]]
branch_cells = [cell_text(c) for c in branch["cells"]]

main_counts = Counter(main_cells)
branch_counts = Counter(branch_cells)

only_main = list((main_counts - branch_counts).elements())
only_branch = list((branch_counts - main_counts).elements())

print("MAIN:", summarize(main))
print("BRANCH:", summarize(branch))

print("\nCells present only in main:", len(only_main))
print("Cells present only in branch:", len(only_branch))

print("\n--- MAIN-ONLY CELL PREVIEWS ---")
for i, source in enumerate(only_main, 1):
    print(f"\n{i}. {source[:350]}")

print("\n--- BRANCH-ONLY CELL PREVIEWS ---")
for i, source in enumerate(only_branch, 1):
    print(f"\n{i}. {source[:350]}")

MAIN: {'total_cells': 50, 'code_cells': 24, 'markdown_cells': 26, 'cells_with_outputs': 20}
BRANCH: {'total_cells': 52, 'code_cells': 25, 'markdown_cells': 27, 'cells_with_outputs': 20}

Cells present only in main: 0
Cells present only in branch: 2

--- MAIN-ONLY CELL PREVIEWS ---

--- BRANCH-ONLY CELL PREVIEWS ---

1. ### Model and Dataset Version Tracking

**Base model:** `openai/whisper-small` (multilingual)

**Original base-model revision:** Not recorded during the original training run.

**Dataset:** Samrómur 21.05 (OpenSLR 112)

**Dataset selection:** `samromur-21.05-20h-v1`

**Dataset preparation reference:** Git commit `55f5a68`

**Original dataset checko

2. # Record exact revisions for future reproducible runs.
# This cell does not load the model or start training.

import subprocess
from huggingface_hub import HfApi

BASE_MODEL_ID = "openai/whisper-small"

# Resolve the current Hugging Face model revision.
BASE_MODEL_REVISION = HfApi().model_info(BASE_MODEL_ID).sha

# Record

In [4]:
import json

main_cells = main["cells"]
branch_cells = branch["cells"]

def source(cell):
    value = cell.get("source", [])
    return "".join(value) if isinstance(value, list) else value

def normalized(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False)

# Match cells by their source and type.
remaining = branch_cells.copy()
differences = []

for i, cell in enumerate(main_cells):
    match = next(
        (
            candidate for candidate in remaining
            if candidate["cell_type"] == cell["cell_type"]
            and source(candidate) == source(cell)
        ),
        None
    )

    if match is None:
        differences.append((i, "Cell missing from branch"))
        continue

    remaining.remove(match)

    if cell["cell_type"] == "code":
        if normalized(cell.get("outputs", [])) != normalized(match.get("outputs", [])):
            differences.append((i, "Outputs differ"))

        if cell.get("execution_count") != match.get("execution_count"):
            differences.append((i, "Execution count differs"))

print("Existing cells checked:", len(main_cells))
print("Differences found:", len(differences))

for index, description in differences[:20]:
    print(f"Cell {index}: {description}")

Existing cells checked: 50
Differences found: 0


In [5]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

# Configure Git identity for the merge commit
git("config", "user.name", "tasneemzouhair-coder")
git("config", "user.email", "2309015259+tasneemzouhair-coder@users.noreply.github.com")

# Merge main without automatically committing
result = subprocess.run(
    ["git", "-C", str(repo), "merge", "--no-commit", "--no-ff", "FETCH_HEAD"],
    capture_output=True,
    text=True
)

print("Merge status:", result.returncode)
print(result.stdout)
print(result.stderr)

# Keep the verified notebook from your feature branch
git("checkout", "--ours", "--", "NLP_Project3_Person3_Whisper_Clean.ipynb")
git("add", "NLP_Project3_Person3_Whisper_Clean.ipynb")

print("\nRemaining conflicts:")
print(git("diff", "--name-only", "--diff-filter=U"))

print("\nRepository status:")
print(git("status", "--short"))

Merge status: 128

fatal: refusing to merge unrelated histories


Remaining conflicts:


Repository status:



In [6]:
import subprocess

repo = "/content/icelandic-asr-merge-check"

result = subprocess.run(
    ["git", "-C", repo, "fetch", "--unshallow", "origin"],
    capture_output=True,
    text=True
)

print("Fetch status:", result.returncode)
print(result.stdout)
print(result.stderr)

if result.returncode == 0:
    result = subprocess.run(
        ["git", "-C", repo, "merge-base", "HEAD", "origin/main"],
        capture_output=True,
        text=True
    )

    print("Common ancestor:", result.stdout.strip())
    print("Merge-base status:", result.returncode)

Fetch status: 0


Common ancestor: 
Merge-base status: 128


In [7]:
import subprocess

repo = "/content/icelandic-asr-merge-check"

commands = [
    ["git", "status", "--short", "--branch"],
    ["git", "rev-parse", "--is-shallow-repository"],
    ["git", "branch", "-a"],
    ["git", "rev-parse", "HEAD"],
    ["git", "rev-parse", "FETCH_HEAD"],
    ["git", "rev-parse", "--verify", "refs/remotes/origin/main"],
    ["git", "merge-base", "HEAD", "FETCH_HEAD"],
]

for command in commands:
    result = subprocess.run(
        command,
        cwd=repo,
        capture_output=True,
        text=True
    )

    print("\nCOMMAND:", " ".join(command))
    print("Exit code:", result.returncode)
    print("Output:", result.stdout.strip())

    if result.stderr.strip():
        print("Error:", result.stderr.strip())


COMMAND: git status --short --branch
Exit code: 0
Output: ## person3-whisper-reproducibility...origin/person3-whisper-reproducibility

COMMAND: git rev-parse --is-shallow-repository
Exit code: 0
Output: false

COMMAND: git branch -a
Exit code: 0
Output: * person3-whisper-reproducibility
  remotes/origin/person3-whisper-reproducibility

COMMAND: git rev-parse HEAD
Exit code: 0
Output: 61baca258fa939bfd16af55545a3600bc2e40098

COMMAND: git rev-parse FETCH_HEAD
Exit code: 0
Output: 61baca258fa939bfd16af55545a3600bc2e40098

COMMAND: git rev-parse --verify refs/remotes/origin/main
Exit code: 128
Output: 
Error: fatal: Needed a single revision

COMMAND: git merge-base HEAD FETCH_HEAD
Exit code: 0
Output: 61baca258fa939bfd16af55545a3600bc2e40098


In [8]:
import subprocess

repo = "/content/icelandic-asr-merge-check"

commands = [
    [
        "git", "fetch", "origin",
        "refs/heads/main:refs/remotes/origin/main"
    ],
    [
        "git", "merge-base",
        "HEAD", "origin/main"
    ],
]

for command in commands:
    result = subprocess.run(
        command,
        cwd=repo,
        capture_output=True,
        text=True
    )

    print("\nCOMMAND:", " ".join(command))
    print("Exit code:", result.returncode)
    print("Output:", result.stdout.strip())

    if result.stderr.strip():
        print("Error:", result.stderr.strip())

    if result.returncode != 0:
        break


COMMAND: git fetch origin refs/heads/main:refs/remotes/origin/main
Exit code: 0
Output: 
Error: From https://github.com/RanimAlyaseen/icelandic-asr-project3
 * [new branch]        main       -> origin/main

COMMAND: git merge-base HEAD origin/main
Exit code: 0
Output: b0612175093817d0a9fe9a35211a09a6bd4056f3


In [9]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook = "NLP_Project3_Person3_Whisper_Clean.ipynb"

def run_git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True
    )

# 1. Start the merge without creating a commit
result = run_git(
    "merge", "--no-commit", "--no-ff", "origin/main"
)

print("MERGE EXIT CODE:", result.returncode)
print(result.stdout)
print(result.stderr)

# 2. Identify all conflicting files
conflicts = run_git(
    "diff", "--name-only", "--diff-filter=U"
).stdout.splitlines()

print("\nCONFLICTING FILES:", conflicts)

# 3. Resolve only the previously verified notebook conflict
if conflicts == [notebook]:
    result = run_git("checkout", "--ours", "--", notebook)

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    result = run_git("add", notebook)

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    print("\nVerified notebook version retained.")
elif conflicts:
    print("\nUnexpected conflicts detected. No files were resolved.")
else:
    print("\nNo file conflicts detected.")

# 4. Check the repository state
print("\nREMAINING CONFLICTS:")
print(run_git("diff", "--name-only", "--diff-filter=U").stdout)

print("GIT STATUS:")
print(run_git("status", "--short").stdout)

MERGE EXIT CODE: 1
Auto-merging NLP_Project3_Person3_Whisper_Clean.ipynb
CONFLICT (content): Merge conflict in NLP_Project3_Person3_Whisper_Clean.ipynb
Automatic merge failed; fix conflicts and then commit the result.



CONFLICTING FILES: ['NLP_Project3_Person3_Whisper_Clean.ipynb']

Verified notebook version retained.

REMAINING CONFLICTS:

GIT STATUS:



In [10]:
import json
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

# 1. Validate notebook JSON
with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook["cells"]

print("NOTEBOOK VALIDATION")
print("Valid JSON: PASS")
print("Total cells:", len(cells))
print("Code cells:", sum(c["cell_type"] == "code" for c in cells))
print("Markdown cells:", sum(c["cell_type"] == "markdown" for c in cells))

# 2. Verify reproducibility documentation
sources = [
    "".join(c.get("source", []))
    for c in cells
]

checks = {
    "Model and Dataset Version Tracking":
        any("Model and Dataset Version Tracking" in s for s in sources),
    "Revision recording code":
        any("BASE_MODEL_REVISION" in s for s in sources),
}

print("\nREPRODUCIBILITY CHECKS")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

# 3. Check merge state
def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True
    )

merge_head = git("rev-parse", "-q", "--verify", "MERGE_HEAD")
conflicts = git("diff", "--name-only", "--diff-filter=U")

print("\nGIT VALIDATION")
print("Merge in progress:", merge_head.returncode == 0)
print("Unresolved conflicts:", conflicts.stdout.strip() or "None")

status = git("status")
print("\nGIT STATUS")
print(status.stdout)

assert len(cells) == 52, "Unexpected notebook cell count"
assert all(checks.values()), "Missing reproducibility cells"
assert merge_head.returncode == 0, "Merge is not in progress"
assert not conflicts.stdout.strip(), "Unresolved conflicts remain"

print("\nFINAL RESULT: PASS")

NOTEBOOK VALIDATION
Valid JSON: PASS
Total cells: 52
Code cells: 25
Markdown cells: 27

REPRODUCIBILITY CHECKS
Model and Dataset Version Tracking: PASS
Revision recording code: PASS

GIT VALIDATION
Merge in progress: True
Unresolved conflicts: None

GIT STATUS
On branch person3-whisper-reproducibility
Your branch is up to date with 'origin/person3-whisper-reproducibility'.

All conflicts fixed but you are still merging.
  (use "git commit" to conclude merge)



FINAL RESULT: PASS


In [11]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

# Configure Git identity
subprocess.run(
    ["git", "-C", str(repo), "config",
     "user.name", "tasneemzouhair-coder"],
    check=True
)

subprocess.run(
    ["git", "-C", str(repo), "config",
     "user.email",
     "tasneemzouhair-coder@users.noreply.github.com"],
    check=True
)

# Complete the merge
result = subprocess.run(
    [
        "git", "-C", str(repo),
        "commit", "-m",
        "Merge main and resolve Whisper notebook conflict"
    ],
    capture_output=True,
    text=True
)

print("COMMIT EXIT CODE:", result.returncode)
print(result.stdout)
print(result.stderr)

# Verify repository status
status = subprocess.run(
    ["git", "-C", str(repo), "status", "--short", "--branch"],
    capture_output=True,
    text=True
)

print("\nREPOSITORY STATUS:")
print(status.stdout)

# Show latest commit
latest = subprocess.run(
    ["git", "-C", str(repo), "log", "-1", "--oneline"],
    capture_output=True,
    text=True
)

print("LATEST COMMIT:")
print(latest.stdout)

COMMIT EXIT CODE: 0
[person3-whisper-reproducibility 1d184e5e] Merge main and resolve Whisper notebook conflict



REPOSITORY STATUS:
## person3-whisper-reproducibility...origin/person3-whisper-reproducibility [ahead 2]

LATEST COMMIT:
1d184e5e Merge main and resolve Whisper notebook conflict



In [12]:
import subprocess

repo = "/content/icelandic-asr-merge-check"

# Check remote configuration without exposing credentials
result = subprocess.run(
    ["git", "-C", repo, "remote", "get-url", "origin"],
    capture_output=True,
    text=True
)

remote = result.stdout.strip()

if "github.com" in remote:
    print("GitHub remote: CONFIGURED")
    print("Protocol:", "HTTPS" if remote.startswith("https://") else "SSH")
else:
    print("GitHub remote: UNEXPECTED CONFIGURATION")

# Check whether GitHub CLI authentication is available
result = subprocess.run(
    ["gh", "auth", "status"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("GitHub CLI authentication: AVAILABLE")
else:
    print("GitHub CLI authentication: NOT AVAILABLE")

# Check whether Git has a credential helper configured
result = subprocess.run(
    ["git", "-C", repo, "config", "--get", "credential.helper"],
    capture_output=True,
    text=True
)

print(
    "Git credential helper:",
    "CONFIGURED" if result.stdout.strip() else "NOT CONFIGURED"
)

GitHub remote: CONFIGURED
Protocol: HTTPS
GitHub CLI authentication: NOT AVAILABLE
Git credential helper: NOT CONFIGURED


In [1]:
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

print("Previous repository exists:", repo.exists())

Previous repository exists: False


In [2]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

# Prevent downloading large audio files through Git LFS
env = os.environ.copy()
env["GIT_LFS_SKIP_SMUDGE"] = "1"

subprocess.run(
    [
        "git", "clone",
        "--branch", "person3-whisper-reproducibility",
        "https://github.com/RanimAlyaseen/icelandic-asr-project3.git",
        str(repo)
    ],
    check=True,
    env=env
)

# Fetch main explicitly
subprocess.run(
    [
        "git", "-C", str(repo),
        "fetch", "origin",
        "refs/heads/main:refs/remotes/origin/main"
    ],
    check=True,
    env=env
)

print("\nRepository restored successfully.")

subprocess.run(
    ["git", "-C", str(repo), "status", "--short", "--branch"],
    check=True
)


Repository restored successfully.


CompletedProcess(args=['git', '-C', '/content/icelandic-asr-merge-check', 'status', '--short', '--branch'], returncode=0)

In [3]:
import json
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

print("Total cells:", len(notebook["cells"]))

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(term in source.lower() for term in [
        "/content/drive",
        "checkpoint-1000",
        "from_pretrained",
        "download_whisper_checkpoint"
    ]):
        print(f"\n{'=' * 60}")
        print(f"CELL {i}")
        print(f"{'=' * 60}")
        print(source[:6000])

Total cells: 52

CELL 9
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-small"

print("=" * 60)
print("WHISPER SMALL — MODEL INITIALIZATION")
print("=" * 60)

# Load processor for Icelandic transcription
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language="icelandic",
    task="transcribe"
)

print("\nProcessor loaded successfully.")

# Load pretrained model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID
)

print("Model loaded successfully.")

# Configure transcription language
model.generation_config.language = "icelandic"
model.generation_config.task = "transcribe"

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("\nMODEL INFORMATION")
print("-" * 60)

print(f"Model ID: {MODEL_ID}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable paramete

In [4]:
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
downloader = repo / "training/download_whisper_checkpoint.py"

assert downloader.exists(), "Checkpoint downloader not found."

print("CHECKPOINT DOWNLOADER")
print("=" * 60)
print(downloader.read_text(encoding="utf-8"))

CHECKPOINT DOWNLOADER
"""Download and verify the recovered Whisper Small checkpoint 1000."""

import argparse
import hashlib
from pathlib import Path

import gdown

FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    "1BKWu0PBXdNlp7pqeX7MCn9lhzFsay_Md"
)

EXPECTED_SHA256 = {
    "config.json": "572ff2d955ca237231a371e756e41b4a4c0c63df3c30e64538b90294f468e449",
    "generation_config.json": "2d3e60d85c13324649773e42932b2a9916edd3ff0f24434b17db657c61c57fd0",
    "model.safetensors": "d056c430a0b4c1d9006a9ddf68a0b8d92f1cf4828ccb4f5bd96f77d5f911f80b",
    "processor_config.json": "8c648fa85aafc59a07435ed751322499bd4238b819ca9200ddd2b5ba4df50e68",
    "recovery_metadata.json": "21e255d551be5020ca1c0304b8b25b6bf88c018928bc6768b0b5cd341bff7f09",
    "tokenizer.json": "783ab7faf82595b8abc49cef57865b055d763753e6383d0d98774abcfe16e57a",
    "tokenizer_config.json": "0940f4bfe73310f1fdc9f86242568178700211fde2d9b49b15dd1ad892552641",
}


def sha256_file(path):
    digest = hashlib.sha

In [5]:
import json
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

cell = notebook["cells"][51]
source = "".join(cell["source"])

old = '''model_dir = Path("/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/recovered_model_step1000")
assert model_dir.is_dir(), "Mount Google Drive using Step 4 first."'''

new = '''# Download and verify the recovered checkpoint automatically.
model_dir = repo / "models/recovered_model_step1000"

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "gdown"],
    check=True
)

subprocess.run(
    [
        "python",
        str(repo / "training/download_whisper_checkpoint.py"),
        "--output",
        str(model_dir),
    ],
    check=True
)'''

assert old in source, "Expected checkpoint-loading code not found."
assert source.count(old) == 1, "Unexpected duplicate checkpoint-loading code."

source = source.replace(old, new)

# Save the updated source without changing other notebook cells.
cell["source"] = source.splitlines(keepends=True)

with notebook_path.open("w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=1)
    f.write("\n")

print("Cell 51 updated successfully.")
print("Personal checkpoint path removed:", old not in source)
print("Automatic downloader added:",
      "download_whisper_checkpoint.py" in source)
print("Total notebook cells:", len(notebook["cells"]))

Cell 51 updated successfully.
Personal checkpoint path removed: True
Automatic downloader added: True
Total notebook cells: 52


In [6]:
import json
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

cell = notebook["cells"][51]
source = "".join(cell["source"])

start = source.index("# Download and verify the recovered checkpoint automatically.")
end = source.index('if not (repo / ".git").exists():', start)

# Extract the downloader block
download_block = source[start:end]

# Remove it from its original position
source = source[:start] + source[end:]

# Find the repository-cloning statement
clone_statement = '''    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/RanimAlyaseen/icelandic-asr-project3.git", str(repo)], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})'''

assert clone_statement in source, "Repository setup code not found."

# Insert downloader after repository setup
source = source.replace(
    clone_statement,
    clone_statement + "\n\n" + download_block.rstrip()
)

cell["source"] = source.splitlines(keepends=True)

with notebook_path.open("w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=1)
    f.write("\n")

# Verify execution order
updated = "".join(notebook["cells"][51]["source"])

clone_position = updated.index('if not (repo / ".git").exists():')
download_position = updated.index("download_whisper_checkpoint.py")
evaluation_position = updated.index("WhisperProcessor.from_pretrained(str(model_dir)")

assert clone_position < download_position < evaluation_position

print("Execution order: PASS")
print("1. Clone repository")
print("2. Download and verify checkpoint")
print("3. Load model and run evaluation")
print("\nTotal notebook cells:", len(notebook["cells"]))
print("Personal checkpoint path removed:",
      "recovered_model_step1000" in updated
      and "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/recovered_model_step1000" not in updated)

Execution order: PASS
1. Clone repository
2. Download and verify checkpoint
3. Load model and run evaluation

Total notebook cells: 52
Personal checkpoint path removed: True


In [7]:
import json
import ast
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

# Check notebook structure
assert len(notebook["cells"]) == 52

# Validate the final evaluation cell's Python syntax
source = "".join(notebook["cells"][51]["source"])
ast.parse(source)

# Check required components
checks = {
    "Valid notebook JSON": True,
    "All 52 cells preserved": len(notebook["cells"]) == 52,
    "Evaluation cell syntax valid": True,
    "Automatic downloader included":
        "download_whisper_checkpoint.py" in source,
    "SHA256 verification enabled":
        (repo / "training/download_whisper_checkpoint.py").exists(),
    "Personal checkpoint path removed":
        "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/recovered_model_step1000"
        not in source,
}

print("NOTEBOOK VALIDATION")
print("=" * 50)

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

assert all(checks.values())

# Check Git status
result = subprocess.run(
    ["git", "-C", str(repo), "status", "--short"],
    capture_output=True,
    text=True,
    check=True
)

print("\nGIT STATUS")
print(result.stdout or "Clean")

print("\nFINAL RESULT: PASS")

NOTEBOOK VALIDATION
Valid notebook JSON: PASS
All 52 cells preserved: PASS
Evaluation cell syntax valid: PASS
Automatic downloader included: PASS
SHA256 verification enabled: PASS
Personal checkpoint path removed: PASS

GIT STATUS
 M NLP_Project3_Person3_Whisper_Clean.ipynb


FINAL RESULT: PASS


In [8]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

# Configure commit identity
git("config", "user.name", "tasneemzouhair-coder")
git(
    "config",
    "user.email",
    "tasneemzouhair-coder@users.noreply.github.com"
)

# Stage only the modified notebook
git("add", "NLP_Project3_Person3_Whisper_Clean.ipynb")

# Commit the evaluation fix
print(git(
    "commit",
    "-m",
    "Use automatic checkpoint downloader in Whisper evaluation"
))

# Verify the commit
print("\nLATEST COMMIT:")
print(git("log", "-1", "--oneline"))

print("\nREPOSITORY STATUS:")
print(git("status", "--short", "--branch"))

[person3-whisper-reproducibility 741e40b6] Use automatic checkpoint downloader in Whisper evaluation
 1 file changed, 14203 insertions(+), 14187 deletions(-)

LATEST COMMIT:
741e40b6 Use automatic checkpoint downloader in Whisper evaluation

REPOSITORY STATUS:
## person3-whisper-reproducibility...origin/person3-whisper-reproducibility [ahead 1]


In [10]:
import subprocess
import difflib
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

def read_git_version(stage):
    result = subprocess.run(
        ["git", "-C", str(repo), "show", f":{stage}:README.md"],
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout

# Stage 2 = our feature branch
# Stage 3 = latest main
ours = read_git_version(2)
theirs = read_git_version(3)

diff = list(difflib.unified_diff(
    ours.splitlines(),
    theirs.splitlines(),
    fromfile="Our PR README",
    tofile="Latest main README",
    lineterm=""
))

print("OUR README:", len(ours.splitlines()), "lines")
print("MAIN README:", len(theirs.splitlines()), "lines")
print("\nREADME DIFFERENCES")
print("=" * 60)

for line in diff[:200]:
    print(line)

if len(diff) > 200:
    print(f"\n... {len(diff) - 200} additional diff lines omitted.")

OUR README: 602 lines
MAIN README: 495 lines

README DIFFERENCES
--- Our PR README
+++ Latest main README
@@ -42,16 +42,18 @@
 
 Use the hydrated cleaned LFS root, the local full release, or the **audio/** directory of the optional LF handoff. Each has train, dev, and test directories and works with the frozen manifests. The Windows verification environment used Python 3.14.10, NumPy 2.5.5, SoundFile 0.14.0 with libsndfile 1.2.2, and Transformers 5.23.0.
 
-The preparation environment was verified on Windows with Python 3.14.10. Run from the repository root.
+Use a Python **3.14** environment for the data pipeline and its tests. Run from the repository root on Windows.
 
 ```powershell
-python -m venv .venv
+py -3.14 -m venv .venv
 $venvPy = '.\.venv\Scripts\python.exe'
 & $venvPy -m pip install --only-binary=:all: -r requirements-windows-py314.lock.txt
 & $venvPy -m pip check
 ```
 
 On macOS or Linux with Python 3.14, use `python3.14 -m venv .venv`, `.venv/bin/python`, and [requireme

In [11]:
import re

for label, content in [
    ("OUR BRANCH", ours),
    ("LATEST MAIN", theirs)
]:
    print(f"\n{'=' * 60}")
    print(label)
    print("=" * 60)

    lines = content.splitlines()

    for i, line in enumerate(lines):
        if re.match(r"^#{1,3}\s", line):
            if any(term in line.lower() for term in [
                "whisper",
                "person 3",
                "checkpoint",
                "reproducib",
                "evaluation"
            ]):
                print(f"Line {i + 1}: {line}")


OUR BRANCH
Line 251: ### Baselines: published checkpoints, no training by our group
Line 353: ## Person 3:Tasnim Alibraheem 2309015259 — Whisper Small Fine-Tuning
Line 401: ### Remaining Evaluation
Line 451: ## Person 3 — Whisper Small Fine-Tuning Reproducibility
Line 486: ### Reproduce the Five-Sample Evaluation
Line 546: ### Verified Reproducibility Results
Line 578: ### Training Results and Checkpoint Limitations

LATEST MAIN
Line 285: ### Baselines: published checkpoints, no training by our group
Line 423: ## Whisper Small fine-tuning
Line 455: ### Recovered checkpoint evaluation


In [12]:
def extract_section(content, heading):
    lines = content.splitlines()
    start = next(
        i for i, line in enumerate(lines)
        if line.startswith(heading)
    )

    end = len(lines)

    for i in range(start + 1, len(lines)):
        if lines[i].startswith("## "):
            end = i
            break

    return "\n".join(lines[start:end])


our_section = extract_section(
    ours,
    "## Person 3 — Whisper Small Fine-Tuning Reproducibility"
)

main_section = extract_section(
    theirs,
    "## Whisper Small fine-tuning"
)

print("=" * 60)
print("OUR REPRODUCIBILITY SECTION")
print("=" * 60)
print(our_section)

print("\n" + "=" * 60)
print("LATEST MAIN WHISPER SECTION")
print("=" * 60)
print(main_section)

OUR REPRODUCIBILITY SECTION
## Person 3 — Whisper Small Fine-Tuning Reproducibility

**Contributor:** Tasnim Alibraheem  
**Student ID:** 2309015259  
**GitHub:** @tasneemzouhair-coder

### Overview

Person 3 fine-tuned multilingual Whisper Small for Icelandic automatic speech recognition (ASR) using the Samrómur 21.05 dataset.

The original training run is complete. The recovered checkpoint 1000 is publicly downloadable and can be evaluated without retraining or accessing the contributor's personal Google Drive.

### Training Configuration

| Parameter | Value |
|---|---|
| Base model | `openai/whisper-small` |
| Dataset | Samrómur 21.05 |
| Dataset selection | `samromur-21.05-20h-v1` |
| Training data | Approximately 15 hours |
| Validation data | Approximately 2 hours |
| Test data | Approximately 3 hours |
| Epochs | 3 |
| Learning rate | 1e-5 |
| Batch size | 2 |
| Gradient accumulation | 8 |
| Effective batch size | 16 |
| Mixed precision | FP16 |
| Training GPU | NVIDIA Tesla T4

In [14]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
readme_path = repo / "README.md"

# Get the latest main README
result = subprocess.run(
    ["git", "-C", str(repo), "show", ":3:README.md"],
    capture_output=True,
    text=True,
    check=True
)
main_readme = result.stdout

# Reproducibility instructions
addition = "\n".join([
    "### Reproduce Person 3's Five-Sample Evaluation",
    "",
    "The recovered Whisper Small checkpoint can be downloaded and",
    "evaluated without retraining or accessing a personal Google Drive.",
    "",
    "The original training did not record the exact base-model revision",
    "or dataset checkout commit.",
    "",
    "**1. Install dependencies**",
    "",
    "```bash",
    "python -m pip install -r training/requirements-person3.txt gdown",
    "```",
    "",
    "**2. Download and verify the checkpoint**",
    "",
    "```bash",
    "python training/download_whisper_checkpoint.py",
    "```",
    "",
    "The downloader verifies SHA-256 checksums for seven checkpoint files.",
    "The model is saved to `models/recovered_model_step1000/`.",
    "",
    "**3. Download development audio**",
    "",
    "```bash",
    "git lfs install",
    'git lfs pull --include="data/cleaned/samromur-21.05-20h-v1/dev/**" --exclude=""',
    "```",
    "",
    "**4. Run the five-sample evaluation**",
    "",
    "```bash",
    "python training/evaluate_whisper_step1000.py \\",
    "  --model models/recovered_model_step1000 \\",
    "  --audio-root data/cleaned/samromur-21.05-20h-v1 \\",
    "  --split dev \\",
    "  --max-samples 5 \\",
    "  --output-dir results/person3_whisper_repro_test",
    "```",
    "",
    "The verified five-sample CPU results were **40.54% WER** and",
    "**12.36% CER**. These are smoke-test results, not full-split metrics.",
    "",
    "No retraining is required."
])

# Insert immediately after the existing Whisper section
heading = "## Whisper Small fine-tuning"
assert heading in main_readme, "Whisper section not found."

start = main_readme.index(heading)
next_section = main_readme.find("\n## ", start + len(heading))

if next_section == -1:
    merged_readme = main_readme.rstrip() + "\n\n" + addition + "\n"
else:
    merged_readme = (
        main_readme[:next_section].rstrip()
        + "\n\n"
        + addition
        + "\n\n"
        + main_readme[next_section:].lstrip("\n")
    )

# Validate before writing
assert merged_readme.count(heading) == 1
assert merged_readme.count("### Reproduce Person 3's Five-Sample Evaluation") == 1
assert "download_whisper_checkpoint.py" in merged_readme

# Write and stage
readme_path.write_text(merged_readme, encoding="utf-8")

subprocess.run(
    ["git", "-C", str(repo), "add", "README.md"],
    check=True
)

print("README MERGE: SUCCESS")
print("Latest main documentation preserved: YES")
print("Person 3 reproducibility instructions added: YES")
print("Automatic downloader documented: YES")
print("Total README lines:", len(merged_readme.splitlines()))

README MERGE: SUCCESS
Latest main documentation preserved: YES
Person 3 reproducibility instructions added: YES
Automatic downloader documented: YES
Total README lines: 541


In [15]:
import subprocess
import json
import ast
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook_name = "NLP_Project3_Person3_Whisper_Clean.ipynb"
notebook_path = repo / notebook_name

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

# Restore our feature branch's notebook
git("checkout", "--ours", "--", notebook_name)

# Validate notebook structure
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

assert len(nb["cells"]) == 52, "Unexpected notebook cell count."

# Check the final evaluation cell
final_cell = nb["cells"][51]
source = "".join(final_cell["source"])

assert final_cell["cell_type"] == "code"
ast.parse(source)

assert "download_whisper_checkpoint.py" in source
assert "recovered_model_step1000" in source

# Ensure the personal Drive checkpoint path is absent
assert (
    "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/recovered_model_step1000"
    not in source
)

# Stage the resolved notebook
git("add", notebook_name)

# Check remaining conflicts
remaining = git("diff", "--name-only", "--diff-filter=U")

print("NOTEBOOK VALIDATION")
print("Notebook JSON: PASS")
print("All 52 cells preserved: PASS")
print("Evaluation cell syntax: PASS")
print("Automatic checkpoint downloader: PASS")
print("Personal checkpoint path removed: PASS")
print()
print("Remaining merge conflicts:", remaining or "None")
print("Merge commit created: NO")

NOTEBOOK VALIDATION
Notebook JSON: PASS
All 52 cells preserved: PASS
Evaluation cell syntax: PASS
Automatic checkpoint downloader: PASS
Personal checkpoint path removed: PASS

Remaining merge conflicts: None
Merge commit created: NO


In [18]:
import json
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

with notebook_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = "".join(nb["cells"][51]["source"])
lines = source.splitlines()

print("FINAL EVALUATION CELL — REPOSITORY SETUP")
print("=" * 60)

for i, line in enumerate(lines):
    if any(term in line for term in [
        "git",
        "clone",
        "repo =",
        "model_dir =",
        "download_whisper_checkpoint.py"
    ]):
        start = max(0, i - 2)
        end = min(len(lines), i + 4)

        print(f"\nLines {start + 1}–{end}:")
        for j in range(start, end):
            print(f"{j + 1:3}: {lines[j]}")

FINAL EVALUATION CELL — REPOSITORY SETUP

Lines 5–10:
  5: from transformers import WhisperProcessor, WhisperForConditionalGeneration
  6: 
  7: repo = Path("/content/icelandic-asr-project3")
  8: if not (repo / ".git").exists():
  9:     subprocess.run(["git", "clone", "--depth", "1", "https://github.com/RanimAlyaseen/icelandic-asr-project3.git", str(repo)], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
 10: 

Lines 6–11:
  6: 
  7: repo = Path("/content/icelandic-asr-project3")
  8: if not (repo / ".git").exists():
  9:     subprocess.run(["git", "clone", "--depth", "1", "https://github.com/RanimAlyaseen/icelandic-asr-project3.git", str(repo)], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
 10: 
 11: # Download and verify the recovered checkpoint automatically.

Lines 7–12:
  7: repo = Path("/content/icelandic-asr-project3")
  8: if not (repo / ".git").exists():
  9:     subprocess.run(["git", "clone", "--depth", "1", "https://github.com/RanimAlyaseen/icel

In [19]:
import json
import ast
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

with notebook_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

cell = nb["cells"][51]
source = "".join(cell["source"])

old = '["git", "clone", "--depth", "1", "https://github.com/RanimAlyaseen/icelandic-asr-project3.git", str(repo)]'

new = '["git", "clone", "--depth", "1", "--branch", "person3-whisper-reproducibility", "https://github.com/RanimAlyaseen/icelandic-asr-project3.git", str(repo)]'

# Verify the original command exists exactly once
assert source.count(old) == 1, "Expected clone command not found."

# Replace only the clone command
source = source.replace(old, new)

# Validate the updated code
ast.parse(source)

assert "download_whisper_checkpoint.py" in source
assert "--branch" in source
assert "person3-whisper-reproducibility" in source

# Update notebook
cell["source"] = source.splitlines(keepends=True)

with notebook_path.open("w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)
    f.write("\n")

# Stage the updated notebook
subprocess.run(
    ["git", "-C", str(repo), "add", notebook_path.name],
    check=True
)

# Check remaining conflicts
result = subprocess.run(
    ["git", "-C", str(repo), "diff", "--name-only", "--diff-filter=U"],
    capture_output=True,
    text=True,
    check=True
)

print("CLONE COMMAND FIX")
print("=" * 45)
print("Feature branch specified: PASS")
print("Notebook syntax: PASS")
print("Automatic downloader preserved: PASS")
print("Remaining merge conflicts:", result.stdout.strip() or "None")
print("Ready for merge commit: YES")

CLONE COMMAND FIX
Feature branch specified: PASS
Notebook syntax: PASS
Automatic downloader preserved: PASS
Remaining merge conflicts: None
Ready for merge commit: YES


In [20]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

def git(*args):
    result = subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout.strip()

# Confirm we are on the correct branch
branch = git("branch", "--show-current")
assert branch == "person3-whisper-reproducibility", branch

# Confirm the merge is still in progress
merge_head = git("rev-parse", "--verify", "MERGE_HEAD")
assert merge_head, "No merge in progress."

# Confirm no unresolved conflicts remain
conflicts = git("diff", "--name-only", "--diff-filter=U")
assert not conflicts, f"Unresolved conflicts: {conflicts}"

# Configure commit identity
git("config", "user.name", "tasneemzouhair-coder")
git(
    "config",
    "user.email",
    "tasneemzouhair-coder@users.noreply.github.com"
)

# Create the merge commit
commit = git(
    "commit",
    "-m",
    "Merge main and resolve Person 3 reproducibility conflicts"
)

print("MERGE COMMIT")
print(commit)

print("\nLATEST COMMITS")
print(git("log", "-3", "--oneline"))

print("\nBRANCH STATUS")
print(git("status", "--short", "--branch"))

print("\nMERGE PARENTS")
print(git("rev-list", "--parents", "-n", "1", "HEAD"))

MERGE COMMIT
[person3-whisper-reproducibility c12178d4] Merge main and resolve Person 3 reproducibility conflicts

LATEST COMMITS
c12178d4 Merge main and resolve Person 3 reproducibility conflicts
741e40b6 Use automatic checkpoint downloader in Whisper evaluation
86ab3c3e Merge pull request #7 from MuhammetGoksenin/main

BRANCH STATUS
## person3-whisper-reproducibility...origin/person3-whisper-reproducibility [ahead 19]

MERGE PARENTS
c12178d4eac6eba469597835048a90380f13bd49 741e40b605f354c211f47113c4d384d847316d13 86ab3c3e9373d6017e4ecd0eb8feabfef8bc09ed


In [21]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

checks = {
    "Git remote": ["git", "-C", str(repo), "remote", "-v"],
    "Credential helper": ["git", "-C", str(repo), "config", "--get", "credential.helper"],
    "GitHub CLI installed": ["which", "gh"],
}

for name, command in checks.items():
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print(f"\n{name}:")
    print(result.stdout.strip() or "Not configured")

print("\nCurrent branch:")
result = subprocess.run(
    ["git", "-C", str(repo), "branch", "--show-current"],
    capture_output=True,
    text=True
)
print(result.stdout.strip())


Git remote:
origin	https://github.com/RanimAlyaseen/icelandic-asr-project3.git (fetch)
origin	https://github.com/RanimAlyaseen/icelandic-asr-project3.git (push)

Credential helper:
Not configured

GitHub CLI installed:
/usr/bin/gh

Current branch:
person3-whisper-reproducibility


In [22]:
import subprocess

result = subprocess.run(
    ["gh", "auth", "status"],
    capture_output=True,
    text=True
)

print("GITHUB AUTHENTICATION STATUS")
print("=" * 45)
print(result.stdout or result.stderr)
print("\nAuthenticated:", result.returncode == 0)

GITHUB AUTHENTICATION STATUS
You are not logged into any GitHub hosts. To log in, run: gh auth login


Authenticated: False


In [32]:
!gh auth login --hostname github.com --git-protocol https --web

? Authenticate Git with your GitHub credentials? (Y/n) 7^C


In [33]:
import subprocess

result = subprocess.run(
    "printf 'Y\\n' | gh auth login --hostname github.com --git-protocol https --web",
    shell=True,
    capture_output=True,
    text=True,
    timeout=60
)

print(result.stdout)
print(result.stderr)

KeyboardInterrupt: 

In [30]:
import subprocess

process = subprocess.Popen(
    [
        "gh", "auth", "login",
        "--hostname", "github.com",
        "--git-protocol", "https",
        "--web"
    ],
    stdin=subprocess.PIPE,
    stdout=None,
    stderr=None,
    text=True
)

process.stdin.write("Y\n")
process.stdin.flush()

print("Authentication started. Follow the GitHub instructions above.")

Authentication started. Follow the GitHub instructions above.


In [34]:
import subprocess

result = subprocess.run(
    ["gh", "auth", "status"],
    capture_output=True,
    text=True,
    timeout=10
)

print("Authenticated:", result.returncode == 0)
print(result.stdout or result.stderr)

Authenticated: False
You are not logged into any GitHub hosts. To log in, run: gh auth login



In [35]:
import subprocess

result = subprocess.run(
    ["gh", "auth", "login", "--hostname", "github.com",
     "--git-protocol", "https", "--web"],
    input="Y\n",
    capture_output=True,
    text=True,
    timeout=15
)

print(result.stdout)
print(result.stderr)

TimeoutExpired: Command '['gh', 'auth', 'login', '--hostname', 'github.com', '--git-protocol', 'https', '--web']' timed out after 15 seconds

In [36]:
import subprocess
import time
from pathlib import Path

log = Path("/content/github_login.log")

# Start authentication without blocking Colab
with log.open("w") as f:
    process = subprocess.Popen(
        [
            "gh", "auth", "login",
            "--hostname", "github.com",
            "--git-protocol", "https",
            "--web"
        ],
        stdin=subprocess.PIPE,
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True
    )

    # Automatically answer the Git authentication question
    process.stdin.write("Y\n")
    process.stdin.flush()

# Allow GitHub CLI to generate the device code
time.sleep(5)

print("GITHUB AUTHENTICATION")
print("=" * 50)
print(log.read_text(errors="replace"))
print("\nProcess running:", process.poll() is None)

GITHUB AUTHENTICATION

! Failed to copy one-time code to clipboard
  No clipboard utilities available. Please install xsel, xclip, wl-clipboard or Termux:API add-on for termux-clipboard-get/set.
! First copy your one-time code: 2220-25E1
Open this URL to continue in your web browser: https://github.com/login/device


Process running: True


In [37]:
import subprocess

result = subprocess.run(
    ["gh", "auth", "status"],
    capture_output=True,
    text=True
)

print("Authenticated:", result.returncode == 0)
print(result.stdout or result.stderr)

Authenticated: True
github.com
  ✓ Logged in to github.com account tasneemzouhair-coder (/root/.config/gh/hosts.yml)
  - Active account: true
  - Git operations protocol: https
  - Token: gho_************************************
  - Token scopes: 'gist', 'read:org', 'repo'



In [38]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")
branch = "person3-whisper-reproducibility"

# Configure Git to use GitHub CLI authentication
subprocess.run(
    ["gh", "auth", "setup-git"],
    check=True
)

# Verify the current branch
result = subprocess.run(
    ["git", "-C", str(repo), "branch", "--show-current"],
    capture_output=True,
    text=True,
    check=True
)

assert result.stdout.strip() == branch, "Incorrect branch!"

# Verify there are no uncommitted changes
result = subprocess.run(
    ["git", "-C", str(repo), "status", "--porcelain"],
    capture_output=True,
    text=True,
    check=True
)

assert not result.stdout.strip(), "Uncommitted changes detected!"

# Push only our feature branch
print("Pushing Person 3 branch to GitHub...")

result = subprocess.run(
    ["git", "-C", str(repo), "push", "origin", branch],
    capture_output=True,
    text=True
)

print("\nPUSH RESULT")
print("=" * 50)
print(result.stdout)
print(result.stderr)

print("\nPush successful:", result.returncode == 0)

Pushing Person 3 branch to GitHub...

PUSH RESULT

To https://github.com/RanimAlyaseen/icelandic-asr-project3.git
   61baca25..c12178d4  person3-whisper-reproducibility -> person3-whisper-reproducibility


Push successful: True


In [39]:
import subprocess
import json

result = subprocess.run(
    [
        "gh", "pr", "view", "4",
        "--repo", "RanimAlyaseen/icelandic-asr-project3",
        "--json",
        "number,title,state,mergeable,mergeStateStatus,headRefName,baseRefName,url"
    ],
    capture_output=True,
    text=True,
    check=True
)

pr = json.loads(result.stdout)

print("PULL REQUEST #4")
print("=" * 50)
print("Title:", pr["title"])
print("State:", pr["state"])
print("Mergeable:", pr["mergeable"])
print("Merge status:", pr["mergeStateStatus"])
print("Source branch:", pr["headRefName"])
print("Target branch:", pr["baseRefName"])
print("URL:", pr["url"])

PULL REQUEST #4
Title: Person 3: Reproducible Whisper checkpoint download and evaluation
State: OPEN
Mergeable: UNKNOWN
Merge status: UNKNOWN
Source branch: person3-whisper-reproducibility
Target branch: main
URL: https://github.com/RanimAlyaseen/icelandic-asr-project3/pull/4


In [40]:
import subprocess
import json
import time

repo = "RanimAlyaseen/icelandic-asr-project3"

for attempt in range(1, 6):
    result = subprocess.run(
        [
            "gh", "pr", "view", "4",
            "--repo", repo,
            "--json", "state,mergeable,mergeStateStatus"
        ],
        capture_output=True,
        text=True,
        check=True
    )

    pr = json.loads(result.stdout)

    print(f"Check {attempt}")
    print("State:", pr["state"])
    print("Mergeable:", pr["mergeable"])
    print("Merge status:", pr["mergeStateStatus"])
    print("-" * 40)

    if pr["mergeable"] != "UNKNOWN":
        break

    if attempt < 5:
        time.sleep(5)

Check 1
State: OPEN
Mergeable: MERGEABLE
Merge status: CLEAN
----------------------------------------


In [2]:
from pathlib import Path

print("CHECKING COLAB FILES")
print("=" * 55)

paths = [
    Path("/content/icelandic-asr-merge-check"),
    Path("/content/icelandic-asr-project3"),
]

for path in paths:
    print(f"{path}: {'EXISTS' if path.exists() else 'NOT FOUND'}")

print("\nNOTEBOOK SEARCH")
print("=" * 55)

for notebook in Path("/content").glob("**/*Whisper*.ipynb"):
    print(notebook)

print("\nCURRENT DIRECTORY")
print(Path.cwd())

CHECKING COLAB FILES
/content/icelandic-asr-merge-check: NOT FOUND
/content/icelandic-asr-project3: NOT FOUND

NOTEBOOK SEARCH

CURRENT DIRECTORY
/content


In [3]:
import subprocess
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

result = subprocess.run(
    [
        "git", "clone",
        "--branch", "person3-whisper-reproducibility",
        "--single-branch",
        "https://github.com/RanimAlyaseen/icelandic-asr-project3.git",
        str(repo)
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Repository clone failed.")

print("\nREPOSITORY RESTORED")
print("=" * 55)

# Confirm branch and latest commit
subprocess.run(
    ["git", "-C", str(repo), "log", "-1", "--oneline"],
    check=True
)

print("\nNOTEBOOK FILES")
for path in sorted(repo.rglob("*.ipynb")):
    print(path.relative_to(repo))


Cloning into '/content/icelandic-asr-merge-check'...
Updating files:  31% (9783/31103)
Updating files:  32% (9953/31103)
Updating files:  33% (10264/31103)
Updating files:  34% (10576/31103)
Updating files:  35% (10887/31103)
Updating files:  36% (11198/31103)
Updating files:  37% (11509/31103)
Updating files:  38% (11820/31103)
Updating files:  39% (12131/31103)
Updating files:  40% (12442/31103)
Updating files:  41% (12753/31103)
Updating files:  42% (13064/31103)
Updating files:  43% (13375/31103)
Updating files:  44% (13686/31103)
Updating files:  45% (13997/31103)
Updating files:  46% (14308/31103)
Updating files:  47% (14619/31103)
Updating files:  48% (14930/31103)
Updating files:  49% (15241/31103)
Updating files:  50% (15552/31103)
Updating files:  51% (15863/31103)
Updating files:  51% (15931/31103)
Updating files:  52% (16174/31103)
Updating files:  53% (16485/31103)
Updating files:  54% (16796/31103)
Updating files:  55% (17107/31103)
Updating files:  56% (17418/31103)
Upd

In [4]:
from pathlib import Path
import json

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

assert notebook_path.exists(), "Notebook not found!"

notebook = json.loads(notebook_path.read_text())

keywords = [
    "from_pretrained(",
    "git clone",
    "git checkout",
    "gdown",
    "pip install",
    "Seq2SeqTrainingArguments",
    "Seq2SeqTrainer("
]

print("CURRENT TRAINING CONFIGURATION")
print("=" * 65)

for index, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):
        print(f"\nCELL {index} [{cell['cell_type']}]")
        print("-" * 65)

        for line in source.splitlines():
            if any(keyword in line for keyword in keywords):
                print(line[:200])

print("\nEXISTING TRAINING FILES")
print("=" * 65)

training_dir = repo / "training"

if training_dir.exists():
    for path in sorted(training_dir.iterdir()):
        if path.is_file():
            print(path.name)

CURRENT TRAINING CONFIGURATION

CELL 1 [code]
-----------------------------------------------------------------
%pip install -q -r https://raw.githubusercontent.com/RanimAlyaseen/icelandic-asr-project3/person3-whisper-reproducibility/training/requirements-person3.txt

CELL 6 [code]
-----------------------------------------------------------------
%pip install -q "evaluate==0.4.6" "jiwer==4.0.0"

CELL 7 [code]
-----------------------------------------------------------------
%pip install -q "evaluate==0.4.6" "jiwer==4.0.0"

CELL 9 [code]
-----------------------------------------------------------------
processor = WhisperProcessor.from_pretrained(
model = WhisperForConditionalGeneration.from_pretrained(

CELL 17 [code]
-----------------------------------------------------------------
processor = WhisperProcessor.from_pretrained(

CELL 19 [code]
-----------------------------------------------------------------
from transformers import Seq2SeqTrainingArguments
    Seq2SeqTrainingArguments

In [5]:
from pathlib import Path
import importlib.metadata

repo = Path("/content/icelandic-asr-merge-check")
requirements = repo / "training/requirements-person3.txt"

print("PERSON 3 REQUIREMENTS")
print("=" * 60)
print(requirements.read_text())

print("\nINSTALLED PACKAGE VERSIONS")
print("=" * 60)

for package in [
    "gdown",
    "transformers",
    "datasets",
    "accelerate",
    "torch",
    "evaluate",
    "jiwer"
]:
    try:
        version = importlib.metadata.version(package)
        print(f"{package}: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

PERSON 3 REQUIREMENTS
# Person 3 — Whisper Small Fine-Tuning
# Package versions recorded from the Google Colab training environment.

torch==2.14.1
transformers==5.22.1
datasets==4.8.5
accelerate==1.15.0
evaluate==0.4.6
jiwer==4.0.0
librosa==0.11.0
soundfile==0.14.0


INSTALLED PACKAGE VERSIONS
gdown: 5.2.2
transformers: 5.22.1
datasets: 4.8.5
accelerate: 1.15.0
torch: 2.14.1+cpu
evaluate: NOT INSTALLED
jiwer: NOT INSTALLED


In [6]:
from pathlib import Path
import json

repo = Path("/content/icelandic-asr-merge-check")

# 1. Update requirements
requirements_path = repo / "training/requirements-person3.txt"
requirements = requirements_path.read_text()

if "gdown==" not in requirements:
    requirements = requirements.rstrip() + "\ngdown==5.2.2\n"
    requirements_path.write_text(requirements)

# 2. Update notebook
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"
notebook = json.loads(notebook_path.read_text())

cell = notebook["cells"][51]
source = "".join(cell["source"])

old = '"install", "-q", "gdown"'
new = '"install", "-q", "gdown==5.2.2"'

assert old in source or new in source, "Expected gdown installation not found."

source = source.replace(old, new)
cell["source"] = source.splitlines(keepends=True)

notebook_path.write_text(
    json.dumps(notebook, ensure_ascii=False, indent=1) + "\n"
)

# 3. Validate
updated_requirements = requirements_path.read_text()
updated_notebook = json.loads(notebook_path.read_text())
updated_cell = "".join(updated_notebook["cells"][51]["source"])

assert "gdown==5.2.2" in updated_requirements
assert '"gdown==5.2.2"' in updated_cell

print("STEP 25 — VALIDATION")
print("=" * 55)
print("Requirements pinned: PASS")
print("Notebook Cell 51 pinned: PASS")
print("Pinned version: gdown==5.2.2")
print("Training executed: NO")

STEP 25 — VALIDATION
Requirements pinned: PASS
Notebook Cell 51 pinned: PASS
Pinned version: gdown==5.2.2
Training executed: NO


In [7]:
from pathlib import Path
import json

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

notebook = json.loads(notebook_path.read_text())

print("STEP 26 — WHISPER MODEL CONFIGURATION")
print("=" * 65)

for index in [9, 17]:
    cell = notebook["cells"][index]
    source = "".join(cell.get("source", []))

    print(f"\nCELL {index}")
    print("-" * 65)
    print(source)

print("\nMODEL REFERENCES IN REPOSITORY")
print("=" * 65)

for path in [
    repo / "training/requirements-person3.txt",
    repo / "training/evaluate_whisper_step1000.py",
]:
    if path.exists():
        print(f"\nFILE: {path.relative_to(repo)}")
        for line in path.read_text().splitlines():
            if any(
                keyword in line.lower()
                for keyword in ["whisper-small", "revision", "from_pretrained"]
            ):
                print(line[:200])

STEP 26 — WHISPER MODEL CONFIGURATION

CELL 9
-----------------------------------------------------------------
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-small"

print("=" * 60)
print("WHISPER SMALL — MODEL INITIALIZATION")
print("=" * 60)

# Load processor for Icelandic transcription
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language="icelandic",
    task="transcribe"
)

print("\nProcessor loaded successfully.")

# Load pretrained model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID
)

print("Model loaded successfully.")

# Configure transcription language
model.generation_config.language = "icelandic"
model.generation_config.task = "transcribe"

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("\nMODEL INFORMATION")
print("-" * 60)

print(f"Model 

In [8]:
import subprocess
import sys

# Install the lightweight Hugging Face Hub client if needed
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
    check=True
)

from huggingface_hub import HfApi

MODEL_ID = "openai/whisper-small"

api = HfApi()
model_info = api.model_info(MODEL_ID, revision="main")

MODEL_REVISION = model_info.sha

assert len(MODEL_REVISION) == 40
assert all(c in "0123456789abcdef" for c in MODEL_REVISION)

print("STEP 27 — MODEL REVISION")
print("=" * 65)
print("Model:", MODEL_ID)
print("Resolved commit:", MODEL_REVISION)
print("Commit length:", len(MODEL_REVISION))
print("Validation: PASS")
print("Model downloaded: NO")
print("Training executed: NO")

STEP 27 — MODEL REVISION
Model: openai/whisper-small
Resolved commit: 973afd24965f72e36ca33b3055d56a652f456b4d
Commit length: 40
Validation: PASS
Model downloaded: NO
Training executed: NO


In [9]:
from pathlib import Path
import json
import re

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

MODEL_REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"

notebook = json.loads(notebook_path.read_text())

# Update the model and processor initialization cells
for index in [9, 17]:
    cell = notebook["cells"][index]
    source = "".join(cell["source"])

    # Define the fixed revision alongside MODEL_ID
    source = source.replace(
        'MODEL_ID = "openai/whisper-small"',
        f'MODEL_ID = "openai/whisper-small"\nMODEL_REVISION = "{MODEL_REVISION}"'
    )

    # Add revision to pretrained loading calls
    source = re.sub(
        r'(WhisperProcessor\.from_pretrained\(\s*\n\s*MODEL_ID,)',
        r'\1\n    revision=MODEL_REVISION,',
        source
    )

    source = re.sub(
        r'(WhisperForConditionalGeneration\.from_pretrained\(\s*\n\s*MODEL_ID)',
        r'\1,\n    revision=MODEL_REVISION',
        source
    )

    cell["source"] = source.splitlines(keepends=True)

# Save notebook
notebook_path.write_text(
    json.dumps(notebook, ensure_ascii=False, indent=1) + "\n"
)

# Validate the changes
updated = json.loads(notebook_path.read_text())

cell9 = "".join(updated["cells"][9]["source"])
cell17 = "".join(updated["cells"][17]["source"])

assert cell9.count("revision=MODEL_REVISION") == 2
assert cell17.count("revision=MODEL_REVISION") == 1

# Check Python syntax without running the model
compile(cell9, "<cell9>", "exec")
compile(cell17, "<cell17>", "exec")

print("STEP 28 — VALIDATION")
print("=" * 60)
print("Cell 9 processor revision: PASS")
print("Cell 9 model revision: PASS")
print("Cell 17 processor revision: PASS")
print("Python syntax: PASS")
print("Revision:", MODEL_REVISION)
print("Training executed: NO")

STEP 28 — VALIDATION
Cell 9 processor revision: PASS
Cell 9 model revision: PASS
Cell 17 processor revision: PASS
Python syntax: PASS
Revision: 973afd24965f72e36ca33b3055d56a652f456b4d
Training executed: NO


In [10]:
from pathlib import Path
import json

repo = Path("/content/icelandic-asr-merge-check")
notebook = json.loads(
    (repo / "NLP_Project3_Person3_Whisper_Clean.ipynb").read_text()
)

keywords = [
    "dataset",
    "data_dir",
    "train.jsonl",
    "dev.jsonl",
    "test.jsonl",
    "git clone",
    "git checkout",
    "git pull",
    "load_dataset",
    "drive.mount",
    "prepare.py",
    "preprocess.py",
]

print("STEP 29 — DATASET SOURCE INSPECTION")
print("=" * 65)

for index, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    matches = [
        line for line in source.splitlines()
        if any(keyword in line.lower() for keyword in keywords)
    ]

    if matches:
        print(f"\nCELL {index}")
        print("-" * 65)

        for line in matches[:20]:
            print(line[:180])

print("\nREPOSITORY DATA DIRECTORIES")
print("=" * 65)

for name in ["data", "datasets", "dataset", "scripts"]:
    path = repo / name

    if path.exists():
        print(f"\n{name}/")
        for item in sorted(path.iterdir())[:15]:
            print(" ", item.name)

print("\nTRAINING DATA FILES")
print("=" * 65)

for filename in ["train.jsonl", "dev.jsonl", "test.jsonl"]:
    matches = list(repo.rglob(filename))
    print(f"{filename}: {len(matches)} found")

    for match in matches[:3]:
        print(" ", match.relative_to(repo))

STEP 29 — DATASET SOURCE INSPECTION

CELL 0
-----------------------------------------------------------------
**Dataset:** Samrómur 21.05 (OpenSLR 112)  

CELL 2
-----------------------------------------------------------------
### Model and Dataset Version Tracking
**Dataset:** Samrómur 21.05 (OpenSLR 112)
**Dataset selection:** `samromur-21.05-20h-v1`
**Dataset preparation reference:** Git commit `55f5a68`
**Original dataset checkout revision:** Not recorded during the original training run.
**Reproducibility note:** The dataset preparation reference identifies the team's frozen dataset selection. It should not be interpreted as a verified record of the exact checkout 

CELL 3
-----------------------------------------------------------------
# Record the repository revision if running inside a Git checkout.
DATASET_REPOSITORY_REVISION = (
print("Dataset repository revision:", DATASET_REPOSITORY_REVISION)
print("Dataset selection: samromur-21.05-20h-v1")

CELL 5
----------------------

In [11]:
from pathlib import Path
import subprocess
import json

repo = Path("/content/icelandic-asr-merge-check")

print("STEP 30 — DATASET COMMIT VERIFICATION")
print("=" * 65)

# Verify the preparation commit exists
result = subprocess.run(
    ["git", "-C", str(repo), "rev-parse", "55f5a68^{commit}"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Preparation commit: FOUND")
    print("Full commit:", result.stdout.strip())
else:
    print("Preparation commit: NOT FOUND")
    print(result.stderr.strip())

# Inspect existing dataset setup
notebook = json.loads(
    (repo / "NLP_Project3_Person3_Whisper_Clean.ipynb").read_text()
)

for index in [2, 3, 23, 25]:
    cell = notebook["cells"][index]
    source = "".join(cell.get("source", []))

    print(f"\nCELL {index}")
    print("-" * 65)
    print(source[:5000])

# Check whether the manifests are tracked by Git
print("\nDATASET MANIFEST TRACKING")
print("-" * 65)

for filename in [
    "output/train.jsonl",
    "output/dev.jsonl",
    "output/test.jsonl"
]:
    result = subprocess.run(
        ["git", "-C", str(repo), "ls-files", "--error-unmatch", filename],
        capture_output=True,
        text=True
    )

    print(
        filename,
        "TRACKED" if result.returncode == 0 else "NOT TRACKED"
    )

print("\nTraining executed: NO")

STEP 30 — DATASET COMMIT VERIFICATION
Preparation commit: FOUND
Full commit: 55f5a6856e92cdf89608c492cc470aea0d019e1b

CELL 2
-----------------------------------------------------------------
### Model and Dataset Version Tracking

**Base model:** `openai/whisper-small` (multilingual)

**Original base-model revision:** Not recorded during the original training run.

**Dataset:** Samrómur 21.05 (OpenSLR 112)

**Dataset selection:** `samromur-21.05-20h-v1`

**Dataset preparation reference:** Git commit `55f5a68`

**Original dataset checkout revision:** Not recorded during the original training run.

**Reproducibility note:** The dataset preparation reference identifies the team's frozen dataset selection. It should not be interpreted as a verified record of the exact checkout used during the original training run.

**Recovered model:** Whisper Small fine-tuning checkpoint 1000.

**Checkpoint verification:** SHA-256 checksums are documented and verified by `training/download_whisper_check

In [12]:
from pathlib import Path
import subprocess
import hashlib

repo = Path("/content/icelandic-asr-merge-check")

DATASET_COMMIT = "55f5a6856e92cdf89608c492cc470aea0d019e1b"

manifests = [
    "output/train.jsonl",
    "output/dev.jsonl",
    "output/test.jsonl"
]

print("STEP 31 — DATASET SNAPSHOT VERIFICATION")
print("=" * 65)

all_match = True

for filename in manifests:

    result = subprocess.run(
        [
            "git", "-C", str(repo),
            "show", f"{DATASET_COMMIT}:{filename}"
        ],
        capture_output=True
    )

    print(f"\n{filename}")
    print("-" * 50)

    if result.returncode != 0:
        print("Present at preparation commit: NO")
        all_match = False
        continue

    historical_hash = hashlib.sha256(result.stdout).hexdigest()

    current_path = repo / filename
    current_hash = hashlib.sha256(
        current_path.read_bytes()
    ).hexdigest()

    matches = historical_hash == current_hash

    print("Present at preparation commit: YES")
    print("Matches current manifest:", matches)
    print("Preparation SHA-256:", historical_hash)
    print("Current SHA-256:    ", current_hash)

    if not matches:
        all_match = False

print("\nFINAL RESULT")
print("=" * 65)
print("All manifests match preparation commit:", all_match)
print("Training executed: NO")

STEP 31 — DATASET SNAPSHOT VERIFICATION

output/train.jsonl
--------------------------------------------------
Present at preparation commit: YES
Matches current manifest: True
Preparation SHA-256: d36a1adccdb10682b30a6fd8b2f7e20f8c86babb4b1d20a007af79179c4436ed
Current SHA-256:     d36a1adccdb10682b30a6fd8b2f7e20f8c86babb4b1d20a007af79179c4436ed

output/dev.jsonl
--------------------------------------------------
Present at preparation commit: YES
Matches current manifest: True
Preparation SHA-256: f0e7675cd31497788250728bba966d1d884ad5f929b3488724b2fef2110f9bf4
Current SHA-256:     f0e7675cd31497788250728bba966d1d884ad5f929b3488724b2fef2110f9bf4

output/test.jsonl
--------------------------------------------------
Present at preparation commit: YES
Matches current manifest: True
Preparation SHA-256: f5e346f545a5d7f4298f1838bc63b978a4d7ab9d063ddc97897fa065223d0481
Current SHA-256:     f5e346f545a5d7f4298f1838bc63b978a4d7ab9d063ddc97897fa065223d0481

FINAL RESULT
All manifests match pr

In [13]:
from pathlib import Path
import json
import textwrap

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

MODEL_REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"
DATASET_COMMIT = "55f5a6856e92cdf89608c492cc470aea0d019e1b"

notebook = json.loads(notebook_path.read_text())

# Cell 3: Record fixed revisions
cell3 = textwrap.dedent(f"""\
# Fixed revisions for future reproducible runs.
# These are not verified records of the original training checkout.

BASE_MODEL_ID = "openai/whisper-small"
BASE_MODEL_REVISION = "{MODEL_REVISION}"

DATASET_SELECTION = "samromur-21.05-20h-v1"
DATASET_REPOSITORY_REVISION = "{DATASET_COMMIT}"

print("Base model:", BASE_MODEL_ID)
print("Pinned model revision:", BASE_MODEL_REVISION)
print("Dataset selection:", DATASET_SELECTION)
print("Pinned dataset commit:", DATASET_REPOSITORY_REVISION)
""")

notebook["cells"][3]["source"] = cell3.splitlines(keepends=True)

# Cell 23: Clone and check out the fixed dataset commit
cell23 = textwrap.dedent(f"""\
from pathlib import Path
import subprocess
import os

print("=" * 60)
print("STEP 7A — DATASET REPOSITORY SETUP")
print("=" * 60)

repo_url = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
repo_path = Path("/content/icelandic-asr-project3")

DATASET_COMMIT = "{DATASET_COMMIT}"

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--no-checkout", repo_url, str(repo_path)],
        check=True,
        env={{**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}}
    )
else:
    print("Repository already exists.")

# Ensure the required commit is available locally.
subprocess.run(
    ["git", "-C", str(repo_path), "fetch", "origin", DATASET_COMMIT],
    check=True
)

# Do not overwrite uncommitted changes in an existing checkout.
status = subprocess.run(
    ["git", "-C", str(repo_path), "status", "--porcelain"],
    capture_output=True,
    text=True,
    check=True
)

if status.stdout.strip():
    raise RuntimeError(
        "Dataset repository contains uncommitted changes. "
        "Use a clean checkout before switching revisions."
    )

subprocess.run(
    ["git", "-C", str(repo_path), "checkout", "--detach", DATASET_COMMIT],
    check=True
)

actual_commit = subprocess.run(
    ["git", "-C", str(repo_path), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

assert actual_commit == DATASET_COMMIT, "Dataset revision mismatch!"

print("\\nDataset checkout:", actual_commit)
print("Dataset revision verified: PASS")
print("\\nSTEP 7A COMPLETE")
""")

notebook["cells"][23]["source"] = cell23.splitlines(keepends=True)

# Save
notebook_path.write_text(
    json.dumps(notebook, ensure_ascii=False, indent=1) + "\n"
)

# Validate without executing the checkout
updated = json.loads(notebook_path.read_text())

for index in [3, 23]:
    source = "".join(updated["cells"][index]["source"])
    compile(source, f"<cell{index}>", "exec")

assert MODEL_REVISION in "".join(updated["cells"][3]["source"])
assert DATASET_COMMIT in "".join(updated["cells"][3]["source"])
assert DATASET_COMMIT in "".join(updated["cells"][23]["source"])
assert '"checkout", "--detach", DATASET_COMMIT' in "".join(
    updated["cells"][23]["source"]
)

print("\nSTEP 32 — VALIDATION")
print("=" * 60)
print("Fixed model revision tracking: PASS")
print("Fixed dataset commit tracking: PASS")
print("Dataset checkout command: PASS")
print("Python syntax: PASS")
print("Training executed: NO")
print("Dataset checkout executed: NO")


STEP 32 — VALIDATION
Fixed model revision tracking: PASS
Fixed dataset commit tracking: PASS
Dataset checkout command: PASS
Python syntax: PASS
Training executed: NO
Dataset checkout executed: NO


In [14]:
from pathlib import Path
import json

repo = Path("/content/icelandic-asr-merge-check")
notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

notebook = json.loads(notebook_path.read_text())

# Cells containing dataset loading, preprocessing,
# training configuration, and training execution.
cells_to_inspect = [
    11, 15, 17, 19, 33, 35, 37, 39, 43, 45, 46
]

print("STEP 33 — TRAINING WORKFLOW INSPECTION")
print("=" * 70)

for index in cells_to_inspect:
    cell = notebook["cells"][index]
    source = "".join(cell.get("source", []))

    print(f"\n{'=' * 70}")
    print(f"CELL {index} [{cell['cell_type']}]")
    print("=" * 70)

    print(source[:12000])

print("\nINSPECTION COMPLETE")
print("Files modified: NO")
print("Training executed: NO")

STEP 33 — TRAINING WORKFLOW INSPECTION

CELL 11 [code]
import json
import random
import numpy as np
import torch

from pathlib import Path
from datetime import datetime, timezone

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Project configuration
TRAINING_CONFIG = {
    "project": "Icelandic ASR - NLP Project 3",
    "model_id": "openai/whisper-small",
    "language": "icelandic",
    "task": "transcribe",

    # Dataset targets
    "train_hours_target": 15,
    "validation_hours_target": 2,
    "test_hours_target": 3,

    # Initial training hyperparameters
    "learning_rate": 1e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "linear",

    # Memory optimization
    "fp16": True,
    "gradient_chec

In [15]:
from pathlib import Path
import json

repo = Path("/content/icelandic-asr-merge-check")

notebook = json.loads(
    (repo / "NLP_Project3_Person3_Whisper_Clean.ipynb").read_text()
)

print("STEP 34 — FINAL TRAINING COMPONENTS")
print("=" * 70)

# Find the metrics function and training execution
keywords = [
    "def compute_metrics",
    "trainer.train(",
    "trainer.save_model(",
    "processor.save_pretrained(",
    "resume_from_checkpoint",
]

for index, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):
        print(f"\n{'=' * 70}")
        print(f"CELL {index}")
        print("=" * 70)
        print(source[:15000])

print("\nTRAINING FILES")
print("=" * 70)

for path in sorted((repo / "training").glob("*whisper*")):
    print(path.relative_to(repo))

print("\nInspection complete.")
print("Files modified: NO")
print("Training executed: NO")

STEP 34 — FINAL TRAINING COMPONENTS

CELL 41
import numpy as np
import evaluate

print("=" * 60)
print("STEP 9B — EVALUATION METRICS")
print("=" * 60)

# Load evaluation metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):

    # Predicted token IDs
    pred_ids = pred.predictions

    # Handle models returning additional outputs
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # Reference token IDs
    label_ids = pred.label_ids.copy()

    # Replace ignored labels with padding token
    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    # Decode predictions
    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    # Decode references
    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    # Normalize whitespace
    pred_str = [
        " ".join(text.strip().split())
  

In [16]:
from pathlib import Path
import json
import textwrap

repo = Path("/content/icelandic-asr-merge-check")
notebook = json.loads(
    (repo / "NLP_Project3_Person3_Whisper_Clean.ipynb").read_text()
)

script_path = repo / "training" / "train_whisper.py"

MODEL_REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"
DATASET_COMMIT = "55f5a6856e92cdf89608c492cc470aea0d019e1b"

script = r'''#!/usr/bin/env python3
"""Reproducible Icelandic Whisper Small fine-tuning.

Based on NLP_Project3_Person3_Whisper_Clean.ipynb.

This script does not train when imported.
Use --check-only to validate paths and configuration.
"""

import argparse
import inspect
import json
import math
import random
import subprocess
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "openai/whisper-small"
MODEL_REVISION = "__MODEL_REVISION__"
DATASET_COMMIT = "__DATASET_COMMIT__"

DATASET_SELECTION = "samromur-21.05-20h-v1"

EXPECTED_TRAIN = 10488
EXPECTED_VALIDATION = 1314
EXPECTED_TEST = 1884

SEED = 42


def parse_args():
    parser = argparse.ArgumentParser(
        description="Fine-tune Whisper Small on Icelandic Samromur data."
    )

    parser.add_argument(
        "--repo-dir",
        type=Path,
        default=Path("/content/icelandic-asr-project3"),
        help="Dataset repository containing output/ and data/.",
    )

    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("./whisper_training_output"),
        help="Directory for checkpoints and results.",
    )

    parser.add_argument(
        "--check-only",
        action="store_true",
        help="Validate dataset paths without loading models or training.",
    )

    return parser.parse_args()


def get_repository_revision(repo_dir):
    result = subprocess.run(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()


def validate_dataset(repo_dir):
    repo_dir = repo_dir.resolve()

    if not repo_dir.is_dir():
        raise FileNotFoundError(
            f"Dataset repository not found: {repo_dir}"
        )

    actual_commit = get_repository_revision(repo_dir)

    if actual_commit != DATASET_COMMIT:
        raise RuntimeError(
            "Dataset repository revision mismatch.\n"
            f"Expected: {DATASET_COMMIT}\n"
            f"Actual:   {actual_commit}\n"
            "Check out the pinned dataset commit before training."
        )

    output_dir = repo_dir / "output"

    manifests = {
        "train": output_dir / "train.jsonl",
        "validation": output_dir / "dev.jsonl",
        "test": output_dir / "test.jsonl",
    }

    for split, path in manifests.items():
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {split} manifest: {path}"
            )

    audio_root = (
        repo_dir / "data" / "cleaned" / DATASET_SELECTION
    )

    if not audio_root.is_dir():
        raise FileNotFoundError(
            f"Prepared audio directory not found: {audio_root}"
        )

    print("Dataset repository:", repo_dir)
    print("Dataset commit:", actual_commit)
    print("Dataset selection:", DATASET_SELECTION)

    return manifests, audio_root


def main():
    args = parse_args()

    manifests, audio_root = validate_dataset(args.repo_dir)

    if args.check_only:
        print("Configuration and dataset paths: PASS")
        print("Training executed: NO")
        return

    import numpy as np
    import torch
    import soundfile as sf
    import evaluate

    from datasets import load_dataset
    from transformers import (
        WhisperProcessor,
        WhisperForConditionalGeneration,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer,
    )
    from transformers.trainer_utils import get_last_checkpoint

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU is required for this FP16 training configuration."
        )

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    output_dir = args.output_dir.resolve()
    checkpoint_dir = output_dir / "checkpoints"
    results_dir = output_dir / "results"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    print("Loading pinned Whisper Small model and processor...")

    processor = WhisperProcessor.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        language="icelandic",
        task="transcribe",
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
    )

    model.generation_config.language = "icelandic"
    model.generation_config.task = "transcribe"
    model.config.use_cache = False

    processor.tokenizer.clean_up_tokenization_spaces = False

    print("Loading prepared dataset manifests...")

    dataset = load_dataset(
        "json",
        data_files={
            split: str(path)
            for split, path in manifests.items()
        },
    )

    assert len(dataset["train"]) == EXPECTED_TRAIN
    assert len(dataset["validation"]) == EXPECTED_VALIDATION
    assert len(dataset["test"]) == EXPECTED_TEST

    def prepare_whisper_sample(example):
        audio_path = audio_root / example["audio_path"]

        audio, sample_rate = sf.read(
            str(audio_path),
            dtype="float32",
        )

        if sample_rate != 16000:
            raise ValueError(
                f"Unexpected sample rate: {sample_rate}"
            )

        if audio.ndim != 1:
            raise ValueError(
                f"Expected mono audio: {audio_path}"
            )

        features = processor.feature_extractor(
            audio,
            sampling_rate=16000,
        ).input_features[0]

        labels = processor.tokenizer(
            example["text"]
        ).input_ids

        return {
            "input_features": features,
            "labels": labels,
        }

    print("Preparing training features...")

    train_dataset = dataset["train"].map(
        prepare_whisper_sample,
        remove_columns=dataset["train"].column_names,
        num_proc=1,
        desc="Preparing training audio",
    )

    print("Preparing validation features...")

    eval_dataset = dataset["validation"].map(
        prepare_whisper_sample,
        remove_columns=dataset["validation"].column_names,
        num_proc=1,
        desc="Preparing validation audio",
    )

    @dataclass
    class DataCollatorSpeechSeq2SeqWithPadding:
        processor: Any

        def __call__(self, features):
            input_features = [
                {"input_features": item["input_features"]}
                for item in features
            ]

            batch = self.processor.feature_extractor.pad(
                input_features,
                return_tensors="pt",
            )

            label_features = [
                {"input_ids": item["labels"]}
                for item in features
            ]

            labels_batch = self.processor.tokenizer.pad(
                label_features,
                return_tensors="pt",
            )

            labels = labels_batch["input_ids"].masked_fill(
                labels_batch["attention_mask"].ne(1),
                -100,
            )

            decoder_start_token_id = (
                self.processor.tokenizer.convert_tokens_to_ids(
                    "<|startoftranscript|>"
                )
            )

            if (
                labels.shape[1] > 0
                and (
                    labels[:, 0] == decoder_start_token_id
                ).all().item()
            ):
                labels = labels[:, 1:]

            batch["labels"] = labels
            return batch

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor
    )

    wer_metric = evaluate.load("wer")
    cer_metric = evaluate.load("cer")

    def compute_metrics(pred):
        pred_ids = pred.predictions

        if isinstance(pred_ids, tuple):
            pred_ids = pred_ids[0]

        label_ids = pred.label_ids.copy()

        label_ids[label_ids == -100] = (
            processor.tokenizer.pad_token_id
        )

        pred_str = processor.tokenizer.batch_decode(
            pred_ids,
            skip_special_tokens=True,
        )

        label_str = processor.tokenizer.batch_decode(
            label_ids,
            skip_special_tokens=True,
        )

        pred_str = [
            " ".join(text.strip().split())
            for text in pred_str
        ]

        label_str = [
            " ".join(text.strip().split())
            for text in label_str
        ]

        return {
            "wer": wer_metric.compute(
                predictions=pred_str,
                references=label_str,
            ),
            "cer": cer_metric.compute(
                predictions=pred_str,
                references=label_str,
            ),
        }

    # Preserve the final notebook configuration.
    batch_size = 2
    gradient_accumulation = 8
    epochs = 3

    batches_per_epoch = math.ceil(
        len(train_dataset) / batch_size
    )

    steps_per_epoch = math.ceil(
        batches_per_epoch / gradient_accumulation
    )

    total_steps = math.ceil(
        steps_per_epoch * epochs
    )

    warmup_steps = math.ceil(total_steps * 0.05)

    supported_args = inspect.signature(
        Seq2SeqTrainingArguments
    ).parameters

    if "eval_strategy" in supported_args:
        eval_key = "eval_strategy"
    elif "evaluation_strategy" in supported_args:
        eval_key = "evaluation_strategy"
    else:
        raise RuntimeError(
            "Unsupported Transformers evaluation strategy API."
        )

    training_kwargs = {
        "output_dir": str(checkpoint_dir),
        "learning_rate": 1e-5,
        "num_train_epochs": epochs,
        "per_device_train_batch_size": batch_size,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": gradient_accumulation,
        "weight_decay": 0.01,
        "warmup_steps": warmup_steps,
        "lr_scheduler_type": "linear",
        "fp16": True,
        "gradient_checkpointing": True,
        "logging_strategy": "steps",
        "logging_steps": 25,
        "save_strategy": "steps",
        "save_steps": 200,
        "eval_steps": 200,
        "save_total_limit": 2,
        "predict_with_generate": True,
        "generation_max_length": 225,
        "load_best_model_at_end": True,
        "metric_for_best_model": "wer",
        "greater_is_better": False,
        "seed": SEED,
        "report_to": "none",
        "remove_unused_columns": False,
        "dataloader_num_workers": 0,
    }

    training_kwargs[eval_key] = "steps"

    unsupported = [
        key
        for key in training_kwargs
        if key not in supported_args
    ]

    if unsupported:
        raise RuntimeError(
            f"Unsupported training arguments: {unsupported}"
        )

    training_args = Seq2SeqTrainingArguments(
        **training_kwargs
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=processor,
    )

    last_checkpoint = get_last_checkpoint(
        str(checkpoint_dir)
    )

    print("GPU:", torch.cuda.get_device_name(0))
    print("Training samples:", len(train_dataset))
    print("Validation samples:", len(eval_dataset))
    print("Warmup steps:", warmup_steps)
    print("Checkpoint interval:", 200)
    print("Resume checkpoint:", last_checkpoint)

    config = {
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "dataset_selection": DATASET_SELECTION,
        "dataset_commit": DATASET_COMMIT,
        "seed": SEED,
        "epochs": epochs,
        "learning_rate": 1e-5,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation,
        "warmup_steps": warmup_steps,
        "save_steps": 200,
        "eval_steps": 200,
    }

    (results_dir / "training_config.json").write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    start_time = time.time()

    train_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )

    duration = time.time() - start_time

    results = {
        **config,
        "gpu": torch.cuda.get_device_name(0),
        "training_samples": len(train_dataset),
        "validation_samples": len(eval_dataset),
        "training_duration_seconds": duration,
        "training_loss": train_result.training_loss,
        "global_step": trainer.state.global_step,
    }

    (results_dir / "training_results.json").write_text(
        json.dumps(results, indent=2),
        encoding="utf-8",
    )

    print("Training completed.")
    print("Results:", results_dir)


if __name__ == "__main__":
    main()
'''

script = script.replace(
    "__MODEL_REVISION__", MODEL_REVISION
).replace(
    "__DATASET_COMMIT__", DATASET_COMMIT
)

script_path.write_text(script, encoding="utf-8")

# Validate Python syntax without importing training libraries.
compile(script, str(script_path), "exec")

print("STEP 35 — SCRIPT CREATION")
print("=" * 60)
print("File:", script_path.relative_to(repo))
print("Python syntax: PASS")
print("Pinned model revision: PASS")
print("Pinned dataset commit: PASS")
print("Standalone entry point: PASS")
print("Training executed: NO")

STEP 35 — SCRIPT CREATION
File: training/train_whisper.py
Python syntax: PASS
Pinned model revision: PASS
Pinned dataset commit: PASS
Standalone entry point: PASS
Training executed: NO


In [17]:
from pathlib import Path
import subprocess
import sys

repo = Path("/content/icelandic-asr-merge-check")
script = repo / "training" / "train_whisper.py"

dataset_repo = Path("/content/icelandic-asr-project3")

expected_commit = "55f5a6856e92cdf89608c492cc470aea0d019e1b"

print("STEP 36 — STANDALONE SCRIPT VALIDATION")
print("=" * 65)

# Test 1: Python syntax
result = subprocess.run(
    [sys.executable, "-m", "py_compile", str(script)],
    capture_output=True,
    text=True
)

print("\nTEST 1 — PYTHON SYNTAX")
print("PASS" if result.returncode == 0 else "FAIL")

if result.returncode != 0:
    print(result.stderr)

# Test 2: CLI help
result = subprocess.run(
    [sys.executable, str(script), "--help"],
    capture_output=True,
    text=True
)

print("\nTEST 2 — COMMAND-LINE INTERFACE")
print("PASS" if result.returncode == 0 else "FAIL")
print(result.stdout[:2000])

if result.returncode != 0:
    print(result.stderr)

# Test 3: Dataset revision
print("\nTEST 3 — DATASET REVISION")

if dataset_repo.exists():
    result = subprocess.run(
        ["git", "-C", str(dataset_repo), "rev-parse", "HEAD"],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        actual_commit = result.stdout.strip()

        print("Expected:", expected_commit)
        print("Actual:  ", actual_commit)
        print(
            "Revision:",
            "PASS" if actual_commit == expected_commit
            else "DIFFERENT — checkout not modified"
        )
    else:
        print("Dataset directory exists but is not a Git checkout.")
else:
    print("Dataset directory not found.")

# Test 4: Check-only mode
print("\nTEST 4 — CHECK-ONLY MODE")

result = subprocess.run(
    [
        sys.executable,
        str(script),
        "--repo-dir",
        str(dataset_repo),
        "--check-only"
    ],
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print(result.stdout[:2000])

if result.returncode != 0:
    print(result.stderr[:2000])

print("\nFINAL STATUS")
print("=" * 65)
print("Training executed: NO")
print("Repository modified: NO")

STEP 36 — STANDALONE SCRIPT VALIDATION

TEST 1 — PYTHON SYNTAX
PASS

TEST 2 — COMMAND-LINE INTERFACE
PASS
usage: train_whisper.py [-h] [--repo-dir REPO_DIR] [--output-dir OUTPUT_DIR]
                        [--check-only]

Fine-tune Whisper Small on Icelandic Samromur data.

options:
  -h, --help            show this help message and exit
  --repo-dir REPO_DIR   Dataset repository containing output/ and data/.
  --output-dir OUTPUT_DIR
                        Directory for checkpoints and results.
  --check-only          Validate dataset paths without loading models or
                        training.


TEST 3 — DATASET REVISION
Dataset directory not found.

TEST 4 — CHECK-ONLY MODE
Exit code: 1

Traceback (most recent call last):
  File "/content/icelandic-asr-merge-check/training/train_whisper.py", line 489, in <module>
    main()
    ~~~~^^
  File "/content/icelandic-asr-merge-check/training/train_whisper.py", line 123, in main
    manifests, audio_root = validate_dataset(args.repo

In [18]:
from pathlib import Path
import subprocess
import os

print("STEP 37 — RESTORE PINNED DATASET")
print("=" * 65)

repo_url = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"

dataset_repo = Path("/content/icelandic-asr-project3")

dataset_commit = "55f5a6856e92cdf89608c492cc470aea0d019e1b"

env = {
    **os.environ,
    "GIT_LFS_SKIP_SMUDGE": "1"
}

# Clone without checking out files initially.
if not dataset_repo.exists():
    print("Cloning dataset repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "--no-checkout",
            repo_url,
            str(dataset_repo)
        ],
        check=True,
        env=env
    )
else:
    print("Dataset repository already exists.")

# Fetch the pinned commit.
subprocess.run(
    [
        "git", "-C", str(dataset_repo),
        "fetch", "origin", dataset_commit
    ],
    check=True
)

# Protect existing uncommitted changes.
status = subprocess.run(
    [
        "git", "-C", str(dataset_repo),
        "status", "--porcelain"
    ],
    capture_output=True,
    text=True,
    check=True
)

if status.stdout.strip():
    raise RuntimeError(
        "Dataset checkout contains uncommitted changes. "
        "Checkout aborted to protect existing files."
    )

# Check out the pinned dataset revision.
subprocess.run(
    [
        "git", "-C", str(dataset_repo),
        "checkout", "--detach", dataset_commit
    ],
    check=True,
    env=env
)

# Verify revision.
actual_commit = subprocess.run(
    [
        "git", "-C", str(dataset_repo),
        "rev-parse", "HEAD"
    ],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

assert actual_commit == dataset_commit

print("\nDATASET REVISION")
print("-" * 65)
print("Expected:", dataset_commit)
print("Actual:  ", actual_commit)
print("Revision verification: PASS")

# Check required files.
print("\nREQUIRED FILES")
print("-" * 65)

for filename in [
    "output/train.jsonl",
    "output/dev.jsonl",
    "output/test.jsonl"
]:
    path = dataset_repo / filename

    print(
        filename,
        "FOUND" if path.is_file() else "MISSING"
    )

print("\nSTEP 37 COMPLETE")
print("Training executed: NO")

STEP 37 — RESTORE PINNED DATASET
Cloning dataset repository...


RuntimeError: Dataset checkout contains uncommitted changes. Checkout aborted to protect existing files.

In [19]:
from pathlib import Path
import subprocess

dataset_repo = Path("/content/icelandic-asr-project3")

print("STEP 37A — DATASET CHECKOUT DIAGNOSIS")
print("=" * 65)

commands = {
    "Current HEAD": ["git", "rev-parse", "HEAD"],
    "Working tree status": ["git", "status", "--short", "-uno"],
    "Staged changes": ["git", "diff", "--cached", "--stat"],
    "Unstaged changes": ["git", "diff", "--stat"],
    "Checkout configuration": [
        "git", "config", "--get", "core.sparseCheckout"
    ],
}

for label, command in commands.items():
    result = subprocess.run(
        command,
        cwd=dataset_repo,
        capture_output=True,
        text=True
    )

    print(f"\n{label}")
    print("-" * 65)

    output = result.stdout.strip()

    if output:
        lines = output.splitlines()
        print("\n".join(lines[:15]))

        if len(lines) > 15:
            print(f"... and {len(lines) - 15} more lines")
    else:
        print("(none)")

print("\nDIAGNOSIS COMPLETE")
print("Files modified: NO")
print("Training executed: NO")

STEP 37A — DATASET CHECKOUT DIAGNOSIS

Current HEAD
-----------------------------------------------------------------
96e5839b27d9871d6b2408b3c3719dc809a730b7

Working tree status
-----------------------------------------------------------------
D  .gitattributes
D  .gitignore
D  BASELINES.md
D  DATA_STORAGE.md
D  LICENSE
D  NLP_Project3_Person3_Whisper_Clean.ipynb
D  README.md
D  audit.py
D  baselines/run_pretrained.py
D  contributions/Azaa_Almousli_2309115421_AzaaAlmousli.md
D  contributions/Muhammet_Celik_2509011060_MuhammetGoksenin.md
D  contributions/Ranim_Alyaseen_2309115360_RanimAlyaseen.md
D  contributions/Tasnim_Alibraheem_2309015259_tasneemzouhair-coder.md
D  data/cleaned/samromur-21.05-20h-v1/dev/000028/000028-0000186.flac
D  data/cleaned/samromur-21.05-20h-v1/dev/000028/000028-0000188.flac
... and 31122 more lines

Staged changes
-----------------------------------------------------------------
.gitattributes                                     |     15 -
 .gitignore       

In [20]:
from pathlib import Path
import subprocess
import os

dataset_repo = Path("/content/icelandic-asr-project3")

dataset_commit = "55f5a6856e92cdf89608c492cc470aea0d019e1b"

print("STEP 37B — COMPLETE DATASET CHECKOUT")
print("=" * 65)

env = {
    **os.environ,
    "GIT_LFS_SKIP_SMUDGE": "1"
}

# Confirm the directory is the intended Git repository.
assert dataset_repo.is_dir(), "Dataset repository not found."

remote = subprocess.run(
    ["git", "-C", str(dataset_repo), "remote", "get-url", "origin"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

assert "RanimAlyaseen/icelandic-asr-project3" in remote, (
    f"Unexpected repository: {remote}"
)

# Confirm there are no unstaged or untracked files.
unstaged = subprocess.run(
    ["git", "-C", str(dataset_repo), "diff", "--name-only"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

untracked = subprocess.run(
    ["git", "-C", str(dataset_repo), "ls-files", "--others", "--exclude-standard"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

assert not unstaged, "Unexpected unstaged changes detected."
assert not untracked, "Unexpected untracked files detected."

# Verify that staged changes are only deletions.
staged = subprocess.run(
    ["git", "-C", str(dataset_repo), "diff", "--cached", "--name-status"],
    capture_output=True,
    text=True,
    check=True
).stdout.splitlines()

assert staged, "Expected no-checkout state was not detected."

assert all(line.startswith("D\t") for line in staged), (
    "Unexpected staged changes detected."
)

print("Repository verified: PASS")
print("Initial no-checkout state verified: PASS")

# Restore the index and working tree at the pinned commit.
# Safe here because the checks above confirmed the initial
# no-checkout state and absence of additional user files.
subprocess.run(
    [
        "git", "-C", str(dataset_repo),
        "reset", "--hard", dataset_commit
    ],
    check=True,
    env=env
)

# Detach HEAD to make the dataset revision explicit.
subprocess.run(
    [
        "git", "-C", str(dataset_repo),
        "checkout", "--detach", dataset_commit
    ],
    check=True,
    env=env
)

actual_commit = subprocess.run(
    ["git", "-C", str(dataset_repo), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

assert actual_commit == dataset_commit

print("\nDATASET REVISION")
print("-" * 65)
print("Expected:", dataset_commit)
print("Actual:  ", actual_commit)
print("Revision verification: PASS")

print("\nREQUIRED MANIFESTS")
print("-" * 65)

for filename in [
    "output/train.jsonl",
    "output/dev.jsonl",
    "output/test.jsonl"
]:
    path = dataset_repo / filename
    print(filename, "FOUND" if path.is_file() else "MISSING")

print("\nSTEP 37B COMPLETE")
print("Training executed: NO")

STEP 37B — COMPLETE DATASET CHECKOUT
Repository verified: PASS
Initial no-checkout state verified: PASS

DATASET REVISION
-----------------------------------------------------------------
Expected: 55f5a6856e92cdf89608c492cc470aea0d019e1b
Actual:   55f5a6856e92cdf89608c492cc470aea0d019e1b
Revision verification: PASS

REQUIRED MANIFESTS
-----------------------------------------------------------------
output/train.jsonl FOUND
output/dev.jsonl FOUND
output/test.jsonl FOUND

STEP 37B COMPLETE
Training executed: NO


In [21]:
from pathlib import Path
import subprocess
import sys
import json

dev_repo = Path("/content/icelandic-asr-merge-check")
dataset_repo = Path("/content/icelandic-asr-project3")

script = dev_repo / "training" / "train_whisper.py"

print("STEP 38 — DATASET AND AUDIO VALIDATION")
print("=" * 65)

# Test 1: Standalone script validation
print("\nTEST 1 — CHECK-ONLY MODE")
print("-" * 65)

result = subprocess.run(
    [
        sys.executable,
        str(script),
        "--repo-dir",
        str(dataset_repo),
        "--check-only"
    ],
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr[-2000:])

# Test 2: Audio availability
print("\nTEST 2 — AUDIO FILE AVAILABILITY")
print("-" * 65)

audio_root = (
    dataset_repo
    / "data"
    / "cleaned"
    / "samromur-21.05-20h-v1"
)

for split, filename in [
    ("train", "train.jsonl"),
    ("validation", "dev.jsonl"),
    ("test", "test.jsonl")
]:
    manifest = dataset_repo / "output" / filename

    with manifest.open(encoding="utf-8") as f:
        samples = [
            json.loads(line)
            for _, line in zip(range(5), f)
            if line.strip()
        ]

    print(f"\n{split.upper()}")

    for sample in samples:
        audio_path = audio_root / sample["audio_path"]

        if not audio_path.exists():
            status = "MISSING"
            size = 0
        else:
            size = audio_path.stat().st_size

            with audio_path.open("rb") as f:
                header = f.read(80)

            if header.startswith(b"version https://git-lfs.github.com/spec"):
                status = "GIT LFS POINTER"
            else:
                status = "FILE PRESENT"

        print(
            f"{audio_path.name}: "
            f"{status} ({size:,} bytes)"
        )

print("\nSTEP 38 COMPLETE")
print("Training executed: NO")

STEP 38 — DATASET AND AUDIO VALIDATION

TEST 1 — CHECK-ONLY MODE
-----------------------------------------------------------------
Exit code: 1

Traceback (most recent call last):
  File "/content/icelandic-asr-merge-check/training/train_whisper.py", line 489, in <module>
    main()
    ~~~~^^
  File "/content/icelandic-asr-merge-check/training/train_whisper.py", line 123, in main
    manifests, audio_root = validate_dataset(args.repo_dir)
                            ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/content/icelandic-asr-merge-check/training/train_whisper.py", line 109, in validate_dataset
    raise FileNotFoundError(
        f"Prepared audio directory not found: {audio_root}"
    )
FileNotFoundError: Prepared audio directory not found: /content/icelandic-asr-project3/data/cleaned/samromur-21.05-20h-v1


TEST 2 — AUDIO FILE AVAILABILITY
-----------------------------------------------------------------

TRAIN
007478-0072202.flac: MISSING (0 bytes)
007310-0069817.flac: MISSING (0

In [22]:
from pathlib import Path
import subprocess
import json

repo = Path("/content/icelandic-asr-project3")

print("STEP 39 — GIT LFS AUDIO INSPECTION")
print("=" * 65)

# Check Git LFS availability
result = subprocess.run(
    ["git", "lfs", "version"],
    capture_output=True,
    text=True
)

print("\nGIT LFS")
print("-" * 65)
print(result.stdout.strip() or result.stderr.strip())

# Inspect repository LFS configuration
attributes = repo / ".gitattributes"

print("\nGIT ATTRIBUTES")
print("-" * 65)

if attributes.exists():
    print(attributes.read_text()[:3000])
else:
    print("No .gitattributes file found.")

# Inspect one audio file from the training manifest
manifest = repo / "output" / "train.jsonl"

with manifest.open(encoding="utf-8") as f:
    sample = json.loads(next(f))

relative_audio_path = (
    Path("data")
    / "cleaned"
    / "samromur-21.05-20h-v1"
    / sample["audio_path"]
)

print("\nSAMPLE AUDIO")
print("-" * 65)
print("Path:", relative_audio_path)

# Check whether Git tracks this audio file
result = subprocess.run(
    [
        "git", "-C", str(repo),
        "ls-files", "--error-unmatch",
        str(relative_audio_path)
    ],
    capture_output=True,
    text=True
)

print("Tracked by Git:", result.returncode == 0)

# Inspect the Git object without downloading audio
result = subprocess.run(
    [
        "git", "-C", str(repo),
        "show",
        f"HEAD:{relative_audio_path}"
    ],
    capture_output=True
)

if result.returncode == 0:
    content = result.stdout[:500]

    if content.startswith(
        b"version https://git-lfs.github.com/spec"
    ):
        print("Storage type: GIT LFS")
        print(content.decode("utf-8", errors="replace"))
    else:
        print("Storage type: Regular Git object")
        print("Object size:", len(result.stdout), "bytes")
else:
    print("Audio file not found in Git history.")
    print(result.stderr.decode("utf-8", errors="replace")[:500])

print("\nSTEP 39 COMPLETE")
print("Audio downloaded: NO")
print("Training executed: NO")

STEP 39 — GIT LFS AUDIO INSPECTION

GIT LFS
-----------------------------------------------------------------
git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	log

GIT ATTRIBUTES
-----------------------------------------------------------------
# Preserve the verified byte representation of manifests, references, and reports.
output/** -text whitespace=cr-at-eol
# Empty trailing TSV fields are part of the data.
output/*.tsv whitespace=cr-at-eol,-blank-at-eol


SAMPLE AUDIO
-----------------------------------------------------------------
Path: data/cleaned/samromur-21.05-20h-v1/train/007478/007478-0072202.flac
Tracked by Git: False
Audio file not found in Git history.
fatal: path 'data/cleaned/samromur-21.05-20h-v1/train/007478/007478-0072202.flac' does not exist in 'HEAD'


STEP 39 COMPLETE
Audio downloaded: NO
Training executed: NO


In [23]:
from pathlib import Path

repo = Path("/content/icelandic-asr-merge-check")

print("STEP 40 — DATASET RECOVERY INSTRUCTIONS")
print("=" * 70)

files = [
    "DATA_STORAGE.md",
    "README.md",
    "prepare.py",
    "preprocess.py",
]

keywords = [
    "download",
    "openslr",
    "samromur",
    "audio",
    "archive",
    "storage",
    "dataset",
    "cleaned",
    "drive",
    "tar",
    "extract",
]

for filename in files:
    path = repo / filename

    if not path.exists():
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    matching = [
        i for i, line in enumerate(lines)
        if any(
            keyword in line.lower()
            for keyword in keywords
        )
    ]

    selected = set()

    for i in matching:
        selected.update(
            range(
                max(0, i - 2),
                min(len(lines), i + 4)
            )
        )

    print(f"\n{'=' * 70}")
    print(filename)
    print("=" * 70)

    for i in sorted(selected)[:160]:
        print(f"{i + 1:4}: {lines[i]}")

print("\nSTEP 40 COMPLETE")
print("Files modified: NO")
print("Audio downloaded: NO")
print("Training executed: NO")

STEP 40 — DATASET RECOVERY INSTRUCTIONS

DATA_STORAGE.md
   1: # Raw and cleaned data
   2: 
   3: The professor accepts a documented raw subset and Git LFS in this public repository. The raw root is **data/raw/samromur_21.05/** and the cleaned root is **data/cleaned/samromur-21.05-20h-v1/**. The original release documents and the frozen manifests use ordinary Git.
   4: 
   5: ## Raw scope and cleaned output
   6: 
   7: Raw audio is the original **released** FLAC. The upstream release converted earlier recording formats. The bounded input contains each frozen v1 utterance plus the smallest numeric utterance ID in each represented official split and session. If that ID is already selected, include it once. This adds a first-utterance example from every represented session using metadata order before considering audio quality or validation flags. Unrepresented sessions stay outside the scope. [Subset inventory](output/raw_subset_inventory.tsv)
   8: 
   9: | Audio | Raw candidates | Ra

In [24]:
from pathlib import Path
import subprocess

repo = Path("/content/icelandic-asr-project3")

print("STEP 41 — VERIFY CURRENT AUDIO STORAGE")
print("=" * 65)

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True
    )

# Fetch current main without changing the working tree.
result = git("fetch", "origin", "main")

if result.returncode != 0:
    raise RuntimeError(result.stderr)

revisions = {
    "Pinned dataset commit":
        "55f5a6856e92cdf89608c492cc470aea0d019e1b",
    "Current origin/main":
        "origin/main",
}

sample = (
    "data/cleaned/samromur-21.05-20h-v1/"
    "train/007478/007478-0072202.flac"
)

for label, revision in revisions.items():

    print(f"\n{label}")
    print("-" * 65)

    commit = git("rev-parse", revision)
    print("Commit:", commit.stdout.strip())

    tracked = git(
        "ls-tree",
        "-r",
        "--name-only",
        revision,
        "--",
        sample
    )

    print(
        "Sample audio tracked:",
        bool(tracked.stdout.strip())
    )

    attributes = git(
        "show",
        f"{revision}:.gitattributes"
    )

    if attributes.returncode == 0:
        lfs_lines = [
            line
            for line in attributes.stdout.splitlines()
            if "filter=lfs" in line
        ]

        print("Git LFS rules:")
        print("\n".join(lfs_lines) if lfs_lines else "None")
    else:
        print(".gitattributes unavailable")

    audio = git(
        "show",
        f"{revision}:{sample}"
    )

    if audio.returncode == 0:
        if audio.stdout.startswith(
            "version https://git-lfs.github.com/spec"
        ):
            print("Sample storage: Git LFS pointer")
            print(audio.stdout[:250])
        else:
            print("Sample storage: Regular Git object")
    else:
        print("Sample storage: Not tracked")

print("\nSTEP 41 COMPLETE")
print("Working tree modified: NO")
print("Audio downloaded: NO")
print("Training executed: NO")

STEP 41 — VERIFY CURRENT AUDIO STORAGE

Pinned dataset commit
-----------------------------------------------------------------
Commit: 55f5a6856e92cdf89608c492cc470aea0d019e1b
Sample audio tracked: False
Git LFS rules:
None
Sample storage: Not tracked

Current origin/main
-----------------------------------------------------------------
Commit: 96e5839b27d9871d6b2408b3c3719dc809a730b7
Sample audio tracked: True
Git LFS rules:
data/raw/samromur_21.05/*/*/*.flac filter=lfs diff=lfs merge=lfs -text
data/cleaned/samromur-21.05-20h-v1/*/*/*.flac filter=lfs diff=lfs merge=lfs -text
models/mfcc_gmm_hmm/*.joblib filter=lfs diff=lfs merge=lfs -text
models/mfcc_gmm_hmm_aligned/*.joblib filter=lfs diff=lfs merge=lfs -text
Sample storage: Git LFS pointer
version https://git-lfs.github.com/spec/v1
oid sha256:74f488d29d5b30ffc60dc9a7fda3594a4223586f7ab9d06680fbaa3244f00296
size 47430


STEP 41 COMPLETE
Working tree modified: NO
Audio downloaded: NO
Training executed: NO


In [25]:
from pathlib import Path
import subprocess
import hashlib

repo = Path("/content/icelandic-asr-project3")

frozen_commit = "55f5a6856e92cdf89608c492cc470aea0d019e1b"
audio_revision = "origin/main"

manifests = [
    "output/train.jsonl",
    "output/dev.jsonl",
    "output/test.jsonl",
]

print("STEP 42 — MANIFEST INTEGRITY VERIFICATION")
print("=" * 70)

def git_file(revision, filename):
    result = subprocess.run(
        [
            "git", "-C", str(repo),
            "show", f"{revision}:{filename}"
        ],
        capture_output=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.decode(errors="replace")
        )

    return result.stdout

all_match = True

for filename in manifests:
    frozen = git_file(frozen_commit, filename)
    current = git_file(audio_revision, filename)

    frozen_hash = hashlib.sha256(frozen).hexdigest()
    current_hash = hashlib.sha256(current).hexdigest()

    match = frozen_hash == current_hash
    all_match &= match

    print(f"\n{filename}")
    print("-" * 70)
    print("Frozen SHA256: ", frozen_hash)
    print("Current SHA256:", current_hash)
    print("Byte-for-byte: ", "PASS" if match else "FAIL")

print("\nFINAL RESULT")
print("=" * 70)

print("All manifests identical:", all_match)

if all_match:
    print("PASS — Current main preserves the frozen manifests.")
else:
    print("FAIL — Dataset revisions differ; investigate before switching.")

print("\nWorking tree modified: NO")
print("Audio downloaded: NO")
print("Training executed: NO")

STEP 42 — MANIFEST INTEGRITY VERIFICATION

output/train.jsonl
----------------------------------------------------------------------
Frozen SHA256:  d36a1adccdb10682b30a6fd8b2f7e20f8c86babb4b1d20a007af79179c4436ed
Current SHA256: d36a1adccdb10682b30a6fd8b2f7e20f8c86babb4b1d20a007af79179c4436ed
Byte-for-byte:  PASS

output/dev.jsonl
----------------------------------------------------------------------
Frozen SHA256:  f0e7675cd31497788250728bba966d1d884ad5f929b3488724b2fef2110f9bf4
Current SHA256: f0e7675cd31497788250728bba966d1d884ad5f929b3488724b2fef2110f9bf4
Byte-for-byte:  PASS

output/test.jsonl
----------------------------------------------------------------------
Frozen SHA256:  f5e346f545a5d7f4298f1838bc63b978a4d7ab9d063ddc97897fa065223d0481
Current SHA256: f5e346f545a5d7f4298f1838bc63b978a4d7ab9d063ddc97897fa065223d0481
Byte-for-byte:  PASS

FINAL RESULT
All manifests identical: True
PASS — Current main preserves the frozen manifests.

Working tree modified: NO
Audio downloaded

In [26]:
from pathlib import Path
import ast

script = Path(
    "/content/icelandic-asr-merge-check/training/train_whisper.py"
)

print("STEP 43 — INSPECT DATASET VALIDATION")
print("=" * 70)

source = script.read_text(encoding="utf-8")
lines = source.splitlines()

tree = ast.parse(source)

# Locate the dataset validation function.
functions = [
    node
    for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    and node.name == "validate_dataset"
]

assert len(functions) == 1, (
    "Expected exactly one validate_dataset function."
)

function = functions[0]

print("\nCURRENT VALIDATION FUNCTION")
print("-" * 70)

for line_number in range(function.lineno, function.end_lineno + 1):
    print(f"{line_number:4}: {lines[line_number - 1]}")

print("\nDATASET-RELATED CONSTANTS")
print("-" * 70)

for line_number, line in enumerate(lines[:125], start=1):
    if any(
        keyword in line
        for keyword in [
            "DATASET_COMMIT",
            "DATASET_REVISION",
            "MODEL_REVISION",
            "MANIFEST",
            "AUDIO_ROOT",
        ]
    ):
        print(f"{line_number:4}: {line}")

print("\nSTEP 43 COMPLETE")
print("Files modified: NO")
print("Training executed: NO")

STEP 43 — INSPECT DATASET VALIDATION

CURRENT VALIDATION FUNCTION
----------------------------------------------------------------------
  72: def validate_dataset(repo_dir):
  73:     repo_dir = repo_dir.resolve()
  74: 
  75:     if not repo_dir.is_dir():
  76:         raise FileNotFoundError(
  77:             f"Dataset repository not found: {repo_dir}"
  78:         )
  79: 
  80:     actual_commit = get_repository_revision(repo_dir)
  81: 
  82:     if actual_commit != DATASET_COMMIT:
  83:         raise RuntimeError(
  84:             "Dataset repository revision mismatch.\n"
  85:             f"Expected: {DATASET_COMMIT}\n"
  86:             f"Actual:   {actual_commit}\n"
  87:             "Check out the pinned dataset commit before training."
  88:         )
  89: 
  90:     output_dir = repo_dir / "output"
  91: 
  92:     manifests = {
  93:         "train": output_dir / "train.jsonl",
  94:         "validation": output_dir / "dev.jsonl",
  95:         "test": output_dir / "t

In [27]:
from pathlib import Path
import ast
import hashlib
import subprocess

script = Path(
    "/content/icelandic-asr-merge-check/training/train_whisper.py"
)

print("STEP 44 — UPDATE DATASET VALIDATION")
print("=" * 70)

source = script.read_text(encoding="utf-8")

# Locate the existing function using Python's AST.
tree = ast.parse(source)

matches = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "validate_dataset"
]

assert len(matches) == 1, "Expected exactly one validation function."

node = matches[0]
lines = source.splitlines(keepends=True)

# Preserve a backup before changing anything.
backup = script.with_suffix(".py.step44_backup")
assert not backup.exists(), (
    f"Backup already exists: {backup}. "
    "Stop to avoid overwriting an earlier backup."
)
backup.write_text(source, encoding="utf-8")

replacement = '''def validate_dataset(repo_dir):
    """Verify frozen manifests and prepared audio availability."""
    repo_dir = repo_dir.resolve()

    if not repo_dir.is_dir():
        raise FileNotFoundError(
            f"Dataset repository not found: {repo_dir}"
        )

    actual_commit = get_repository_revision(repo_dir)

    expected_hashes = {
        "train": "d36a1adccdb10682b30a6fd8b2f7e20f8c86babb4b1d20a007af79179c4436ed",
        "validation": "f0e7675cd31497788250728bba966d1d884ad5f929b3488724b2fef2110f9bf4",
        "test": "f5e346f545a5d7f4298f1838bc63b978a4d7ab9d063ddc97897fa065223d0481",
    }

    output_dir = repo_dir / "output"

    manifests = {
        "train": output_dir / "train.jsonl",
        "validation": output_dir / "dev.jsonl",
        "test": output_dir / "test.jsonl",
    }

    for split, path in manifests.items():
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {split} manifest: {path}"
            )

        actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()

        if actual_hash != expected_hashes[split]:
            raise RuntimeError(
                f"{split} manifest SHA-256 mismatch.\\\\n"
                f"Expected: {expected_hashes[split]}\\\\n"
                f"Actual:   {actual_hash}\\\\n"
                "The dataset differs from frozen v1."
            )

    audio_root = (
        repo_dir / "data" / "cleaned" / DATASET_SELECTION
    )

    if not audio_root.is_dir():
        raise FileNotFoundError(
            f"Prepared audio directory not found: {audio_root}\\\\n"
            "The frozen manifests are verified, but the audio "
            "is not available. Use the repository revision "
            "containing Git LFS audio pointers and download "
            "the cleaned recordings before training."
        )

    print("Dataset repository:", repo_dir)
    print("Dataset checkout:", actual_commit)
    print("Frozen preparation commit:", DATASET_COMMIT)
    print("Dataset selection:", DATASET_SELECTION)
    print("Manifest SHA-256 verification: PASS")

    return manifests, audio_root
'''

# Add hashlib only if not already imported.
if not any(
    isinstance(item, ast.Import)
    and any(alias.name == "hashlib" for alias in item.names)
    for item in tree.body
):
    lines.insert(0, "import hashlib\n")

    # Adjust AST line positions after insertion.
    start = node.lineno
    end = node.end_lineno
else:
    start = node.lineno - 1
    end = node.end_lineno

# Replace the original function.
lines[start:end] = [replacement + "\n"]

updated = "".join(lines)

try:
    compile(updated, str(script), "exec")
except SyntaxError:
    # Leave the original file untouched if validation fails.
    raise

script.write_text(updated, encoding="utf-8")

print("Validation function updated: PASS")
print("Python syntax: PASS")
print("Backup:", backup.name)
print("Training executed: NO")

STEP 44 — UPDATE DATASET VALIDATION
Validation function updated: PASS
Python syntax: PASS
Backup: train_whisper.py.step44_backup
Training executed: NO


In [28]:
from pathlib import Path
import ast
import subprocess

script = Path(
    "/content/icelandic-asr-merge-check/training/train_whisper.py"
)

print("STEP 45 — STRENGTHEN AUDIO VALIDATION")
print("=" * 70)

source = script.read_text(encoding="utf-8")
tree = ast.parse(source)

# Locate the validation function.
matches = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "validate_dataset"
]

assert len(matches) == 1

node = matches[0]
lines = source.splitlines(keepends=True)

function_source = "".join(
    lines[node.lineno - 1:node.end_lineno]
)

# Ensure we are modifying the expected version.
assert "Manifest SHA-256 verification: PASS" in function_source
assert "return manifests, audio_root" in function_source
assert "Audio availability verification: PASS" not in function_source

# Ensure required imports exist.
imports = {
    alias.name
    for item in tree.body
    if isinstance(item, ast.Import)
    for alias in item.names
}

assert "hashlib" in imports, "hashlib import missing"

# Add json import if necessary.
if "json" not in imports:
    source = "import json\n" + source

# Insert the audio check before the function returns.
old = '''    return manifests, audio_root
'''

new = '''    # Verify every referenced audio file without decoding it.
    missing = []
    pointers = []
    checked = 0

    for split, manifest_path in manifests.items():
        with manifest_path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue

                record = json.loads(line)
                relative = Path(record["audio_path"])

                # Prevent absolute paths and directory traversal.
                if (
                    relative.is_absolute()
                    or ".." in relative.parts
                ):
                    raise ValueError(
                        f"Unsafe audio path in {split} "
                        f"manifest line {line_number}: {relative}"
                    )

                audio_path = audio_root / relative
                checked += 1

                if not audio_path.is_file():
                    missing.append(str(relative))
                    continue

                with audio_path.open("rb") as audio_file:
                    header = audio_file.read(80)

                if header.startswith(
                    b"version https://git-lfs.github.com/spec"
                ):
                    pointers.append(str(relative))
                elif not header.startswith(b"fLaC"):
                    raise RuntimeError(
                        f"Invalid FLAC header: {audio_path}"
                    )

    if missing or pointers:
        raise RuntimeError(
            "Audio hydration incomplete.\\\\n"
            f"Checked: {checked} recordings\\\\n"
            f"Missing: {len(missing)}\\\\n"
            f"Git LFS pointers: {len(pointers)}\\\\n"
            f"First missing: {missing[:3]}\\\\n"
            f"First pointers: {pointers[:3]}\\\\n"
            "Install Git LFS and download the cleaned "
            "audio files before training."
        )

    print(
        f"Audio availability verification: PASS "
        f"({checked} recordings)"
    )

    return manifests, audio_root
'''

assert source.count(old) == 1, (
    "Expected exactly one return statement."
)

updated = source.replace(old, new, 1)

# Validate before writing.
compile(updated, str(script), "exec")

script.write_text(updated, encoding="utf-8")

# Confirm the file remains syntactically valid.
result = subprocess.run(
    ["python", "-m", "py_compile", str(script)],
    capture_output=True,
    text=True
)

assert result.returncode == 0, result.stderr

print("Audio validation added: PASS")
print("Python syntax: PASS")
print("Frozen manifest verification preserved: PASS")
print("Training executed: NO")

STEP 45 — STRENGTHEN AUDIO VALIDATION


AssertionError: Expected exactly one return statement.

In [29]:
from pathlib import Path
import ast
import subprocess

script = Path(
    "/content/icelandic-asr-merge-check/training/train_whisper.py"
)

print("STEP 45A — CORRECT AUDIO VALIDATION PATCH")
print("=" * 70)

source = script.read_text(encoding="utf-8")
tree = ast.parse(source)

# Find the existing validation function.
matches = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "validate_dataset"
]

assert len(matches) == 1, "Expected one validate_dataset function."

function = matches[0]

# Ensure Step 44 was applied.
function_source = ast.get_source_segment(source, function)

assert "Manifest SHA-256 verification: PASS" in function_source
assert "Audio availability verification: PASS" not in function_source

# Find the final return statement inside validate_dataset.
returns = [
    node for node in function.body
    if isinstance(node, ast.Return)
]

assert len(returns) == 1, (
    "Expected exactly one top-level return statement."
)

return_node = returns[0]

# Insert the check immediately before that return.
validation_code = '''
    # Verify that all manifest recordings are present and hydrated.
    missing = []
    pointers = []
    checked = 0

    for split, manifest_path in manifests.items():
        with manifest_path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue

                record = json.loads(line)
                relative = Path(record["audio_path"])

                if relative.is_absolute() or ".." in relative.parts:
                    raise ValueError(
                        f"Unsafe audio path in {split} "
                        f"manifest line {line_number}: {relative}"
                    )

                audio_path = audio_root / relative
                checked += 1

                if not audio_path.is_file():
                    missing.append(str(relative))
                    continue

                with audio_path.open("rb") as audio_file:
                    header = audio_file.read(80)

                if header.startswith(
                    b"version https://git-lfs.github.com/spec"
                ):
                    pointers.append(str(relative))
                elif not header.startswith(b"fLaC"):
                    raise RuntimeError(
                        f"Invalid FLAC header: {audio_path}"
                    )

    if missing or pointers:
        raise RuntimeError(
            "Audio hydration incomplete.\\\\n"
            f"Checked: {checked} recordings\\\\n"
            f"Missing: {len(missing)}\\\\n"
            f"Git LFS pointers: {len(pointers)}\\\\n"
            f"First missing: {missing[:3]}\\\\n"
            f"First pointers: {pointers[:3]}\\\\n"
            "Install Git LFS and download the cleaned "
            "recordings before training."
        )

    print(
        f"Audio availability verification: PASS "
        f"({checked} recordings)"
    )

'''

lines = source.splitlines(keepends=True)

# Insert using the AST-derived line number.
lines.insert(
    return_node.lineno - 1,
    validation_code
)

updated = "".join(lines)

# Ensure json is imported.
updated_tree = ast.parse(updated)

has_json_import = any(
    isinstance(node, ast.Import)
    and any(alias.name == "json" for alias in node.names)
    for node in updated_tree.body
)

if not has_json_import:
    # Insert after the module docstring and any __future__ imports.
    parsed = ast.parse(updated)
    insertion_line = 0

    if (
        parsed.body
        and isinstance(parsed.body[0], ast.Expr)
        and isinstance(parsed.body[0].value, ast.Constant)
        and isinstance(parsed.body[0].value.value, str)
    ):
        insertion_line = parsed.body[0].end_lineno

    for node in parsed.body:
        if isinstance(node, ast.ImportFrom) and node.module == "__future__":
            insertion_line = max(insertion_line, node.end_lineno)

    updated_lines = updated.splitlines(keepends=True)
    updated_lines.insert(insertion_line, "import json\n")
    updated = "".join(updated_lines)

# Verify syntax before writing.
compile(updated, str(script), "exec")

script.write_text(updated, encoding="utf-8")

result = subprocess.run(
    ["python", "-m", "py_compile", str(script)],
    capture_output=True,
    text=True
)

assert result.returncode == 0, result.stderr

print("Audio validation added: PASS")
print("Python syntax: PASS")
print("Frozen manifest verification preserved: PASS")
print("Training executed: NO")

STEP 45A — CORRECT AUDIO VALIDATION PATCH


AssertionError: Expected exactly one top-level return statement.

In [30]:
from pathlib import Path
import ast

script = Path(
    "/content/icelandic-asr-merge-check/training/train_whisper.py"
)

print("STEP 45B — INSPECT VALIDATION FUNCTION")
print("=" * 70)

source = script.read_text(encoding="utf-8")
tree = ast.parse(source)
lines = source.splitlines()

functions = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "validate_dataset"
]

assert len(functions) == 1
function = functions[0]

print("\nCOMPLETE VALIDATION FUNCTION")
print("-" * 70)

for number in range(function.lineno, function.end_lineno + 1):
    print(f"{number:4}: {lines[number - 1]}")

print("\nRETURN STATEMENTS")
print("-" * 70)

for node in ast.walk(function):
    if isinstance(node, ast.Return):
        expression = ast.get_source_segment(source, node)
        print(f"Line {node.lineno}: {expression}")

print("\nFILE STATUS")
print("-" * 70)
print("Syntax:", "PASS")
print("Files modified: NO")
print("Training executed: NO")

STEP 45B — INSPECT VALIDATION FUNCTION

COMPLETE VALIDATION FUNCTION
----------------------------------------------------------------------
  73: def validate_dataset(repo_dir):
  74:     """Verify frozen manifests and prepared audio availability."""
  75:     repo_dir = repo_dir.resolve()
  76: 
  77:     if not repo_dir.is_dir():
  78:         raise FileNotFoundError(
  79:             f"Dataset repository not found: {repo_dir}"
  80:         )
  81: 
  82:     actual_commit = get_repository_revision(repo_dir)
  83: 
  84:     expected_hashes = {
  85:         "train": "d36a1adccdb10682b30a6fd8b2f7e20f8c86babb4b1d20a007af79179c4436ed",
  86:         "validation": "f0e7675cd31497788250728bba966d1d884ad5f929b3488724b2fef2110f9bf4",
  87:         "test": "f5e346f545a5d7f4298f1838bc63b978a4d7ab9d063ddc97897fa065223d0481",
  88:     }
  89: 
  90:     output_dir = repo_dir / "output"
  91: 
  92:     manifests = {
  93:         "train": output_dir / "train.jsonl",
  94:         "validatio

In [31]:
from pathlib import Path
import ast

script = Path(
    "/content/icelandic-asr-merge-check/training/train_whisper.py"
)

print("STEP 45C — FINAL AUDIO VALIDATION FIX")
print("=" * 70)

source = script.read_text(encoding="utf-8")
tree = ast.parse(source)
lines = source.splitlines(keepends=True)

functions = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "validate_dataset"
]

assert len(functions) == 1
function = functions[0]

returns = [
    node for node in function.body
    if isinstance(node, ast.Return)
]

assert len(returns) == 2, (
    "Unexpected function structure. No changes made."
)

assert all(
    ast.unparse(node.value) == "manifests, audio_root"
    for node in returns
), "Unexpected return values. No changes made."

assert (
    "Audio availability verification: PASS"
    not in ast.get_source_segment(source, function)
), "Audio validation already exists."

# Remove both original returns, then insert one validated return.
for node in sorted(returns, key=lambda x: x.lineno, reverse=True):
    del lines[node.lineno - 1:node.end_lineno]

# Locate the insertion point immediately after the existing
# manifest verification success message.
source_without_returns = "".join(lines)

marker = '    print("Manifest SHA-256 verification: PASS")\n'

assert source_without_returns.count(marker) == 1, (
    "Expected insertion point not found."
)

audio_validation = '''
    # Check that all frozen-manifest audio files are available.
    missing = []
    pointers = []
    checked = 0

    for split, manifest_path in manifests.items():
        with manifest_path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue

                record = json.loads(line)
                relative = Path(record["audio_path"])

                if (
                    relative.is_absolute()
                    or ".." in relative.parts
                    or relative.parts[0] not in ("train", "dev", "test")
                ):
                    raise ValueError(
                        f"Unsafe audio path in {split} "
                        f"manifest line {line_number}: {relative}"
                    )

                audio_path = audio_root / relative
                checked += 1

                if not audio_path.is_file():
                    missing.append(str(relative))
                    continue

                with audio_path.open("rb") as audio_file:
                    header = audio_file.read(80)

                if header.startswith(
                    b"version https://git-lfs.github.com/spec"
                ):
                    pointers.append(str(relative))

                elif not header.startswith(b"fLaC"):
                    raise RuntimeError(
                        f"Invalid FLAC header: {audio_path}"
                    )

    if missing or pointers:
        raise RuntimeError(
            "Audio hydration incomplete.\\n"
            f"Checked: {checked} recordings\\n"
            f"Missing: {len(missing)}\\n"
            f"Git LFS pointers: {len(pointers)}\\n"
            f"First missing: {missing[:3]}\\n"
            f"First pointers: {pointers[:3]}\\n"
            "Install Git LFS and download the cleaned "
            "recordings before training."
        )

    print(
        f"Audio availability verification: PASS "
        f"({checked} recordings)"
    )

    return manifests, audio_root
'''

updated = source_without_returns.replace(
    marker,
    marker + audio_validation,
    1
)

# Ensure json is imported.
updated_tree = ast.parse(updated)

has_json = any(
    isinstance(node, ast.Import)
    and any(alias.name == "json" for alias in node.names)
    for node in updated_tree.body
)

if not has_json:
    # Insert after any module docstring or future imports.
    insertion_line = 0

    for node in updated_tree.body:
        if isinstance(node, ast.ImportFrom) and node.module == "__future__":
            insertion_line = max(insertion_line, node.end_lineno)

    updated_lines = updated.splitlines(keepends=True)
    updated_lines.insert(insertion_line, "import json\n")
    updated = "".join(updated_lines)

# Verify before writing.
compile(updated, str(script), "exec")

# Confirm there is now exactly one return.
final_tree = ast.parse(updated)

final_function = next(
    node for node in final_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "validate_dataset"
)

final_returns = [
    node for node in final_function.body
    if isinstance(node, ast.Return)
]

assert len(final_returns) == 1

script.write_text(updated, encoding="utf-8")

print("Duplicate return removed: PASS")
print("Audio validation added: PASS")
print("Manifest verification preserved: PASS")
print("Python syntax: PASS")
print("Training executed: NO")

STEP 45C — FINAL AUDIO VALIDATION FIX


AssertionError: Unexpected return values. No changes made.

In [32]:
from pathlib import Path
import ast

script = Path(
    "/content/icelandic-asr-merge-check/training/train_whisper.py"
)

print("STEP 45D — REPAIR VALIDATION FUNCTION")
print("=" * 65)

source = script.read_text(encoding="utf-8")
tree = ast.parse(source)
lines = source.splitlines(keepends=True)

function = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "validate_dataset"
)

returns = [
    node for node in function.body
    if isinstance(node, ast.Return)
]

# Check the AST structure, not its formatted text.
assert len(returns) == 2
assert all(
    isinstance(node.value, ast.Tuple)
    and len(node.value.elts) == 2
    and all(isinstance(x, ast.Name) for x in node.value.elts)
    and [x.id for x in node.value.elts] == [
        "manifests", "audio_root"
    ]
    for node in returns
)

assert "Audio availability verification: PASS" not in source

# Remove the duplicate return statements.
for node in sorted(returns, key=lambda n: n.lineno, reverse=True):
    del lines[node.lineno - 1:node.end_lineno]

source = "".join(lines)

marker = '    print("Manifest SHA-256 verification: PASS")'
assert source.count(marker) == 1

addition = '''
    # Verify all frozen-manifest audio files.
    missing = []
    pointers = []
    checked = 0

    for split, manifest_path in manifests.items():
        with manifest_path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip():
                    continue

                record = json.loads(line)
                relative = Path(record["audio_path"])

                expected_split = (
                    "dev" if split == "validation" else split
                )

                if (
                    relative.is_absolute()
                    or ".." in relative.parts
                    or not relative.parts
                    or relative.parts[0] != expected_split
                ):
                    raise ValueError(
                        f"Invalid audio path at {split} "
                        f"line {line_number}: {relative}"
                    )

                audio_path = audio_root / relative
                checked += 1

                if not audio_path.is_file():
                    missing.append(str(relative))
                    continue

                with audio_path.open("rb") as audio_file:
                    header = audio_file.read(80)

                if header.startswith(
                    b"version https://git-lfs.github.com/spec"
                ):
                    pointers.append(str(relative))
                elif not header.startswith(b"fLaC"):
                    raise RuntimeError(
                        f"Invalid FLAC header: {audio_path}"
                    )

    if missing or pointers:
        raise RuntimeError(
            "Audio hydration incomplete.\\n"
            f"Checked: {checked}\\n"
            f"Missing: {len(missing)}\\n"
            f"Git LFS pointers: {len(pointers)}\\n"
            f"First missing: {missing[:3]}\\n"
            f"First pointers: {pointers[:3]}\\n"
            "Download the cleaned audio through Git LFS."
        )

    print(
        f"Audio availability verification: PASS "
        f"({checked} recordings)"
    )

    return manifests, audio_root
'''

source = source.replace(marker, marker + "\n" + addition, 1)

# Add json import safely after any future imports.
tree = ast.parse(source)

has_json = any(
    isinstance(node, ast.Import)
    and any(alias.name == "json" for alias in node.names)
    for node in tree.body
)

if not has_json:
    insertion = 0

    for node in tree.body:
        if (
            isinstance(node, ast.ImportFrom)
            and node.module == "__future__"
        ):
            insertion = max(insertion, node.end_lineno)

    updated_lines = source.splitlines(keepends=True)
    updated_lines.insert(insertion, "import json\n")
    source = "".join(updated_lines)

# Validate before writing.
compile(source, str(script), "exec")

final_tree = ast.parse(source)
final_function = next(
    node for node in final_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "validate_dataset"
)

assert sum(
    isinstance(node, ast.Return)
    for node in final_function.body
) == 1

script.write_text(source, encoding="utf-8")

print("Duplicate return removed: PASS")
print("Audio validation added: PASS")
print("Python syntax: PASS")
print("Training executed: NO")

STEP 45D — REPAIR VALIDATION FUNCTION
Duplicate return removed: PASS
Audio validation added: PASS
Python syntax: PASS
Training executed: NO


In [33]:
from pathlib import Path
import subprocess

print("STEP 46 — RUNTIME VALIDATION TEST")
print("=" * 70)

script = Path(
    "/content/icelandic-asr-merge-check/training/train_whisper.py"
)

dataset = Path("/content/icelandic-asr-project3")

assert script.is_file(), "Training script not found"
assert dataset.is_dir(), "Dataset repository not found"

result = subprocess.run(
    [
        "python",
        str(script),
        "--repo-dir",
        str(dataset),
        "--check-only",
    ],
    capture_output=True,
    text=True,
    timeout=120,
)

print("\nSTANDARD OUTPUT")
print("-" * 70)
print(result.stdout[-5000:] or "(empty)")

print("\nSTANDARD ERROR")
print("-" * 70)
print(result.stderr[-5000:] or "(empty)")

combined = result.stdout + result.stderr

# The dataset is deliberately missing its audio.
expected_failure = (
    result.returncode != 0
    and (
        "Prepared audio directory not found" in combined
        or "Audio hydration incomplete" in combined
    )
)

print("\nTEST RESULT")
print("-" * 70)

print("Missing-audio detection:", "PASS" if expected_failure else "FAIL")
print("Exit code:", result.returncode)
print("Training executed: NO")
print("Audio downloaded: NO")

assert expected_failure, (
    "Unexpected result. Review the output before continuing."
)

print("\nSTEP 46 COMPLETE")


STEP 46 — RUNTIME VALIDATION TEST

STANDARD OUTPUT
----------------------------------------------------------------------
(empty)

STANDARD ERROR
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/icelandic-asr-merge-check/training/train_whisper.py", line 572, in <module>
    main()
    ~~~~^^
  File "/content/icelandic-asr-merge-check/training/train_whisper.py", line 206, in main
    manifests, audio_root = validate_dataset(args.repo_dir)
                            ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/content/icelandic-asr-merge-check/training/train_whisper.py", line 119, in validate_dataset
    raise FileNotFoundError(
    ...<5 lines>...
    )
FileNotFoundError: Prepared audio directory not found: /content/icelandic-asr-project3/data/cleaned/samromur-21.05-20h-v1\nThe frozen manifests are verified, but the audio is not available. Use the repository revision containing Git LFS audio pointers and download the

In [34]:
from pathlib import Path
import nbformat

print("STEP 47 — INSPECT NOTEBOOK DATASET SETUP")
print("=" * 70)

notebook_path = Path(
    "/content/icelandic-asr-merge-check/"
    "NLP_Project3_Person3_Whisper_Clean.ipynb"
)

assert notebook_path.is_file(), "Notebook not found"

nb = nbformat.read(notebook_path, as_version=4)
nbformat.validate(nb)

keywords = [
    "DATASET_COMMIT",
    "DATASET_SELECTION",
    "git clone --no-checkout",
    "git checkout",
    "git lfs",
    "icelandic-asr-project3",
]

matches = []

for index, cell in enumerate(nb.cells):
    source = cell.source

    if any(keyword in source for keyword in keywords):
        matches.append(index)

print("Notebook validation: PASS")
print("Matching cells:", matches)

for index in matches:
    cell = nb.cells[index]

    print(f"\n{'=' * 70}")
    print(f"CELL {index} — {cell.cell_type.upper()}")
    print("=" * 70)

    for line_number, line in enumerate(
        cell.source.splitlines(), start=1
    ):
        print(f"{line_number:3}: {line}")

print("\nNotebook modified: NO")
print("Training executed: NO")

STEP 47 — INSPECT NOTEBOOK DATASET SETUP


NotebookValidationError: Additional properties are not allowed ('metadata' was unexpected)

Failed validating 'additionalProperties' in stream:

On instance['cells'][49]['outputs'][0]:
{'metadata': {'tags': None},
 'name': 'stdout',
 'output_type': 'stream',
 'text': '============================================================\n'
         'STE...'}

In [35]:
from pathlib import Path
import nbformat

print("STEP 47A — REPAIR NOTEBOOK OUTPUT METADATA")
print("=" * 70)

notebook_path = Path(
    "/content/icelandic-asr-merge-check/"
    "NLP_Project3_Person3_Whisper_Clean.ipynb"
)

assert notebook_path.is_file(), "Notebook not found"

nb = nbformat.read(notebook_path, as_version=4)

repairs = []

for cell_index, cell in enumerate(nb.cells):
    if cell.cell_type != "code":
        continue

    for output_index, output in enumerate(cell.get("outputs", [])):
        if output.get("output_type") == "stream":
            if "metadata" in output:
                metadata = output.pop("metadata")
                repairs.append(
                    (cell_index, output_index, metadata)
                )

# Verify the repaired notebook before saving.
nbformat.validate(nb)

# Preserve the original in case we need to revert.
backup = notebook_path.with_suffix(".step47_backup.ipynb")

assert not backup.exists(), (
    f"Backup already exists: {backup}"
)

backup.write_bytes(notebook_path.read_bytes())

nbformat.write(nb, notebook_path)

# Verify the saved version.
verified = nbformat.read(notebook_path, as_version=4)
nbformat.validate(verified)

print("Removed invalid stream metadata:", len(repairs))

for cell_index, output_index, metadata in repairs:
    print(
        f"  Cell {cell_index}, output {output_index}: "
        f"{metadata}"
    )

print("Notebook validation: PASS")
print("Backup created:", backup.name)
print("Code cells changed: NO")
print("Training executed: NO")

STEP 47A — REPAIR NOTEBOOK OUTPUT METADATA
Removed invalid stream metadata: 2
  Cell 49, output 0: {'tags': None}
  Cell 49, output 2: {'tags': None}
Notebook validation: PASS
Backup created: NLP_Project3_Person3_Whisper_Clean.step47_backup.ipynb
Code cells changed: NO
Training executed: NO


In [36]:
from pathlib import Path
import nbformat

print("STEP 47B — INSPECT NOTEBOOK DATASET SETUP")
print("=" * 70)

notebook_path = Path(
    "/content/icelandic-asr-merge-check/"
    "NLP_Project3_Person3_Whisper_Clean.ipynb"
)

nb = nbformat.read(notebook_path, as_version=4)
nbformat.validate(nb)

keywords = [
    "DATASET_COMMIT",
    "DATASET_SELECTION",
    "git clone",
    "git checkout",
    "git lfs",
    "icelandic-asr-project3",
]

matches = []

for index, cell in enumerate(nb.cells):
    if any(
        keyword in cell.source
        for keyword in keywords
    ):
        matches.append(index)

print("Notebook validation: PASS")
print("Matching cells:", matches)

for index in matches:
    cell = nb.cells[index]

    print(f"\n{'=' * 70}")
    print(f"CELL {index} — {cell.cell_type.upper()}")
    print("=" * 70)

    for line_number, line in enumerate(
        cell.source.splitlines(), start=1
    ):
        print(f"{line_number:3}: {line}")

print("\nNotebook modified: NO")
print("Training executed: NO")

STEP 47B — INSPECT NOTEBOOK DATASET SETUP
Notebook validation: PASS
Matching cells: [1, 3, 23, 25, 27, 29, 31, 33, 35, 37, 51]

CELL 1 — CODE
  1: # Reproducibility setup — Person 3
  2: # Install the exact package versions documented in the repository.
  3: # Run this cell in a fresh Colab environment before running other code.
  4: 
  5: %pip install -q -r https://raw.githubusercontent.com/RanimAlyaseen/icelandic-asr-project3/person3-whisper-reproducibility/training/requirements-person3.txt

CELL 3 — CODE
  1: # Fixed revisions for future reproducible runs.
  2: # These are not verified records of the original training checkout.
  3: 
  4: BASE_MODEL_ID = "openai/whisper-small"
  5: BASE_MODEL_REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"
  6: 
  7: DATASET_SELECTION = "samromur-21.05-20h-v1"
  8: DATASET_REPOSITORY_REVISION = "55f5a6856e92cdf89608c492cc470aea0d019e1b"
  9: 
 10: print("Base model:", BASE_MODEL_ID)
 11: print("Pinned model revision:", BASE_MODEL_REVISION)
 12

In [37]:
from pathlib import Path
import nbformat

print("STEP 48 — FIX NOTEBOOK DATASET CHECKOUT")
print("=" * 70)

notebook_path = Path(
    "/content/icelandic-asr-merge-check/"
    "NLP_Project3_Person3_Whisper_Clean.ipynb"
)

nb = nbformat.read(notebook_path, as_version=4)
nbformat.validate(nb)

assert "DATASET_REPOSITORY_REVISION" in nb.cells[3].source
assert 'git", "clone", "--no-checkout"' in nb.cells[23].source

# Cell 3: document both dataset revisions.
nb.cells[3].source = '''# Fixed revisions for future reproducible runs.
# These are not verified records of the original training checkout.

BASE_MODEL_ID = "openai/whisper-small"
BASE_MODEL_REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"

DATASET_SELECTION = "samromur-21.05-20h-v1"

# Original dataset preparation revision.
DATASET_PREPARATION_REVISION = (
    "55f5a6856e92cdf89608c492cc470aea0d019e1b"
)

# Revision containing Git LFS pointers for prepared audio.
DATASET_REPOSITORY_REVISION = (
    "96e5839b27d9871d6b2408b3c3719dc809a730b7"
)

print("Base model:", BASE_MODEL_ID)
print("Pinned model revision:", BASE_MODEL_REVISION)
print("Dataset selection:", DATASET_SELECTION)
print("Preparation revision:", DATASET_PREPARATION_REVISION)
print("Audio-enabled revision:", DATASET_REPOSITORY_REVISION)
'''

# Cell 23: safely check out the audio-enabled revision.
nb.cells[23].source = '''from pathlib import Path
import subprocess
import os
import hashlib

print("=" * 60)
print("STEP 7A — DATASET REPOSITORY SETUP")
print("=" * 60)

repo_url = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
repo_path = Path("/content/icelandic-asr-project3")

DATASET_COMMIT = (
    "96e5839b27d9871d6b2408b3c3719dc809a730b7"
)

EXPECTED_MANIFEST_HASHES = {
    "train.jsonl": "d36a1adccdb10682b30a6fd8b2f7e20f8c86babb4b1d20a007af79179c4436ed",
    "dev.jsonl": "f0e7675cd31497788250728bba966d1d884ad5f929b3488724b2fef2110f9bf4",
    "test.jsonl": "f5e346f545a5d7f4298f1838bc63b978a4d7ab9d063ddc97897fa065223d0481",
}

env = {
    **os.environ,
    "GIT_LFS_SKIP_SMUDGE": "1",
}

if not repo_path.exists():
    # Normal clone avoids the staged-deletion issue caused by
    # git clone --no-checkout.
    subprocess.run(
        ["git", "clone", repo_url, str(repo_path)],
        check=True,
        env=env,
    )
else:
    print("Repository already exists.")

    # Never overwrite local changes in an existing checkout.
    status = subprocess.run(
        ["git", "-C", str(repo_path), "status", "--porcelain"],
        capture_output=True,
        text=True,
        check=True,
    )

    if status.stdout.strip():
        raise RuntimeError(
            "Dataset repository contains uncommitted changes. "
            "Use a clean checkout before switching revisions."
        )

# Fetch the exact pinned commit.
subprocess.run(
    ["git", "-C", str(repo_path), "fetch", "origin", DATASET_COMMIT],
    check=True,
)

# Skip automatic LFS downloads during checkout.
subprocess.run(
    ["git", "-C", str(repo_path), "checkout", "--detach", DATASET_COMMIT],
    check=True,
    env=env,
)

actual_commit = subprocess.run(
    ["git", "-C", str(repo_path), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

assert actual_commit == DATASET_COMMIT, "Dataset revision mismatch!"

# Verify the exact prepared manifests.
for filename, expected_hash in EXPECTED_MANIFEST_HASHES.items():
    manifest = repo_path / "output" / filename

    assert manifest.is_file(), f"Missing manifest: {filename}"

    actual_hash = hashlib.sha256(
        manifest.read_bytes()
    ).hexdigest()

    assert actual_hash == expected_hash, (
        f"Manifest SHA-256 mismatch: {filename}"
    )

    print(f"{filename}: SHA-256 PASS")

print("\\\\nDataset checkout:", actual_commit)
print("Dataset revision verified: PASS")
print("Frozen manifests verified: PASS")
print("Audio download: NOT STARTED")
print("\\\\nSTEP 7A COMPLETE")
'''

# Validate all notebook cells before saving.
nbformat.validate(nb)

for index in [3, 23]:
    compile(
        nb.cells[index].source,
        f"notebook_cell_{index}",
        "exec",
    )

nbformat.write(nb, notebook_path)

# Verify the saved notebook.
verified = nbformat.read(notebook_path, as_version=4)
nbformat.validate(verified)

print("Cell 3 updated: PASS")
print("Cell 23 updated: PASS")
print("Notebook validation: PASS")
print("Python syntax: PASS")
print("Dataset downloaded: NO")
print("Training executed: NO")

STEP 48 — FIX NOTEBOOK DATASET CHECKOUT
Cell 3 updated: PASS
Cell 23 updated: PASS
Notebook validation: PASS
Python syntax: PASS
Dataset downloaded: NO
Training executed: NO


In [38]:
from pathlib import Path
import nbformat

print("STEP 49 — UPDATE GIT LFS AUDIO RETRIEVAL")
print("=" * 70)

notebook_path = Path(
    "/content/icelandic-asr-merge-check/"
    "NLP_Project3_Person3_Whisper_Clean.ipynb"
)

nb = nbformat.read(notebook_path, as_version=4)
nbformat.validate(nb)

assert "git" in nb.cells[27].source
assert "lfs" in nb.cells[27].source

nb.cells[27].source = '''import subprocess
from pathlib import Path

print("=" * 60)
print("STEP 7C — GIT LFS AUDIO RETRIEVAL")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")
dataset_selection = "samromur-21.05-20h-v1"

audio_root = repo / "data" / "cleaned" / dataset_selection

assert (repo / ".git").exists(), "Dataset repository not found"

# Install Git LFS if necessary.
check = subprocess.run(
    ["git", "lfs", "version"],
    capture_output=True,
    text=True,
)

if check.returncode != 0:
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git-lfs"],
        check=True,
    )

subprocess.run(
    ["git", "lfs", "install"],
    cwd=repo,
    check=True,
)

# Download only the selected prepared dataset.
include_pattern = (
    f"data/cleaned/{dataset_selection}/**"
)

print("Downloading prepared audio through Git LFS...")
print("Include pattern:", include_pattern)

subprocess.run(
    [
        "git", "lfs", "pull",
        f"--include={include_pattern}",
        "--exclude=",
    ],
    cwd=repo,
    check=True,
)

# Verify that FLAC files were hydrated.
audio_files = sorted(audio_root.rglob("*.flac"))

if not audio_files:
    raise FileNotFoundError(
        f"No FLAC files found under {audio_root}"
    )

pointer_files = []
invalid_files = []

for audio_file in audio_files:
    with audio_file.open("rb") as handle:
        header = handle.read(80)

    if header.startswith(
        b"version https://git-lfs.github.com/spec"
    ):
        pointer_files.append(audio_file)

    elif not header.startswith(b"fLaC"):
        invalid_files.append(audio_file)

if pointer_files or invalid_files:
    raise RuntimeError(
        "Audio verification failed. "
        f"LFS pointers: {len(pointer_files)}, "
        f"invalid FLAC headers: {len(invalid_files)}"
    )

print("\\\\nAUDIO VERIFICATION")
print("-" * 60)
print(f"FLAC files: {len(audio_files):,}")
print("Remaining Git LFS pointers: 0")
print("Invalid FLAC headers: 0")
print("Audio retrieval verification: PASS")
print("\\\\nSTEP 7C COMPLETE")
'''

# Check syntax and notebook structure.
compile(
    nb.cells[27].source,
    "notebook_cell_27",
    "exec",
)

nbformat.validate(nb)

nbformat.write(nb, notebook_path)

verified = nbformat.read(notebook_path, as_version=4)
nbformat.validate(verified)

print("Cell 27 updated: PASS")
print("Python syntax: PASS")
print("Notebook validation: PASS")
print("Audio downloaded: NO")
print("Training executed: NO")

STEP 49 — UPDATE GIT LFS AUDIO RETRIEVAL
Cell 27 updated: PASS
Python syntax: PASS
Notebook validation: PASS
Audio downloaded: NO
Training executed: NO


In [39]:
from pathlib import Path

print("STEP 50 — INSPECT README STRUCTURE")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")
readme = repo / "README.md"
script = repo / "training/train_whisper.py"
requirements = repo / "training/requirements-person3.txt"

assert readme.is_file(), "README.md not found"
assert script.is_file(), "Standalone training script not found"
assert requirements.is_file(), "Requirements file not found"

lines = readme.read_text(encoding="utf-8").splitlines()

print("\nREADME HEADINGS")
print("-" * 70)

for number, line in enumerate(lines, 1):
    if line.lstrip().startswith("#"):
        print(f"{number:4}: {line}")

print("\nPERSON 3 / WHISPER REFERENCES")
print("-" * 70)

keywords = [
    "person 3",
    "whisper",
    "fine-tun",
    "reproducib",
    "git lfs",
    "training",
]

matches = [
    index
    for index, line in enumerate(lines)
    if any(keyword in line.lower() for keyword in keywords)
]

# Merge nearby matches into readable excerpts.
shown = set()

for index in matches:
    start = max(0, index - 2)
    end = min(len(lines), index + 4)

    for position in range(start, end):
        if position not in shown:
            print(f"{position + 1:4}: {lines[position]}")
            shown.add(position)

print("\nREQUIREMENTS FILE")
print("-" * 70)
print(requirements.read_text(encoding="utf-8"))

print("\nFILE STATUS")
print("-" * 70)
print("README modified: NO")
print("Training executed: NO")
print("Audio downloaded: NO")

STEP 50 — INSPECT README STRUCTURE

README HEADINGS
----------------------------------------------------------------------
   1: # Samrómur 21.05 Icelandic ASR dataset preparation
  19: ## Download the audio
  30: # For model preprocessing, fetch the cleaned v1 audio
  33: # For raw-to-cleaned reproduction or review of shipped exclusions, fetch the raw subset
  41: ## Get started
  58: ## Repository contents
  87: ### MFCC + GMM-HMM baseline
  93: ## Load and preprocess a clip
 127: ## Focused audio-quality review
 163: ## Cleaning policy
 172: ## Regenerate cleaned audio from the raw subset
 186: ## Replay the frozen artifacts from a fresh checkout
 216: ## Re-audit the original audio
 245: ## Pretrained baselines, HuBERT + CTC and RNN-T
 257: ### Modeling environment and tests
 285: ### Baselines: published checkpoints, no training by our group
 307: ### Course baselines that do not support Icelandic (not run)
 317: ### HuBERT + CTC, fine-tuned for Icelandic by our group
 357: ### RN

In [40]:
from pathlib import Path

print("STEP 51 — DOCUMENT STANDALONE WHISPER TRAINING")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")
readme = repo / "README.md"
training_script = repo / "training/train_whisper.py"
requirements = repo / "training/requirements-person3.txt"

assert readme.is_file()
assert training_script.is_file()
assert requirements.is_file()

source = readme.read_text(encoding="utf-8")

heading = "### Reproduce Person 3's Whisper Small Training"

assert heading not in source, (
    "Training documentation already exists. No changes made."
)

# Insert before the existing five-sample evaluation section.
marker = "### Reproduce Person 3's Five-Sample Evaluation"

assert source.count(marker) == 1, (
    "Expected README insertion point not found."
)

section = """### Reproduce Person 3's Whisper Small Training

Person 3's standalone training entry point is
[`training/train_whisper.py`](training/train_whisper.py).
The original experiment fine-tuned multilingual Whisper Small on
the frozen Samrómur v1 training split and selected checkpoints
using validation WER.

The exact base-model revision and dataset repository checkout
used during the historical training run were not recorded.
The revisions below are therefore **fixed for future reproducible
runs**, not verified historical revisions.

| Component | Fixed revision or selection |
|---|---|
| Base model | `openai/whisper-small` |
| Base-model revision | `973afd24965f72e36ca33b3055d56a652f456b4d` |
| Dataset selection | `samromur-21.05-20h-v1` |
| Original preparation commit | `55f5a6856e92cdf89608c492cc470aea0d019e1b` |
| Audio-enabled repository commit | `96e5839b27d9871d6b2408b3c3719dc809a730b7` |

The preparation commit predates the committed Git LFS audio
pointers. The audio-enabled revision contains those pointers,
and its three prepared manifests were verified to be
byte-identical to the frozen preparation revision.

**1. Prepare a separate dataset checkout**

Run these commands from a terminal. Use a separate checkout
for the dataset rather than switching the revision of a working
development repository.

```bash
git lfs install
GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/RanimAlyaseen/icelandic-asr-project3.git icelandic-asr-dataset
cd icelandic-asr-dataset
GIT_LFS_SKIP_SMUDGE=1 git checkout --detach 96e5839b27d9871d6b2408b3c3719dc809a730b7
git lfs pull --include="data/cleaned/samromur-21.05-20h-v1/**" --exclude=""
cd ..

_IncompleteInputError: incomplete input (3325471608.py, line 30)

In [41]:
from pathlib import Path

print("STEP 51A — DOCUMENT STANDALONE WHISPER TRAINING")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")
readme = repo / "README.md"
script = repo / "training/train_whisper.py"
requirements = repo / "training/requirements-person3.txt"

assert readme.is_file()
assert script.is_file()
assert requirements.is_file()

source = readme.read_text(encoding="utf-8")

heading = "### Reproduce Person 3's Whisper Small Training"
marker = "### Reproduce Person 3's Five-Sample Evaluation"

assert heading not in source, "Section already exists; stopping."
assert source.count(marker) == 1, "README insertion point is ambiguous."

lines = [
    heading,
    "",
    "Person 3's standalone training entry point is",
    "[`training/train_whisper.py`](training/train_whisper.py).",
    "It fine-tunes multilingual Whisper Small using the frozen",
    "Samrómur v1 training and validation manifests.",
    "",
    "**Reproducibility limitation:** The exact base-model revision",
    "and dataset checkout used for the original training were not",
    "recorded. The revisions below are fixed for future runs;",
    "they are not verified historical revisions.",
    "",
    "| Component | Fixed revision or selection |",
    "|---|---|",
    "| Base model | `openai/whisper-small` |",
    "| Base-model revision | `973afd24965f72e36ca33b3055d56a652f456b4d` |",
    "| Dataset selection | `samromur-21.05-20h-v1` |",
    "| Preparation commit | `55f5a6856e92cdf89608c492cc470aea0d019e1b` |",
    "| Audio-enabled commit | `96e5839b27d9871d6b2408b3c3719dc809a730b7` |",
    "",
    "The preparation commit predates the Git LFS audio pointers.",
    "The audio-enabled commit contains those pointers, while its",
    "three prepared manifests are byte-identical to the frozen",
    "preparation revision.",
    "",
    "**1. Prepare a separate dataset checkout**",
    "",
    "Run these commands in a terminal, outside the development",
    "repository. Git LFS must be installed.",
    "",
    "~~~bash",
    "git lfs install",
    "GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/RanimAlyaseen/icelandic-asr-project3.git icelandic-asr-dataset",
    "cd icelandic-asr-dataset",
    "GIT_LFS_SKIP_SMUDGE=1 git checkout --detach 96e5839b27d9871d6b2408b3c3719dc809a730b7",
    'git lfs pull --include="data/cleaned/samromur-21.05-20h-v1/**" --exclude=""',
    "cd ..",
    "~~~",
    "",
    "The Git LFS command retrieves the prepared cleaned audio,",
    "not the complete raw subset.",
    "",
    "**2. Install dependencies**",
    "",
    "Run this command from the development repository:",
    "",
    "~~~bash",
    "python -m pip install -r training/requirements-person3.txt",
    "~~~",
    "",
    "GPU training requires a compatible CUDA-enabled PyTorch",
    "installation and sufficient GPU memory.",
    "",
    "**3. Validate the dataset without training**",
    "",
    "~~~bash",
    "python training/train_whisper.py --repo-dir ../icelandic-asr-dataset --check-only",
    "~~~",
    "",
    "The preflight checks the frozen manifest SHA-256 hashes",
    "and verifies that manifest-referenced audio files exist",
    "and contain FLAC data rather than unresolved LFS pointers.",
    "",
    "| Manifest | Expected SHA-256 |",
    "|---|---|",
    "| `output/train.jsonl` | `d36a1adccdb10682b30a6fd8b2f7e20f8c86babb4b1d20a007af79179c4436ed` |",
    "| `output/dev.jsonl` | `f0e7675cd31497788250728bba966d1d884ad5f929b3488724b2fef2110f9bf4` |",
    "| `output/test.jsonl` | `f5e346f545a5d7f4298f1838bc63b978a4d7ab9d063ddc97897fa065223d0481` |",
    "",
    "**4. Run training when GPU resources are available**",
    "",
    "~~~bash",
    "python training/train_whisper.py --repo-dir ../icelandic-asr-dataset --output-dir ./whisper_training_output",
    "~~~",
    "",
    "The configuration uses three epochs, learning rate `1e-5`,",
    "FP16, gradient checkpointing, and validation WER for",
    "checkpoint selection.",
    "",
    "**Validation status:** The standalone script passed Python",
    "syntax and CLI checks. Its missing-audio failure path was",
    "also tested. A complete training run using this script",
    "has not been performed or independently verified.",
    "",
    "For evaluation without retraining, use the recovered",
    "step-1000 checkpoint procedure below.",
    "",
    "---",
    "",
]

section = "\n".join(lines) + "\n"

updated = source.replace(marker, section + marker, 1)

assert updated.count(heading) == 1
assert updated.count(marker) == 1
assert updated.startswith(source[:source.index(marker)])

backup = repo / "README.step51_backup.md"

assert not backup.exists(), (
    "Backup already exists. Stopping to avoid overwriting it."
)

backup.write_text(source, encoding="utf-8")
readme.write_text(updated, encoding="utf-8")

# Verify saved content.
saved = readme.read_text(encoding="utf-8")

assert heading in saved
assert "train_whisper.py" in saved
assert "--check-only" in saved
assert "96e5839b27d9871d6b2408b3c3719dc809a730b7" in saved
assert marker in saved

print("README section inserted: PASS")
print("Dataset revisions documented: PASS")
print("Manifest hashes documented: PASS")
print("Preflight command documented: PASS")
print("Standalone training command documented: PASS")
print("Historical reproducibility caveat: PASS")
print("Existing evaluation section preserved: PASS")
print("Backup created:", backup.name)
print("Training executed: NO")
print("Audio downloaded: NO")

STEP 51A — DOCUMENT STANDALONE WHISPER TRAINING
README section inserted: PASS
Dataset revisions documented: PASS
Manifest hashes documented: PASS
Preflight command documented: PASS
Standalone training command documented: PASS
Historical reproducibility caveat: PASS
Existing evaluation section preserved: PASS
Backup created: README.step51_backup.md
Training executed: NO
Audio downloaded: NO


In [42]:
from pathlib import Path
import subprocess
import ast
import nbformat

print("STEP 52 — FINAL STATIC VALIDATION")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")

notebook_path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"
script_path = repo / "training/train_whisper.py"
requirements_path = repo / "training/requirements-person3.txt"
readme_path = repo / "README.md"

required_files = [
    notebook_path,
    script_path,
    requirements_path,
    readme_path,
]

print("\n1. REQUIRED FILES")
print("-" * 70)

for path in required_files:
    assert path.is_file(), f"Missing file: {path}"
    print(f"PASS: {path.relative_to(repo)}")

print("\n2. NOTEBOOK VALIDATION")
print("-" * 70)

nb = nbformat.read(notebook_path, as_version=4)
nbformat.validate(nb)

print("Notebook structure: PASS")

checked = 0

for index, cell in enumerate(nb.cells):
    if cell.cell_type != "code":
        continue

    source = cell.source

    # IPython magics are valid notebook syntax but not Python AST syntax.
    if any(
        line.lstrip().startswith(("%", "!"))
        for line in source.splitlines()
    ):
        continue

    try:
        ast.parse(source)
        checked += 1
    except SyntaxError as exc:
        raise RuntimeError(
            f"Syntax error in notebook cell {index}: {exc}"
        ) from exc

print(f"Standard Python cells checked: {checked}")
print("Notebook Python syntax: PASS")

print("\n3. TRAINING SCRIPT")
print("-" * 70)

script_source = script_path.read_text(encoding="utf-8")
ast.parse(script_source)

print("Standalone script syntax: PASS")

assert "--check-only" in script_source
assert "973afd24965f72e36ca33b3055d56a652f456b4d" in script_source

print("Preflight mode: PASS")
print("Pinned model revision: PASS")

print("\n4. REQUIREMENTS")
print("-" * 70)

requirements = requirements_path.read_text(encoding="utf-8")

assert "gdown==5.2.2" in requirements

print("Pinned gdown dependency: PASS")

print("\n5. README")
print("-" * 70)

readme = readme_path.read_text(encoding="utf-8")

required_documentation = [
    "### Reproduce Person 3's Whisper Small Training",
    "training/train_whisper.py",
    "--check-only",
    "96e5839b27d9871d6b2408b3c3719dc809a730b7",
    "### Reproduce Person 3's Five-Sample Evaluation",
]

for item in required_documentation:
    assert item in readme, f"Missing README documentation: {item}"

print("Standalone training documentation: PASS")
print("Existing evaluation documentation: PASS")

print("\n6. GIT WHITESPACE CHECK")
print("-" * 70)

result = subprocess.run(
    ["git", "diff", "--check"],
    cwd=repo,
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Git whitespace check failed")

print("Git diff --check: PASS")

print("\n7. GIT STATUS")
print("-" * 70)

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout or "Working tree clean")

print("\n8. TEMPORARY BACKUPS")
print("-" * 70)

backups = [
    repo / "README.step51_backup.md",
    repo / "training/train_whisper.py.step44_backup",
    repo / "NLP_Project3_Person3_Whisper_Clean.step47_backup.ipynb",
]

for path in backups:
    if path.exists():
        print(f"REMOVE BEFORE COMMIT: {path.relative_to(repo)}")
    else:
        print(f"Not present: {path.name}")

print("\n" + "=" * 70)
print("STEP 52 COMPLETE")
print("Training executed: NO")
print("Audio downloaded: NO")
print("Files modified: NO")

STEP 52 — FINAL STATIC VALIDATION

1. REQUIRED FILES
----------------------------------------------------------------------
PASS: NLP_Project3_Person3_Whisper_Clean.ipynb
PASS: training/train_whisper.py
PASS: training/requirements-person3.txt
PASS: README.md

2. NOTEBOOK VALIDATION
----------------------------------------------------------------------
Notebook structure: PASS
Standard Python cells checked: 22
Notebook Python syntax: PASS

3. TRAINING SCRIPT
----------------------------------------------------------------------
Standalone script syntax: PASS
Preflight mode: PASS
Pinned model revision: PASS

4. REQUIREMENTS
----------------------------------------------------------------------
Pinned gdown dependency: PASS

5. README
----------------------------------------------------------------------
Standalone training documentation: PASS
Existing evaluation documentation: PASS

6. GIT WHITESPACE CHECK
----------------------------------------------------------------------
Git diff --

In [43]:
from pathlib import Path
import subprocess

print("STEP 53 — CLEAN TEMPORARY BACKUPS")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")

backups = [
    repo / "README.step51_backup.md",
    repo / "training/train_whisper.py.step44_backup",
    repo / "NLP_Project3_Person3_Whisper_Clean.step47_backup.ipynb",
]

# Confirm all intended deliverables still exist.
deliverables = [
    repo / "README.md",
    repo / "NLP_Project3_Person3_Whisper_Clean.ipynb",
    repo / "training/train_whisper.py",
    repo / "training/requirements-person3.txt",
]

for path in deliverables:
    assert path.is_file(), f"Missing deliverable: {path}"

print("\nREMOVING TEMPORARY BACKUPS")
print("-" * 70)

for path in backups:
    if path.is_file():
        path.unlink()
        print(f"REMOVED: {path.relative_to(repo)}")
    else:
        print(f"ALREADY ABSENT: {path.name}")

print("\nVERIFY DELIVERABLES")
print("-" * 70)

for path in deliverables:
    assert path.is_file()
    print(f"PASS: {path.relative_to(repo)}")

print("\nGIT STATUS")
print("-" * 70)

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout or "Working tree clean")

print("\nBRANCH VERIFICATION")
print("-" * 70)

branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("Current branch:", branch)

assert branch == "person3-whisper-reproducibility", (
    "Unexpected Git branch. Stop before committing."
)

print("\nSTEP 53 COMPLETE")
print("Temporary backups removed: YES")
print("Project deliverables preserved: YES")
print("Changes committed: NO")
print("Changes pushed: NO")
print("Training executed: NO")

STEP 53 — CLEAN TEMPORARY BACKUPS

REMOVING TEMPORARY BACKUPS
----------------------------------------------------------------------
REMOVED: README.step51_backup.md
REMOVED: training/train_whisper.py.step44_backup
REMOVED: NLP_Project3_Person3_Whisper_Clean.step47_backup.ipynb

VERIFY DELIVERABLES
----------------------------------------------------------------------
PASS: README.md
PASS: NLP_Project3_Person3_Whisper_Clean.ipynb
PASS: training/train_whisper.py
PASS: training/requirements-person3.txt

GIT STATUS
----------------------------------------------------------------------
 M NLP_Project3_Person3_Whisper_Clean.ipynb
 M README.md
 M training/requirements-person3.txt
?? training/train_whisper.py


BRANCH VERIFICATION
----------------------------------------------------------------------
Current branch: person3-whisper-reproducibility

STEP 53 COMPLETE
Temporary backups removed: YES
Project deliverables preserved: YES
Changes committed: NO
Changes pushed: NO
Training executed: NO

In [44]:
from pathlib import Path
import subprocess

print("STEP 54 — FINAL TRAINING SCRIPT REVIEW")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")
script = repo / "training/train_whisper.py"

assert script.is_file(), "Training script not found"

lines = script.read_text(encoding="utf-8").splitlines()

print("\n1. SCRIPT CONTENT")
print("-" * 70)

for number, line in enumerate(lines, start=1):
    print(f"{number:4}: {line}")

print("\n2. CHANGED FILE SUMMARY")
print("-" * 70)

subprocess.run(
    ["git", "diff", "--stat"],
    cwd=repo,
    check=True,
)

print("\n3. REQUIREMENTS CHANGES")
print("-" * 70)

subprocess.run(
    ["git", "diff", "--", "training/requirements-person3.txt"],
    cwd=repo,
    check=True,
)

print("\n4. GIT STATUS")
print("-" * 70)

subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    check=True,
)

print("\nSTEP 54 COMPLETE")
print("Files modified: NO")
print("Changes committed: NO")
print("Changes pushed: NO")
print("Training executed: NO")

STEP 54 — FINAL TRAINING SCRIPT REVIEW

1. SCRIPT CONTENT
----------------------------------------------------------------------
   1: import hashlib
   2: #!/usr/bin/env python3
   3: """Reproducible Icelandic Whisper Small fine-tuning.
   4: 
   5: Based on NLP_Project3_Person3_Whisper_Clean.ipynb.
   6: 
   7: This script does not train when imported.
   8: Use --check-only to validate paths and configuration.
   9: """
  10: 
  11: import argparse
  12: import inspect
  13: import json
  14: import math
  15: import random
  16: import subprocess
  17: import time
  18: from dataclasses import dataclass
  19: from pathlib import Path
  20: from typing import Any
  21: 
  22: MODEL_ID = "openai/whisper-small"
  23: MODEL_REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"
  24: DATASET_COMMIT = "55f5a6856e92cdf89608c492cc470aea0d019e1b"
  25: 
  26: DATASET_SELECTION = "samromur-21.05-20h-v1"
  27: 
  28: EXPECTED_TRAIN = 10488
  29: EXPECTED_VALIDATION = 1314
  30: EXPECTED_TEST 

In [45]:
from pathlib import Path
import subprocess
import ast

print("STEP 55 — SCRIPT HEADER FIX AND GIT DIAGNOSTICS")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")
script = repo / "training/train_whisper.py"

source = script.read_text(encoding="utf-8")

# Correct the shebang placement without changing the program logic.
old_header = (
    "import hashlib\n"
    "#!/usr/bin/env python3\n"
)

new_header = (
    "#!/usr/bin/env python3\n"
    '"""Reproducible Icelandic Whisper Small fine-tuning.'
)

assert source.startswith(old_header), "Unexpected script header."

source = source.replace(
    old_header,
    "#!/usr/bin/env python3\n",
    1,
)

# Insert hashlib alongside the standard-library imports.
source = source.replace(
    "import argparse\n",
    "import argparse\nimport hashlib\n",
    1,
)

ast.parse(source)

script.write_text(source, encoding="utf-8")

print("\n1. SCRIPT HEADER")
print("-" * 70)

for line in source.splitlines()[:15]:
    print(line)

assert source.startswith("#!/usr/bin/env python3")
print("Shebang placement: PASS")
print("Python syntax: PASS")

print("\n2. TRAINING STEP CALCULATION")
print("-" * 70)

train_samples = 10488
batch_size = 2
gradient_accumulation = 8
epochs = 3

import math

batches_per_epoch = math.ceil(train_samples / batch_size)
steps_per_epoch = math.ceil(
    batches_per_epoch / gradient_accumulation
)

total_steps = steps_per_epoch * epochs

print("Training samples:", train_samples)
print("Batch size:", batch_size)
print("Gradient accumulation:", gradient_accumulation)
print("Epochs:", epochs)
print("Optimizer steps per epoch:", steps_per_epoch)
print("Total optimizer steps:", total_steps)

assert total_steps == 1968

print("Training-step calculation: PASS")

print("\n3. GIT STATUS")
print("-" * 70)

subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    check=True,
)

print("\n4. GIT DIFF SUMMARY")
print("-" * 70)

result = subprocess.run(
    ["git", "diff", "--stat", "HEAD"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout if result.stdout.strip() else "(No tracked-file diff)")

print("\n5. UNTRACKED FILES")
print("-" * 70)

result = subprocess.run(
    ["git", "ls-files", "--others", "--exclude-standard"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout if result.stdout.strip() else "(None)")

print("\nSTEP 55 COMPLETE")
print("Training executed: NO")
print("Changes committed: NO")
print("Changes pushed: NO")

STEP 55 — SCRIPT HEADER FIX AND GIT DIAGNOSTICS

1. SCRIPT HEADER
----------------------------------------------------------------------
#!/usr/bin/env python3
"""Reproducible Icelandic Whisper Small fine-tuning.

Based on NLP_Project3_Person3_Whisper_Clean.ipynb.

This script does not train when imported.
Use --check-only to validate paths and configuration.
"""

import argparse
import hashlib
import inspect
import json
import math
import random
Shebang placement: PASS
Python syntax: PASS

2. TRAINING STEP CALCULATION
----------------------------------------------------------------------
Training samples: 10488
Batch size: 2
Gradient accumulation: 8
Epochs: 3
Optimizer steps per epoch: 656
Total optimizer steps: 1968
Training-step calculation: PASS

3. GIT STATUS
----------------------------------------------------------------------

4. GIT DIFF SUMMARY
----------------------------------------------------------------------
 NLP_Project3_Person3_Whisper_Clean.ipynb | 27139 ++++++++++++

In [46]:
from pathlib import Path
import subprocess
import json
import difflib

print("STEP 56 — NOTEBOOK DIFF DIAGNOSTICS")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")
notebook = "NLP_Project3_Person3_Whisper_Clean.ipynb"

# Read original notebook from HEAD.
original_result = subprocess.run(
    ["git", "show", f"HEAD:{notebook}"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
)

original_text = original_result.stdout
current_text = (repo / notebook).read_text(encoding="utf-8")

original = json.loads(original_text)
current = json.loads(current_text)

print("\n1. NOTEBOOK STRUCTURE")
print("-" * 70)

print("Original cells:", len(original["cells"]))
print("Current cells:", len(current["cells"]))

assert len(original["cells"]) == len(current["cells"]), (
    "Cell count changed unexpectedly."
)

print("\n2. CELL SOURCE CHANGES")
print("-" * 70)

changed_cells = []

for index, (old, new) in enumerate(
    zip(original["cells"], current["cells"])
):
    old_source = "".join(old.get("source", []))
    new_source = "".join(new.get("source", []))

    if old_source != new_source:
        changed_cells.append(index)

        print(
            f"Cell {index}: "
            f"{len(old_source)} -> {len(new_source)} characters"
        )

print("\nTotal cells with changed source:", len(changed_cells))

print("\n3. METADATA AND OUTPUT CHANGES")
print("-" * 70)

metadata_changes = []
output_changes = []

for index, (old, new) in enumerate(
    zip(original["cells"], current["cells"])
):
    if old.get("metadata") != new.get("metadata"):
        metadata_changes.append(index)

    if old.get("outputs") != new.get("outputs"):
        output_changes.append(index)

print("Cells with metadata changes:", metadata_changes)
print("Cells with output changes:", output_changes)

print("\n4. RAW FILE FORMATTING")
print("-" * 70)

print("Original size:", len(original_text), "characters")
print("Current size:", len(current_text), "characters")

print(
    "Original indentation sample:",
    repr(original_text[:100])
)

print(
    "Current indentation sample:",
    repr(current_text[:100])
)

print("\n5. SOURCE CHANGE PREVIEW")
print("-" * 70)

for index in changed_cells[:8]:
    old_source = "".join(
        original["cells"][index].get("source", [])
    ).splitlines()

    new_source = "".join(
        current["cells"][index].get("source", [])
    ).splitlines()

    print(f"\nCELL {index}")
    print("-" * 40)

    diff = list(
        difflib.unified_diff(
            old_source,
            new_source,
            fromfile="HEAD",
            tofile="CURRENT",
            lineterm="",
        )
    )

    for line in diff[:35]:
        print(line)

    if len(diff) > 35:
        print("... additional differences omitted")

print("\nSTEP 56 COMPLETE")
print("Files modified: NO")
print("Changes committed: NO")
print("Changes pushed: NO")
print("Training executed: NO")

STEP 56 — NOTEBOOK DIFF DIAGNOSTICS

1. NOTEBOOK STRUCTURE
----------------------------------------------------------------------
Original cells: 52
Current cells: 52

2. CELL SOURCE CHANGES
----------------------------------------------------------------------
Cell 3: 811 -> 791 characters
Cell 9: 1397 -> 1515 characters
Cell 17: 2178 -> 2267 characters
Cell 23: 676 -> 2571 characters
Cell 27: 1097 -> 2118 characters
Cell 51: 2391 -> 2398 characters

Total cells with changed source: 6

3. METADATA AND OUTPUT CHANGES
----------------------------------------------------------------------
Cells with metadata changes: []
Cells with output changes: [49]

4. RAW FILE FORMATTING
----------------------------------------------------------------------
Original size: 456334 characters
Current size: 460421 characters
Original indentation sample: '{\n "nbformat": 4,\n "nbformat_minor": 0,\n "metadata": {\n  "colab": {\n   "provenance": [],\n   "gpuType'
Current indentation sample: '{\n "cells": [\

In [47]:
from pathlib import Path
import subprocess
import json
import nbformat

print("STEP 57 — NORMALIZE NOTEBOOK FORMATTING")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")
path = repo / "NLP_Project3_Person3_Whisper_Clean.ipynb"

original_text = subprocess.run(
    ["git", "show", f"HEAD:{path.name}"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
).stdout

current_text = path.read_text(encoding="utf-8")

original = json.loads(original_text)
current = json.loads(current_text)

assert len(original["cells"]) == len(current["cells"]) == 52

# Preserve a backup before changing the notebook.
backup = repo / "NLP_Project3_Person3_Whisper_Clean.step57_backup.ipynb"

assert not backup.exists(), "Backup already exists; stop to avoid overwriting it."
backup.write_text(current_text, encoding="utf-8")

changed_sources = []
changed_outputs = []

for index, (old_cell, new_cell) in enumerate(
    zip(original["cells"], current["cells"])
):
    assert old_cell["cell_type"] == new_cell["cell_type"]

    if old_cell.get("source") != new_cell.get("source"):
        old_cell["source"] = new_cell["source"]
        changed_sources.append(index)

    if old_cell.get("outputs") != new_cell.get("outputs"):
        old_cell["outputs"] = new_cell["outputs"]
        changed_outputs.append(index)

    # Preserve any other intentional cell-level changes.
    assert old_cell.get("metadata") == new_cell.get("metadata"), (
        f"Unexpected metadata difference in cell {index}"
    )

assert changed_sources == [3, 9, 17, 23, 27, 51]
assert changed_outputs == [49]

# Original notebook used one-space JSON indentation.
normalized_text = json.dumps(
    original,
    ensure_ascii=False,
    indent=1,
) + "\n"

normalized = json.loads(normalized_text)

# Verify that the reconstructed notebook matches the edited notebook
# semantically, including all cell sources, outputs and metadata.
assert normalized == current, (
    "Notebook content mismatch. Original file has not been replaced."
)

nbformat.validate(nbformat.reads(normalized_text, as_version=4))

path.write_text(normalized_text, encoding="utf-8")

print("\nNotebook validation: PASS")
print("Full notebook content preserved: PASS")
print("Changed source cells:", changed_sources)
print("Changed output cells:", changed_outputs)
print("Backup created:", backup.name)

print("\nGIT DIFF SUMMARY")
print("-" * 70)

subprocess.run(
    ["git", "diff", "--stat", "--", path.name],
    cwd=repo,
    check=True,
)

print("\nWHITESPACE CHECK")
print("-" * 70)

subprocess.run(
    ["git", "diff", "--check"],
    cwd=repo,
    check=True,
)

print("Git whitespace check: PASS")

print("\nSTEP 57 COMPLETE")
print("Training executed: NO")
print("Changes committed: NO")
print("Changes pushed: NO")

STEP 57 — NORMALIZE NOTEBOOK FORMATTING

Notebook validation: PASS
Full notebook content preserved: PASS
Changed source cells: [3, 9, 17, 23, 27, 51]
Changed output cells: [49]
Backup created: NLP_Project3_Person3_Whisper_Clean.step57_backup.ipynb

GIT DIFF SUMMARY
----------------------------------------------------------------------

WHITESPACE CHECK
----------------------------------------------------------------------
Git whitespace check: PASS

STEP 57 COMPLETE
Training executed: NO
Changes committed: NO
Changes pushed: NO


In [48]:
from pathlib import Path
import subprocess
import nbformat

print("STEP 58 — FINAL CLEANUP AND STAGING")
print("=" * 70)

repo = Path("/content/icelandic-asr-merge-check")

files = [
    "NLP_Project3_Person3_Whisper_Clean.ipynb",
    "README.md",
    "training/requirements-person3.txt",
    "training/train_whisper.py",
]

backup = repo / "NLP_Project3_Person3_Whisper_Clean.step57_backup.ipynb"

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=repo,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()

print("\n1. BRANCH VERIFICATION")
print("-" * 70)

branch = git("branch", "--show-current")
assert branch == "person3-whisper-reproducibility"
print("Branch:", branch)

print("\n2. DELIVERABLE VERIFICATION")
print("-" * 70)

for name in files:
    assert (repo / name).is_file(), f"Missing: {name}"
    print("PASS:", name)

nb = nbformat.read(
    repo / files[0],
    as_version=4,
)
nbformat.validate(nb)
print("Notebook validation: PASS")

print("\n3. REMOVE TEMPORARY BACKUP")
print("-" * 70)

if backup.exists():
    backup.unlink()
    print("Removed:", backup.name)
else:
    print("Backup already absent")

print("\n4. NOTEBOOK DIFF SIZE")
print("-" * 70)

print(
    git(
        "diff",
        "--numstat",
        "HEAD",
        "--",
        files[0],
    ) or "(No notebook changes)"
)

print("\n5. VERIFY EXACT CHANGE SET")
print("-" * 70)

tracked = set(
    git("diff", "--name-only", "HEAD").splitlines()
)

untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)

actual = tracked | untracked
expected = set(files)

print("Changed files:", sorted(actual))

assert actual == expected, (
    f"Unexpected changes: {sorted(actual ^ expected)}"
)

print("Exact four-file change set: PASS")

print("\n6. STAGE FOUR FILES")
print("-" * 70)

git("add", "--", *files)

staged = set(
    git("diff", "--cached", "--name-only").splitlines()
)

assert staged == expected, (
    f"Unexpected staged files: {sorted(staged ^ expected)}"
)

print("Staged files verified: PASS")

print("\n7. STAGED DIFF SUMMARY")
print("-" * 70)

print(
    git("diff", "--cached", "--stat")
)

print("\n8. STAGED WHITESPACE CHECK")
print("-" * 70)

git("diff", "--cached", "--check")

print("Whitespace check: PASS")

print("\nSTEP 58 COMPLETE")
print("Files staged: 4")
print("Changes committed: NO")
print("Changes pushed: NO")
print("Training executed: NO")

STEP 58 — FINAL CLEANUP AND STAGING

1. BRANCH VERIFICATION
----------------------------------------------------------------------
Branch: person3-whisper-reproducibility

2. DELIVERABLE VERIFICATION
----------------------------------------------------------------------
PASS: NLP_Project3_Person3_Whisper_Clean.ipynb
PASS: README.md
PASS: training/requirements-person3.txt
PASS: training/train_whisper.py
Notebook validation: PASS

3. REMOVE TEMPORARY BACKUP
----------------------------------------------------------------------
Removed: NLP_Project3_Person3_Whisper_Clean.step57_backup.ipynb

4. NOTEBOOK DIFF SIZE
----------------------------------------------------------------------
182	81	NLP_Project3_Person3_Whisper_Clean.ipynb

5. VERIFY EXACT CHANGE SET
----------------------------------------------------------------------
Changed files: ['NLP_Project3_Person3_Whisper_Clean.ipynb', 'README.md', 'training/requirements-person3.txt', 'training/train_whisper.py']
Exact four-file change se

In [49]:
import subprocess

print("STEP 59 — COMMIT AND PUSH")
print("=" * 70)

repo = "/content/icelandic-asr-merge-check"
branch = "person3-whisper-reproducibility"

def git(*args, check=True):
    result = subprocess.run(
        ["git", *args],
        cwd=repo,
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print(result.stdout.strip())

    if result.stderr:
        print(result.stderr.strip())

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Git command failed: git {' '.join(args)}"
        )

    return result

print("\n1. VERIFY BRANCH")
print("-" * 70)

result = git("branch", "--show-current")
assert result.stdout.strip() == branch

print("\n2. VERIFY STAGED FILES")
print("-" * 70)

expected = {
    "NLP_Project3_Person3_Whisper_Clean.ipynb",
    "README.md",
    "training/requirements-person3.txt",
    "training/train_whisper.py",
}

result = git("diff", "--cached", "--name-only")

actual = set(result.stdout.splitlines())

assert actual == expected, (
    f"Unexpected staged files: {actual ^ expected}"
)

print("Staged file verification: PASS")

print("\n3. VERIFY WHITESPACE")
print("-" * 70)

git("diff", "--cached", "--check")

print("Whitespace verification: PASS")

print("\n4. CREATE COMMIT")
print("-" * 70)

git(
    "commit",
    "-m",
    "Add reproducible Whisper Small training workflow",
)

print("\n5. PUSH FEATURE BRANCH")
print("-" * 70)

git(
    "push",
    "origin",
    branch,
)

print("\n6. VERIFY FINAL STATUS")
print("-" * 70)

git("status", "--short")

print("\n7. COMMIT DETAILS")
print("-" * 70)

git("log", "-1", "--oneline")

print("\nSTEP 59 COMPLETE")
print("Training executed: NO")
print("Changes committed: YES")
print("Push command completed: YES")

STEP 59 — COMMIT AND PUSH

1. VERIFY BRANCH
----------------------------------------------------------------------
person3-whisper-reproducibility

2. VERIFY STAGED FILES
----------------------------------------------------------------------
NLP_Project3_Person3_Whisper_Clean.ipynb
README.md
training/requirements-person3.txt
training/train_whisper.py
Staged file verification: PASS

3. VERIFY WHITESPACE
----------------------------------------------------------------------
Whitespace verification: PASS

4. CREATE COMMIT
----------------------------------------------------------------------
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@d87502c9f7a4.(none)')


RuntimeError: Git command failed: git commit -m Add reproducible Whisper Small training workflow

In [50]:
import subprocess

print("STEP 59A — CONFIGURE GIT IDENTITY")
print("=" * 70)

repo = "/content/icelandic-asr-merge-check"

# Configure author identity for this repository only.
subprocess.run(
    ["git", "config", "--local", "user.name", "Tasnim Alibraheem"],
    cwd=repo,
    check=True,
)

# Use GitHub's noreply email format.
# Replace this with your exact GitHub noreply address
# if your account uses an ID-prefixed address.
subprocess.run(
    [
        "git", "config", "--local",
        "user.email",
        "tasneemzouhair-coder@users.noreply.github.com",
    ],
    cwd=repo,
    check=True,
)

print("\nCONFIGURED IDENTITY")
print("-" * 70)

for key in ["user.name", "user.email"]:
    result = subprocess.run(
        ["git", "config", "--local", "--get", key],
        cwd=repo,
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"{key}: {result.stdout.strip()}")

print("\nSTAGED FILES")
print("-" * 70)

subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=repo,
    check=True,
)

print("\nSTEP 59A COMPLETE")
print("Git identity configured: YES")
print("Changes committed: NO")
print("Changes pushed: NO")

STEP 59A — CONFIGURE GIT IDENTITY

CONFIGURED IDENTITY
----------------------------------------------------------------------
user.name: Tasnim Alibraheem
user.email: tasneemzouhair-coder@users.noreply.github.com

STAGED FILES
----------------------------------------------------------------------

STEP 59A COMPLETE
Git identity configured: YES
Changes committed: NO
Changes pushed: NO


In [51]:
import subprocess

print("STEP 59B — VERIFY GIT STATE")
print("=" * 70)

repo = "/content/icelandic-asr-merge-check"

commands = {
    "Current branch": ["git", "branch", "--show-current"],
    "Working tree status": ["git", "status", "--short"],
    "Latest commit": ["git", "log", "-1", "--format=%h %s"],
    "Staged files": ["git", "diff", "--cached", "--name-only"],
    "Unstaged files": ["git", "diff", "--name-only"],
    "Untracked files": [
        "git", "ls-files", "--others", "--exclude-standard"
    ],
    "Local commits not pushed": [
        "git", "log", "origin/person3-whisper-reproducibility..HEAD",
        "--oneline"
    ],
}

for label, command in commands.items():
    print(f"\n{label}")
    print("-" * 70)

    result = subprocess.run(
        command,
        cwd=repo,
        capture_output=True,
        text=True,
    )

    print(result.stdout.strip() or "(none)")

    if result.returncode != 0:
        print("ERROR:", result.stderr.strip())

print("\nSTEP 59B COMPLETE")
print("Files modified: NO")
print("Changes committed: NO — diagnostic only")
print("Changes pushed: NO")
print("Training executed: NO")

STEP 59B — VERIFY GIT STATE

Current branch
----------------------------------------------------------------------
person3-whisper-reproducibility

Working tree status
----------------------------------------------------------------------
M  NLP_Project3_Person3_Whisper_Clean.ipynb
M  README.md
M  training/requirements-person3.txt
A  training/train_whisper.py

Latest commit
----------------------------------------------------------------------
c12178d4 Merge main and resolve Person 3 reproducibility conflicts

Staged files
----------------------------------------------------------------------
NLP_Project3_Person3_Whisper_Clean.ipynb
README.md
training/requirements-person3.txt
training/train_whisper.py

Unstaged files
----------------------------------------------------------------------
(none)

Untracked files
----------------------------------------------------------------------
(none)

Local commits not pushed
----------------------------------------------------------------------
(no

In [52]:
import subprocess

print("STEP 59C — FINAL COMMIT AND PUSH")
print("=" * 70)

repo = "/content/icelandic-asr-merge-check"
branch = "person3-whisper-reproducibility"

expected_files = {
    "NLP_Project3_Person3_Whisper_Clean.ipynb",
    "README.md",
    "training/requirements-person3.txt",
    "training/train_whisper.py",
}

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=repo,
        capture_output=True,
        text=True,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.stderr.strip():
        print(result.stderr.strip())

    if result.returncode != 0:
        raise RuntimeError(
            f"Git command failed: git {' '.join(args)}"
        )

    return result.stdout.strip()

print("\n1. VERIFY BRANCH")
print("-" * 70)

current_branch = run_git("branch", "--show-current")
assert current_branch == branch

print("\n2. VERIFY STAGED FILES")
print("-" * 70)

staged = set(
    run_git("diff", "--cached", "--name-only").splitlines()
)

assert staged == expected_files, (
    f"Unexpected staged files: {staged ^ expected_files}"
)

print("Staged files: PASS")

print("\n3. CREATE COMMIT")
print("-" * 70)

run_git(
    "commit",
    "-m",
    "Add reproducible Whisper Small training workflow",
)

print("\n4. PUSH TO GITHUB")
print("-" * 70)

run_git(
    "push",
    "origin",
    branch,
)

print("\n5. VERIFY REMOTE COMMIT")
print("-" * 70)

local_sha = run_git("rev-parse", "HEAD")

remote_sha = run_git(
    "ls-remote",
    "origin",
    f"refs/heads/{branch}",
).split()[0]

assert local_sha == remote_sha, (
    "Remote branch does not match local HEAD."
)

print("Remote commit verification: PASS")

print("\n6. FINAL WORKING TREE")
print("-" * 70)

status = run_git("status", "--porcelain")

assert not status, "Working tree is not clean."

print("Working tree: CLEAN")

print("\n" + "=" * 70)
print("STEP 59C COMPLETE")
print("Commit created: YES")
print("Push verified: YES")
print("Training executed: NO")

STEP 59C — FINAL COMMIT AND PUSH

1. VERIFY BRANCH
----------------------------------------------------------------------
person3-whisper-reproducibility

2. VERIFY STAGED FILES
----------------------------------------------------------------------
NLP_Project3_Person3_Whisper_Clean.ipynb
README.md
training/requirements-person3.txt
training/train_whisper.py
Staged files: PASS

3. CREATE COMMIT
----------------------------------------------------------------------
[person3-whisper-reproducibility f0430bfe] Add reproducible Whisper Small training workflow
 4 files changed, 844 insertions(+), 81 deletions(-)
 create mode 100644 training/train_whisper.py

4. PUSH TO GITHUB
----------------------------------------------------------------------
fatal: could not read Username for 'https://github.com': No such device or address


RuntimeError: Git command failed: git push origin person3-whisper-reproducibility

In [53]:
import subprocess
import shutil

print("STEP 60 — CHECK GITHUB AUTHENTICATION")
print("=" * 70)

repo = "/content/icelandic-asr-merge-check"

# Verify the commit is preserved.
result = subprocess.run(
    ["git", "log", "-1", "--format=%h %s"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
)

print("\nLATEST LOCAL COMMIT")
print(result.stdout.strip())

assert result.stdout.strip().startswith("f0430bfe"), (
    "Unexpected commit. Stop before pushing."
)

print("\nGITHUB CLI")
print("-" * 70)

gh_path = shutil.which("gh")

if gh_path:
    print("GitHub CLI installed:", gh_path)

    result = subprocess.run(
        ["gh", "auth", "status"],
        cwd=repo,
        capture_output=True,
        text=True,
    )

    print(result.stdout.strip() or result.stderr.strip())

    if result.returncode == 0:
        print("\nGitHub authentication: ACTIVE")
    else:
        print("\nGitHub authentication: REQUIRED")
else:
    print("GitHub CLI: NOT INSTALLED")

print("\nSTEP 60 COMPLETE")
print("Commit preserved: YES")
print("Push attempted: NO")
print("Training executed: NO")

STEP 60 — CHECK GITHUB AUTHENTICATION

LATEST LOCAL COMMIT
f0430bfe Add reproducible Whisper Small training workflow

GITHUB CLI
----------------------------------------------------------------------
GitHub CLI installed: /usr/bin/gh
You are not logged into any GitHub hosts. To log in, run: gh auth login

GitHub authentication: REQUIRED

STEP 60 COMPLETE
Commit preserved: YES
Push attempted: NO
Training executed: NO


In [56]:
import subprocess

print("STEP 61 — GITHUB AUTHENTICATION")
print("=" * 70)

subprocess.run(
    [
        "gh", "auth", "login",
        "--hostname", "github.com",
        "--git-protocol", "https",
        "--web",
    ],
    check=True,
)

print("\nGitHub authentication completed.")

STEP 61 — GITHUB AUTHENTICATION


KeyboardInterrupt: 

In [57]:
import subprocess

print("STEP 61B — GITHUB DEVICE AUTHENTICATION", flush=True)

result = subprocess.run(
    [
        "gh", "auth", "login",
        "--hostname", "github.com",
        "--git-protocol", "https",
        "--web",
    ],
    input="\n",
    text=True,
    capture_output=True,
    timeout=180,
)

print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Exit code:", result.returncode)

STEP 61B — GITHUB DEVICE AUTHENTICATION


TimeoutExpired: Command '['gh', 'auth', 'login', '--hostname', 'github.com', '--git-protocol', 'https', '--web']' timed out after 180 seconds

In [58]:
import subprocess
import time
import threading

print("STEP 61C — GITHUB AUTHENTICATION", flush=True)
print("=" * 60, flush=True)

process = subprocess.Popen(
    [
        "gh", "auth", "login",
        "--hostname", "github.com",
        "--git-protocol", "https",
        "--web",
    ],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def show_output():
    for line in process.stdout:
        print(line, end="", flush=True)

thread = threading.Thread(target=show_output, daemon=True)
thread.start()

# Respond to the initial Enter prompt.
time.sleep(2)

if process.poll() is None:
    try:
        process.stdin.write("\n")
        process.stdin.flush()
    except BrokenPipeError:
        pass

print("\nWaiting for GitHub authorization...", flush=True)
print("Open https://github.com/login/device if a code appears.", flush=True)

try:
    process.wait(timeout=300)
except subprocess.TimeoutExpired:
    process.terminate()
    print("Authentication timed out. No project files were changed.")

if process.returncode == 0:
    print("\nGitHub authentication: SUCCESS")
else:
    print("\nGitHub authentication: NOT COMPLETED")

STEP 61C — GITHUB AUTHENTICATION

! Failed to copy one-time code to clipboard
  No clipboard utilities available. Please install xsel, xclip, wl-clipboard or Termux:API add-on for termux-clipboard-get/set.
! First copy your one-time code: 3D77-38A3
Open this URL to continue in your web browser: https://github.com/login/device

Waiting for GitHub authorization...
Open https://github.com/login/device if a code appears.
✓ Authentication complete.
- gh config set -h github.com git_protocol https
✓ Configured git protocol
! Authentication credentials saved in plain text
✓ Logged in as tasneemzouhair-coder

GitHub authentication: SUCCESS


In [59]:
import subprocess

print("STEP 62 — PUSH AND VERIFY GITHUB")
print("=" * 70)

repo = "/content/icelandic-asr-merge-check"
branch = "person3-whisper-reproducibility"
expected_commit = "f0430bfe"

def run(command):
    result = subprocess.run(
        command,
        cwd=repo,
        capture_output=True,
        text=True,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.stderr.strip():
        print(result.stderr.strip())

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(command)}"
        )

    return result.stdout.strip()

print("\n1. VERIFY GITHUB ACCOUNT")
print("-" * 70)

account = run(["gh", "api", "user", "--jq", ".login"])
assert account == "tasneemzouhair-coder"

print("\n2. CONFIGURE GIT AUTHENTICATION")
print("-" * 70)

run(["gh", "auth", "setup-git"])
print("Git authentication configured: PASS")

print("\n3. VERIFY LOCAL COMMIT")
print("-" * 70)

commit = run(["git", "rev-parse", "HEAD"])
assert commit.startswith(expected_commit)

current_branch = run(["git", "branch", "--show-current"])
assert current_branch == branch

print("\n4. PUSH EXISTING COMMIT")
print("-" * 70)

run(["git", "push", "origin", branch])

print("\n5. VERIFY REMOTE COMMIT")
print("-" * 70)

remote_output = run([
    "git", "ls-remote",
    "origin",
    f"refs/heads/{branch}",
])

remote_commit = remote_output.split()[0]

assert commit == remote_commit
print("Local and remote commits match: PASS")

print("\n6. VERIFY WORKING TREE")
print("-" * 70)

status = run(["git", "status", "--porcelain"])
assert not status
print("Working tree: CLEAN")

print("\n7. VERIFY PULL REQUEST")
print("-" * 70)

run([
    "gh", "pr", "view", "4",
    "--repo", "RanimAlyaseen/icelandic-asr-project3",
    "--json", "number,state,mergeable,headRefName,url",
    "--jq",
    '"PR #\\(.number) | State: \\(.state) | Mergeable: \\(.mergeable) | Branch: \\(.headRefName) | URL: \\(.url)"',
])

print("\n" + "=" * 70)
print("STEP 62 COMPLETE")
print("Commit pushed: YES")
print("Remote verification: PASS")
print("Training executed: NO")

STEP 62 — PUSH AND VERIFY GITHUB

1. VERIFY GITHUB ACCOUNT
----------------------------------------------------------------------
tasneemzouhair-coder

2. CONFIGURE GIT AUTHENTICATION
----------------------------------------------------------------------
Git authentication configured: PASS

3. VERIFY LOCAL COMMIT
----------------------------------------------------------------------
f0430bfe355896aa4394c33bb58580daeb4efd62
person3-whisper-reproducibility

4. PUSH EXISTING COMMIT
----------------------------------------------------------------------
To https://github.com/RanimAlyaseen/icelandic-asr-project3.git
   c12178d4..f0430bfe  person3-whisper-reproducibility -> person3-whisper-reproducibility

5. VERIFY REMOTE COMMIT
----------------------------------------------------------------------
f0430bfe355896aa4394c33bb58580daeb4efd62	refs/heads/person3-whisper-reproducibility
Local and remote commits match: PASS

6. VERIFY WORKING TREE
---------------------------------------------------

In [60]:
import subprocess
import json
import time

print("STEP 63 — FINAL PULL REQUEST VERIFICATION")
print("=" * 70)

repo = "/content/icelandic-asr-merge-check"

for attempt in range(1, 5):
    result = subprocess.run(
        [
            "gh", "pr", "view", "4",
            "--repo", "RanimAlyaseen/icelandic-asr-project3",
            "--json",
            "number,state,mergeable,mergeStateStatus,headRefOid,url",
        ],
        cwd=repo,
        capture_output=True,
        text=True,
        check=True,
    )

    pr = json.loads(result.stdout)

    print(f"\nCheck {attempt}")
    print("PR:", pr["number"])
    print("State:", pr["state"])
    print("Mergeable:", pr["mergeable"])
    print("Merge status:", pr["mergeStateStatus"])
    print("Head commit:", pr["headRefOid"])

    if pr["mergeable"] != "UNKNOWN":
        break

    if attempt < 4:
        time.sleep(5)

assert pr["headRefOid"].startswith("f0430bfe"), (
    "PR head does not match the final commit."
)

print("\n" + "=" * 70)
print("PR HEAD COMMIT: VERIFIED")
print("PR MERGEABILITY:", pr["mergeable"])
print("Training executed: NO")

STEP 63 — FINAL PULL REQUEST VERIFICATION

Check 1
PR: 4
State: OPEN
Mergeable: MERGEABLE
Merge status: CLEAN
Head commit: f0430bfe355896aa4394c33bb58580daeb4efd62

PR HEAD COMMIT: VERIFIED
PR MERGEABILITY: MERGEABLE
Training executed: NO
